# Public 360 shooting-versus-continuation causal study

This notebook owns the research specification and scientific implementation. Small `PublicCausal` helpers only acquire/canonicalize public data and provide logging/execution infrastructure. Tests and workers export the tagged definitions verbatim into a content-addressed module.

Approved design: `../../docs/public_360_approved_plan.md`. Legacy audit: `../../docs/causal_code_audit.md`. Existing private PFF notebooks are unchanged.

Our target is the **supported ATT** of shooting versus the historical continuation mixture at live-play event starts, using signed first-goal reward over 15 seconds (or natural half termination). Effects are reward units per 100 supported decisions, not EPV, net expected goals, or an unconditional scoring-probability difference. Snapshot timing, partial visibility and unmeasured intentions remain limitations.

Development is Euro 2020/2024; World Cup 2022 stays unopened until measured gates and an immutable design lock pass. A small/inconclusive estimate or insufficient support is an acceptable research result. The new primary specification is a disclosed revision after the private-data audit, selected by diagnostics, never effect size.


In [ ]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'PublicCausal').is_dir())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
for variable in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[variable] = '1'


In [ ]:
STUDY_CONFIG = {'phase': 'smoke', 'root': '.', 'run_id': 'interactive-smoke', 'workers': 4, 'threads': 1, 'bootstrap_target': 10, 'null_reps': 3, 'simulation_datasets': 2, 'simulation_bootstrap_target': 10, 'resume': True}


## Specification, match-held-out models and common support

The 3 m visibility gate, start-only features and original-match split registry are explicit. All tuning/preprocessing/calibration and both counterfactual outcome predictions exclude the evaluated match. Histogram boosting disables row-based early stopping. Outcome choice uses ATT-weighted donor validation, with a one-standard-error preference for simpler models.

Support is an X-domain applied to both arms. Portable support excludes identity/cohort while full-adjustment propensity may include them. Thresholds and empirical cells are trained outside each evaluated match. Broadest diagnostics-passing support is primary; failed gates do not become a causal headline. Confirmation keeps the frozen Euro domain and only refits grouped nuisances with frozen settings.


In [ ]:
# %% Core imports and explicit configuration
from __future__ import annotations

import copy
import hashlib
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.metrics import log_loss, brier_score_loss

SOURCE_COMMIT = '4b73468fc5b0f1950f9f66fada70ad3a4f9327cb'
NUMERIC_FEATURES = ['x', 'y', 'distance', 'angle', 'defenders_3m',
                    'nearest_defender', 'second_defender', 'score_difference',
                    'period', 'time_seconds', 'time_since_prev_action',
                    'prior_10s_actions', 'prior_10s_progress']
CATEGORICAL_FEATURES = ['position_role', 'team_id', 'opponent_id', 'cohort']
SUPPORT_FEATURES = list(NUMERIC_FEATURES)
KEY_BALANCE_FEATURES = ['distance', 'angle', 'defenders_3m', 'nearest_defender',
                        'score_difference', 'time_seconds']


def scientific_config(overrides=None):
    profile = (overrides or {}).get('profile', 'extended')
    if profile not in {'submission', 'extended'}:
        raise ValueError(f'Unknown scientific profile: {profile}')
    config = dict(profile=profile, seed=20260930, outer_folds=5, inner_folds=3,
                  feature_names=NUMERIC_FEATURES + CATEGORICAL_FEATURES,
                  support_features=SUPPORT_FEATURES,
                  key_balance_features=KEY_BALANCE_FEATURES,
                  support_thresholds=[.05, .10, .15, .20],
                  donor_min=10, donor_match_min=3,
                  min_shots=200, min_matches=30, min_ess=200,
                  ess_shot_fraction=.25, key_smd_max=.10, other_smd_max=.15,
                  max_weight_share=.01, max_match_weight_share=.10,
                  e_candidates=[{'kind': 'logistic', 'C': c} for c in [.05, .2, 1]] +
                      [{'kind': 'hgb', 'leaves': 7, 'iterations': 100}],
                  outcome_candidates=[{'kind': 'ridge', 'alpha': a} for a in [.1, 1, 10]] +
                      [{'kind': 'gam', 'knots': k, 'alpha': a} for k in [3, 5] for a in [.1, 1, 10]] +
                      [{'kind': 'hgb', 'leaves': k, 'iterations': n} for k in [7, 15] for n in [100, 200]],
                  bootstrap_target=1000, null_reps=200, simulation_reps=200,
                  simulation_bootstrap=1000, workers=4)
    if profile == 'submission':
        config.update(att_only=True, diagnostic_m1=False,
            candidate_device='cuda', bootstrap_target=200, bootstrap_max_target=500,
            bootstrap_precision_extension=True,
            null_reps=50, simulation_reps=50, simulation_bootstrap=0,
            e_candidates=[{'kind': 'logistic', 'C': .2},
                          {'kind': 'hgb', 'leaves': 7, 'iterations': 100},
                          {'kind': 'xgb', 'device': (overrides or {}).get('candidate_device', 'cuda'),
                           'leaves': 7, 'iterations': 100}],
            outcome_candidates=[{'kind': 'ridge', 'alpha': 10},
                                {'kind': 'gam', 'knots': 3, 'alpha': 10},
                                {'kind': 'hgb', 'leaves': 7, 'iterations': 100}])
    if overrides:
        config.update(copy.deepcopy(overrides))
    return config


def stable_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                    separators=(',', ':')).encode()).hexdigest()


def log_event(logger, name, **kwargs):
    if logger is not None:
        logger.event(name, **kwargs)


def grouped_folds(frame, n_splits=5, seed=0):
    """One shared registry for BOTH action arms. Duplicate draws stay together.

    Greedy balance uses cohort and treatment counts, never outcomes. Explicitly
    fails for insufficient original groups/arms; no row-CV fallback.
    """
    groups = frame['original_match_id'].astype(str)
    unique = sorted(groups.unique())
    if len(unique) < n_splits:
        raise ValueError(f'{len(unique)} original matches cannot form {n_splits} grouped folds')
    cohorts = frame['cohort'].astype(str) if 'cohort' in frame else pd.Series('simulation', index=frame.index)
    cohort_levels = sorted(cohorts.unique())
    counts = []
    for group in unique:
        mask = groups == group
        counts.append([np.sum(mask & (cohorts == c) & (frame.Treatment == t))
                       for c in cohort_levels for t in [0, 1]])
    counts = np.asarray(counts, dtype=float)
    scales = np.maximum(counts.sum(axis=0) / n_splits, 1)
    rng = np.random.default_rng(seed)
    ordering = sorted(rng.permutation(len(unique)), key=lambda i: -np.linalg.norm(counts[i] / scales))
    totals = np.zeros((n_splits, counts.shape[1]))
    assignments = {}
    for step, i in enumerate(ordering):
        if step < n_splits:
            fold = step
        else:
            scores = []
            for fold_id in range(n_splits):
                candidate = totals.copy()
                candidate[fold_id] += counts[i]
                scores.append(np.var(candidate / scales, axis=0).sum())
            fold = int(np.argmin(scores))
        assignments[unique[i]] = fold
        totals[fold] += counts[i]
    registry = groups.map(assignments).to_numpy()
    output = []
    for fold in range(n_splits):
        train, test = np.flatnonzero(registry != fold), np.flatnonzero(registry == fold)
        assert set(groups.iloc[train]).isdisjoint(groups.iloc[test])
        if set(frame.iloc[train].Treatment.unique()) != {0, 1}:
            raise ValueError(f'Grouped training fold {fold} lacks an action arm')
        output.append((train, test))
    return output


def prepare_analysis(frame, config):
    rows = frame.copy()
    if 'eligible' in rows:
        rows = rows.loc[rows.eligible].copy()
    if 'Y' not in rows:
        rows['Y'] = rows[config.get('outcome_column', 'Y15')]
    rows = rows.loc[rows.Y.notna()].copy().reset_index(drop=True)
    rows['Treatment'] = rows['T'].astype(int)
    if not set(rows.Treatment.unique()).issubset({0, 1}):
        raise ValueError('Treatment must be binary')
    if not np.isin(rows.Y, [-1, 0, 1]).all():
        raise ValueError('Public signed first-goal outcome must be {-1,0,+1}')
    if 'original_match_id' not in rows:
        rows['original_match_id'] = rows.match_id.astype(str)
    if 'decision_row_id' not in rows:
        rows['decision_row_id'] = rows.original_match_id.astype(str) + ':' + rows.event_id.astype(str)
    missing = set(config['feature_names'] + config['support_features']) - set(rows.columns)
    if missing:
        raise ValueError(f'Missing pre-action adjustment features: {sorted(missing)}')
    for name in set(config['feature_names'] + config['support_features']):
        if not pd.api.types.is_numeric_dtype(rows[name]):
            rows[name] = rows[name].fillna('unknown').astype(str)
    return rows


In [ ]:


def feature_pipeline(frame, features, spec, task='outcome'):
    numeric = [name for name in features if pd.api.types.is_numeric_dtype(frame[name])]
    categorical = [name for name in features if name not in numeric]
    num_steps = [('impute', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True))]
    if spec['kind'] == 'gam':
        num_steps.append(('splines', SplineTransformer(n_knots=spec['knots'], degree=3, include_bias=False)))
    num_steps.append(('scale', StandardScaler()))
    transforms = [('numeric', Pipeline(num_steps), numeric)]
    if categorical:
        transforms.append(('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False,
                                                       min_frequency=5), categorical))
    preprocess = ColumnTransformer(transforms, remainder='drop')
    # pragmatic: sklearn preprocessing and tiny linear fits stay on CPU.
    # The bounded boosted backend receives its explicit frozen device.
    if spec['kind'] == 'xgb':
        from xgboost import XGBClassifier, XGBRegressor
        device = spec.get('device')
        if device not in {'cpu', 'cuda'}:
            raise ValueError('XGBoost specification requires explicit cpu/cuda device')
        factory = XGBClassifier if task == 'propensity' else XGBRegressor
        model = factory(tree_method='hist', device=device, max_leaves=spec['leaves'],
                        grow_policy='lossguide', max_depth=0,
                        n_estimators=spec['iterations'], learning_rate=.05,
                        reg_lambda=1., random_state=20260930, n_jobs=1)
    elif task == 'propensity':
        model = (LogisticRegression(C=spec.get('C', 1), max_iter=1000, solver='lbfgs')
                 if spec['kind'] == 'logistic' else
                 HistGradientBoostingClassifier(max_leaf_nodes=spec['leaves'], max_iter=spec['iterations'],
                                                learning_rate=.05, l2_regularization=1, early_stopping=False,
                                                random_state=20260930))
    else:
        model = (HistGradientBoostingRegressor(max_leaf_nodes=spec['leaves'], max_iter=spec['iterations'],
                                               learning_rate=.05, l2_regularization=1, early_stopping=False,
                                               random_state=20260930)
                 if spec['kind'] == 'hgb' else Ridge(alpha=spec['alpha'], solver='lsqr'))
    return Pipeline([('preprocess', preprocess), ('model', model)])


def fit_checked(model, x, y, spec, logger=None, stage='nuisance'):
    """Fit once and reject silent XGBoost CUDA fallback using booster config."""
    if spec['kind'] != 'xgb':
        return model.fit(x, y)
    import warnings
    import xgboost
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always')
        model.fit(x, y)
    booster_config = json.loads(model.named_steps['model'].get_booster().save_config())
    actual = booster_config['learner']['generic_param']['device']
    model.backend_manifest = dict(requested_device=spec['device'], actual_device=actual,
                                  xgboost_version=xgboost.__version__,
                                  booster_config=booster_config,
                                  warnings=[str(item.message) for item in observed])
    log_event(logger, 'xgb_backend', stage=stage, **model.backend_manifest)
    if spec['device'] == 'cuda' and not actual.startswith('cuda'):
        raise RuntimeError(f'Explicit CUDA XGBoost fit fell back to {actual}')
    if spec['device'] == 'cpu' and actual != 'cpu':
        raise RuntimeError(f'Explicit CPU XGBoost fit used {actual}')
    return model


def provenance_record(train, test, stage, fold, spec=None):
    train_ids = sorted(train.original_match_id.astype(str).unique())
    test_ids = sorted(test.original_match_id.astype(str).unique())
    if not set(train_ids).isdisjoint(test_ids):
        raise AssertionError(f'Match leakage in {stage}')
    return dict(stage=stage, fold=int(fold), train_match_ids=train_ids, test_match_ids=test_ids,
                train_matches_hash=stable_hash(train_ids), spec=spec)


def predict_checked(model, frame, method='predict', logger=None, stage='prediction'):
    """Record CUDA host-input prediction warnings without changing fitted device.

    A DMatrix bridge can increase prediction time/memory; it is not evidence
    that the fitted CUDA booster trained on CPU. Warnings remain visible.
    """
    if not hasattr(model, 'backend_manifest'):
        return getattr(model, method)(frame)
    import time
    import warnings
    started = time.monotonic()
    with warnings.catch_warnings(record=True) as observed:
        warnings.simplefilter('always')
        prediction = getattr(model, method)(frame)
    messages = [str(item.message) for item in observed]
    manifest = model.backend_manifest
    manifest['prediction_calls'] = manifest.get('prediction_calls', 0) + 1
    manifest['prediction_seconds'] = manifest.get('prediction_seconds', 0.) + time.monotonic() - started
    manifest['prediction_warnings'] = sorted(set(manifest.get('prediction_warnings', []) + messages))
    log_event(logger, 'xgb_prediction', stage=stage, method=method, rows=len(frame),
              actual_training_device=manifest['actual_device'], warnings=messages,
              prediction_seconds=time.monotonic() - started,
              warning_interpretation='Host-input DMatrix bridge may add prediction time/memory; fitted backend unchanged')
    for item in observed:
        warnings.warn(str(item.message), item.category, stacklevel=2)
    return prediction


def raw_probability(model, frame, features, logger=None, stage='propensity_prediction'):
    return np.clip(predict_checked(model, frame[features], method='predict_proba', logger=logger,
                                  stage=stage)[:, 1], 1e-6, 1 - 1e-6)


def grouped_sigmoid_calibrator(train, config, features, spec, stage, logger=None):
    """Calibration training predictions exclude their original match.

    When called in an inner validation loop, every component also excludes
    the entire inner validation sample; no indirectly leaking calibrator.
    """
    count = min(config['inner_folds'], train.original_match_id.nunique())
    if count < 2:
        raise ValueError('Grouped calibration needs at least two original matches')
    folds = grouped_folds(train, count, config['seed'] + 909)
    raw_oof = np.full(len(train), np.nan)
    provenance = []
    for fold, (i, j) in enumerate(folds):
        a, b = train.iloc[i], train.iloc[j]
        model = fit_checked(feature_pipeline(a, features, spec, 'propensity'), a[features], a.Treatment, spec, logger, stage)
        raw_oof[j] = raw_probability(model, b, features, logger, stage + '_calibration_prediction')
        provenance.append(provenance_record(a, b, stage + '_calibration_prediction', fold, spec))
        log_event(logger, 'model_fit', stage=stage + '_calibration', fold=fold,
                  train_rows=len(a), test_rows=len(b))
    logits = np.log(raw_oof / (1 - raw_oof)).reshape(-1, 1)
    return LogisticRegression(C=1e3, max_iter=1000).fit(logits, train.Treatment), provenance


def select_propensity_candidate(results, config):
    """Predeclared backend rule, fixed before any confirmation access.

    Default: lowest grouped held-out log-loss. Submission profile with an explicit
    CUDA candidate: one-standard-error rule. If the CUDA candidate's loss is within
    one SE (across inner grouped folds) of the lowest loss, it is selected, since
    statistically tied candidates are then separated by the planned GPU backend,
    never by ATT sign/magnitude.
    """
    best = min(results, key=lambda result: result['loss'])
    folds = np.asarray(best.get('fold_losses') or [], float)
    se = float(folds.std(ddof=1) / np.sqrt(len(folds))) if len(folds) > 1 else 0.
    rule = dict(applied='lowest_loss', lowest_spec=best['spec'], lowest_loss=best['loss'], lowest_loss_se=se,
                predeclared='lowest grouped log-loss; submission+cuda: prefer CUDA candidate within one SE of lowest')
    if config.get('profile') == 'submission' and config.get('candidate_device') == 'cuda':
        cuda = [r for r in results if r['spec'].get('device') == 'cuda' and r['loss'] <= best['loss'] + se]
        if cuda and cuda[0] is not best:
            rule['applied'] = 'one_se_prefer_cuda'
            return min(cuda, key=lambda result: result['loss']), rule
    return best, rule

def fit_propensity(train, config, features, frozen_spec=None, logger=None, stage='propensity', require_oof=False):
    if config.get('profile') == 'submission' and frozen_spec is not None and not require_oof:
        model = fit_checked(feature_pipeline(train, features, frozen_spec, 'propensity'),
                            train[features], train.Treatment, frozen_spec, logger, stage)
        calibration, provenance = None, []
        if frozen_spec['kind'] in {'hgb', 'xgb'}:
            calibration, provenance = grouped_sigmoid_calibrator(train, config, features, frozen_spec, stage, logger)
        log_event(logger, 'model_fit', stage=stage, fixed_spec=True, train_rows=len(train))
        return dict(model=model, calibration=calibration, spec=copy.deepcopy(frozen_spec),
                    oof=None, provenance=provenance, tuning=[], fitting_mode='fixed_direct')
    folds = grouped_folds(train, config['inner_folds'], config['seed'] + 101)
    specs = [frozen_spec] if frozen_spec is not None else config['e_candidates']
    results, provenance = [], []
    for spec_id, spec in enumerate(specs):
        oof = np.full(len(train), np.nan)
        losses = []
        for fold, (i, j) in enumerate(folds):
            a, b = train.iloc[i], train.iloc[j]
            model = fit_checked(feature_pipeline(a, features, spec, 'propensity'), a[features], a.Treatment, spec, logger, stage)
            oof[j] = raw_probability(model, b, features, logger, stage + '_inner_prediction')
            if spec['kind'] in {'hgb', 'xgb'}:
                calibrator, calibration_provenance = grouped_sigmoid_calibrator(a, config, features, spec, stage, logger)
                logits = np.log(oof[j] / (1 - oof[j])).reshape(-1, 1)
                oof[j] = np.clip(calibrator.predict_proba(logits)[:, 1], 1e-6, 1 - 1e-6)
                provenance.extend(calibration_provenance)
                provenance.append(provenance_record(a, b, stage + '_inner_calibration', fold, spec))
            losses.append(log_loss(b.Treatment, oof[j], labels=[0, 1]))
            provenance.append(provenance_record(a, b, stage + '_inner', fold, spec))
            log_event(logger, 'model_fit', stage=stage, candidate=spec_id, fold=fold,
                      train_rows=len(a), test_rows=len(b), loss=losses[-1])
        if not np.isfinite(oof).all():
            raise AssertionError('Incomplete grouped propensity predictions')
        results.append(dict(spec=spec, loss=float(log_loss(train.Treatment, oof, labels=[0, 1])),
                            brier=float(brier_score_loss(train.Treatment, oof)), oof=oof,
                            fold_losses=[float(value) for value in losses]))
    best, selection = select_propensity_candidate(results, config)
    for item in results:
        item['selected'] = item is best
    model = fit_checked(feature_pipeline(train, features, best['spec'], 'propensity'), train[features], train.Treatment, best['spec'], logger, stage)
    calibration = None
    if best['spec']['kind'] in {'hgb', 'xgb'}:
        calibration, calibration_provenance = grouped_sigmoid_calibrator(train, config, features, best['spec'], stage, logger)
        provenance.extend(calibration_provenance)
    log_event(logger, 'propensity_selection', stage=stage, **selection)
    return dict(model=model, calibration=calibration, spec=best['spec'],
                oof=best['oof'], provenance=provenance, selection=selection,
                tuning=[dict({k: v for k, v in item.items() if k != 'oof'}, selection_rule=selection) for item in results])


def predict_propensity(bundle, test, features, logger=None, stage='propensity_prediction'):
    probability = raw_probability(bundle['model'], test, features, logger, stage)
    if bundle['calibration'] is not None:
        logit = np.log(probability / (1 - probability)).reshape(-1, 1)
        probability = bundle['calibration'].predict_proba(logit)[:, 1]
    return np.clip(probability, 1e-6, 1 - 1e-6)


def fit_outcome(train, config, arm, propensity_oof, frozen_spec=None, logger=None):
    features = config['feature_names']
    if config.get('profile') == 'submission' and frozen_spec is not None:
        arm_train = train.loc[train.Treatment == arm]
        if len(arm_train) < 2:
            raise ValueError('Outcome training arm has fewer than two observations')
        model = fit_checked(feature_pipeline(arm_train, features, frozen_spec),
                            arm_train[features], arm_train.Y, frozen_spec, logger, f'm{arm}')
        log_event(logger, 'model_fit', stage=f'm{arm}', fixed_spec=True, train_rows=len(arm_train))
        return dict(model=model, spec=copy.deepcopy(frozen_spec), oof=None, provenance=[],
                    tuning=[], fitting_mode='fixed_direct')
    folds = grouped_folds(train, config['inner_folds'], config['seed'] + 101)
    specs = [frozen_spec] if frozen_spec is not None else config['outcome_candidates']
    candidates, provenance = [], []
    for spec_id, spec in enumerate(specs):
        oof, losses = np.full(len(train), np.nan), []
        for fold, (i, j) in enumerate(folds):
            a = train.iloc[i]
            a = a.loc[a.Treatment == arm]
            b = train.iloc[j]
            if len(a) < 2:
                raise ValueError('Outcome training arm has fewer than two observations')
            model = fit_checked(feature_pipeline(a, features, spec), a[features], a.Y, spec, logger, f'm{arm}')
            prediction = np.clip(predict_checked(model, b[features], logger=logger, stage=f'm{arm}_inner_prediction'), -1, 1)
            oof[j] = prediction
            observed = b.Treatment.to_numpy() == arm
            weight = propensity_oof[j][observed] / (1 - propensity_oof[j][observed]) if arm == 0 else np.ones(observed.sum())
            if observed.any():
                losses.append(float(np.average((b.Y.to_numpy()[observed] - prediction[observed]) ** 2, weights=weight)))
            provenance.append(provenance_record(a, b, f'm{arm}_inner', fold, spec))
            log_event(logger, 'model_fit', stage=f'm{arm}', candidate=spec_id, fold=fold,
                      train_rows=len(a), test_rows=len(b))
        observed = train.Treatment.to_numpy() == arm
        weight = propensity_oof[observed] / (1 - propensity_oof[observed]) if arm == 0 else np.ones(observed.sum())
        loss = float(np.average((train.Y.to_numpy()[observed] - oof[observed]) ** 2, weights=weight))
        se = float(np.std(losses, ddof=1) / math.sqrt(len(losses))) if len(losses) > 1 else 0.
        candidates.append(dict(spec=spec, loss=loss, se=se, oof=oof, fold_losses=losses))
    best = min(candidates, key=lambda item: item['loss'])
    eligible = [item for item in candidates if item['loss'] <= best['loss'] + best['se']]
    complexity = {'ridge': 0, 'gam': 1, 'hgb': 2, 'xgb': 2}
    chosen = min(eligible, key=lambda item: (complexity[item['spec']['kind']], item['loss']))
    arm_train = train.loc[train.Treatment == arm]
    model = fit_checked(feature_pipeline(arm_train, features, chosen['spec']), arm_train[features], arm_train.Y, chosen['spec'], logger, f'm{arm}')
    return dict(model=model, spec=chosen['spec'], oof=chosen['oof'], provenance=provenance,
                tuning=[{k: v for k, v in item.items() if k != 'oof'} for item in candidates])


In [ ]:


def football_cells(frame):
    distance = np.digitize(frame.distance.to_numpy(), [6, 12, 18, 24, 30])
    zone = np.digitize(frame.y.to_numpy(), [-9.16, 9.16])
    pressure = np.minimum(frame.defenders_3m.fillna(0).astype(int).to_numpy(), 2)
    return np.asarray([f'{a}:{b}:{c}' for a, b, c in zip(distance, zone, pressure)])


def learn_domain(train, portable_oof, threshold, config):
    in_gate = (1 - np.asarray(portable_oof)) > threshold
    controls = train.loc[in_gate & (train.Treatment.to_numpy() == 0)].copy()
    controls['football_cell'] = football_cells(controls)
    distinct = controls.drop_duplicates(['original_match_id', 'decision_row_id'])
    counts = distinct.groupby('football_cell').agg(decisions=('decision_row_id', 'nunique'),
                                                matches=('original_match_id', 'nunique'))
    cells = sorted(counts.index[(counts.decisions >= config['donor_min']) &
                                (counts.matches >= config['donor_match_min'])].tolist())
    return dict(threshold=float(threshold), cells=cells, donor_counts=counts.to_dict('index'))


def domain_mask(frame, portable_probability, domain):
    # No evaluation treatment or outcomes here: S(X) is common to both arms.
    return ((1 - np.asarray(portable_probability)) > domain['threshold']) & np.isin(football_cells(frame), domain['cells'])


def support_diagnostics(rows, config):
    selected = rows.loc[rows.support].copy()
    treated = selected.loc[selected.Treatment == 1]
    control = selected.loc[selected.Treatment == 0]
    weights = control.e.to_numpy() / (1 - control.e.to_numpy())
    sum_w = float(weights.sum())
    unique_weights = pd.Series(weights, index=control.decision_row_id.astype(str)).groupby(level=0).sum().to_numpy()
    ess = sum_w ** 2 / float(np.sum(unique_weights ** 2)) if len(unique_weights) and np.sum(unique_weights ** 2) else 0.
    donor_share = float(unique_weights.max() / sum_w) if sum_w else float('inf')
    by_match = pd.Series(weights, index=control.original_match_id.astype(str)).groupby(level=0).sum()
    match_share = float(by_match.max() / sum_w) if sum_w else float('inf')
    balance = {}
    for name in config['feature_names']:
        if not len(treated) or not len(control) or not sum_w:
            balance[name] = float('inf')
        elif pd.api.types.is_numeric_dtype(selected[name]):
            a = treated[name].to_numpy(dtype=float)
            b = control[name].to_numpy(dtype=float)
            valid_a, valid_b = np.isfinite(a), np.isfinite(b)
            missing_a = float((~valid_a).mean())
            missing_b = float(np.average(~valid_b, weights=weights))
            missing_scale = np.sqrt((missing_a * (1 - missing_a) + missing_b * (1 - missing_b)) / 2)
            balance[name + '__missing'] = float(abs(missing_a - missing_b) / missing_scale) if missing_scale > 1e-12 else (0. if missing_a == missing_b else float('inf'))
            if not valid_a.any() or not valid_b.any():
                balance[name] = 0. if not valid_a.any() and not valid_b.any() else float('inf')
                continue
            am, bm = np.mean(a[valid_a]), np.average(b[valid_b], weights=weights[valid_b])
            av = np.var(a[valid_a])
            bv = np.average((b[valid_b] - bm) ** 2, weights=weights[valid_b])
            scale = np.sqrt((av + bv) / 2)
            balance[name] = float(abs(am - bm) / scale) if scale > 1e-12 else (0. if abs(am - bm) < 1e-12 else float('inf'))
        else:
            for level in sorted(selected[name].astype(str).unique()):
                a = (treated[name].astype(str) == level).to_numpy(dtype=float)
                b = (control[name].astype(str) == level).to_numpy(dtype=float)
                am, bm = a.mean(), np.average(b, weights=weights)
                scale = np.sqrt((am * (1 - am) + bm * (1 - bm)) / 2)
                balance[f'{name}={level}'] = float(abs(am - bm) / scale) if scale > 1e-12 else (0. if abs(am - bm) < 1e-12 else float('inf'))
    key_max = max([v for k, v in balance.items() if k.split('=')[0].split('__')[0] in config['key_balance_features']] or [float('inf')])
    other_max = max([v for k, v in balance.items() if k.split('=')[0].split('__')[0] not in config['key_balance_features']] or [0.])
    matches = int(treated.original_match_id.nunique())
    gates = dict(shot_count=len(treated) >= config['min_shots'], match_count=matches >= config['min_matches'],
                 effective_donors=ess >= max(config['min_ess'], config['ess_shot_fraction'] * len(treated)),
                 key_balance=key_max <= config['key_smd_max'], other_balance=other_max <= config['other_smd_max'],
                 donor_concentration=donor_share <= config['max_weight_share'],
                 match_concentration=match_share <= config['max_match_weight_share'])
    return dict(passed=all(gates.values()), gates=gates, n_shots=len(treated), n_controls=len(control),
                n_treated_matches=matches, shot_retention=len(treated) / max(1, int((rows.Treatment == 1).sum())),
                control_ess=ess, weight_sum=sum_w, maximum_weight=float(weights.max()) if len(weights) else 0,
                weight_to_treated_count_ratio=sum_w / max(1, len(treated)),
                max_weight_share=donor_share, max_match_weight_share=match_share,
                key_max_smd=key_max, other_max_smd=other_max, balance=balance)


def select_domain(train, portable_oof, full_oof, config):
    diagnostics, passing = [], []
    for threshold in config['support_thresholds']:
        domain = learn_domain(train, portable_oof, threshold, config)
        candidate = train.copy()
        candidate['e'] = full_oof
        candidate['support'] = domain_mask(candidate, portable_oof, domain)
        diagnostic = support_diagnostics(candidate, config)
        diagnostics.append(dict(threshold=threshold, **diagnostic))
        if diagnostic['passed']:
            passing.append((diagnostic['n_shots'], -threshold, domain))
    if passing:
        domain = max(passing, key=lambda item: item[:2])[2]
        domain['selection_passed'] = True
    else:
        # Fail closed scientifically. An explicit diagnostic branch is kept for
        # learning why candidates failed, never promoted to a headline ATT.
        broad = config['support_thresholds'][0]
        domain = learn_domain(train, portable_oof, broad, config)
        domain['selection_passed'] = False
    domain['candidate_diagnostics'] = diagnostics
    return domain


def fit_study(frame, config=None, frozen_design=None, logger=None):
    """Strict match-OOF nuisances and honest X-domain, shared across arms.

    frozen_design is the persisted EURO portable model/domain/settings. It is
    used unmodified for confirmation; only full-adjustment nuisances refit.
    """
    config = scientific_config(config)
    fit_m1 = not config.get('att_only', False) or config.get('diagnostic_m1', False)
    rows = prepare_analysis(frame, config)
    rows['e'] = rows['m0'] = rows['m1'] = rows['portable_e'] = np.nan
    rows['support'] = False
    rows['outer_fold'] = -1
    rows['domain_selection_passed'] = False
    folds = grouped_folds(rows, config['outer_folds'], config['seed'])
    provenance, fold_models, domains, model_scores = [], [], [], []
    for fold, (train_index, test_index) in enumerate(folds):
        train, test = rows.iloc[train_index].copy(), rows.iloc[test_index].copy()
        log_event(logger, 'outer_fold_start', stage='nuisances', fold=fold,
                  train_matches=train.original_match_id.nunique(), test_matches=test.original_match_id.nunique())
        frozen_models = frozen_design['model_specs'] if frozen_design is not None else {}
        propensity = fit_propensity(train, config, config['feature_names'], frozen_models.get('e'), logger)
        m0 = fit_outcome(train, config, 0, propensity['oof'], frozen_models.get('m0'), logger)
        m1 = fit_outcome(train, config, 1, propensity['oof'], frozen_models.get('m1'), logger) if fit_m1 else None
        full_e = predict_propensity(propensity, test, config['feature_names'], logger, 'e_outer_prediction')
        if frozen_design is None:
            # Portable logistic avoids cohort/team identifiers changing S(X).
            portable = fit_propensity(train, config, config['support_features'],
                                      {'kind': 'logistic', 'C': .2}, logger, 'support', require_oof=True)
            domain = select_domain(train, portable['oof'], propensity['oof'], config)
            portable_e = predict_propensity(portable, test, config['support_features'])
            provenance.extend(portable['provenance'])
        else:
            portable = frozen_design['portable_model']
            domain = frozen_design['domain']
            portable_e = predict_propensity(portable, test, config['support_features'])
        raw0 = predict_checked(m0['model'], test[config['feature_names']], logger=logger, stage='m0_outer_prediction')
        raw1 = predict_checked(m1['model'], test[config['feature_names']], logger=logger, stage='m1_outer_prediction') if fit_m1 else np.full(len(test), np.nan)
        rows.loc[test_index, 'e'] = full_e
        rows.loc[test_index, 'portable_e'] = portable_e
        rows.loc[test_index, 'm0'] = np.clip(raw0, -1, 1)
        rows.loc[test_index, 'm1'] = np.clip(raw1, -1, 1)
        rows.loc[test_index, 'support'] = domain_mask(test, portable_e, domain)
        gate = (1 - portable_e) > domain['threshold']
        cells = football_cells(test)
        rows.loc[test_index, 'football_cell'] = cells
        rows.loc[test_index, 'support_threshold'] = domain['threshold']
        rows.loc[test_index, 'support_reason'] = np.where(~gate, 'portable_score_gate',
            np.where(~np.isin(cells, domain['cells']), 'insufficient_cross_match_cell_donors',
                'supported_X_domain' if domain.get('selection_passed', True) else 'exploratory_domain_failed_diagnostics'))
        rows.loc[test_index, 'outer_fold'] = fold
        rows.loc[test_index, 'domain_selection_passed'] = domain.get('selection_passed', True)
        rows.loc[test_index, 'm0_clipped'] = (raw0 < -1) | (raw0 > 1)
        rows.loc[test_index, 'm1_clipped'] = (raw1 < -1) | (raw1 > 1)
        for name, bundle in [('e', propensity), ('m0', m0), ('m1', m1)]:
            if bundle is None:
                continue
            provenance.extend(bundle['provenance'])
            provenance.append(provenance_record(train, test, name + '_outer', fold, bundle['spec']))
        domains.append(domain)
        model_scores.append(dict(fold=fold, e=propensity['tuning'], m0=m0['tuning'], m1=m1['tuning'] if fit_m1 else []))
        fold_models.append(dict(fold=fold, e=propensity['spec'], m0=m0['spec'], m1=m1['spec'] if fit_m1 else None))
        log_event(logger, 'outer_fold_end', stage='nuisances', fold=fold, predicted_rows=len(test))
    required = ['e', 'm0', 'portable_e'] + (['m1'] if fit_m1 else [])
    if not np.isfinite(rows[required].to_numpy()).all():
        raise AssertionError('OOF nuisance predictions incomplete/nonfinite')
    diagnostics = support_diagnostics(rows, config)
    diagnostics['fold_support_selection_passed'] = all(domain.get('selection_passed', True) for domain in domains)
    diagnostics['passed'] = diagnostics['passed'] and diagnostics['fold_support_selection_passed']
    design = dict(domains=domains, model_specs_by_fold=fold_models, config=config)
    # Freeze/refit only once outside resampling; bootstrap does not save models.
    return dict(rows=rows, diagnostics=diagnostics, design=design, provenance=provenance,
                model_validation=model_scores, m1_status='fitted_diagnostic' if fit_m1 else 'omitted_att_only')


In [ ]:


def fit_frozen_design(frame, config, logger=None):
    config = scientific_config(config)
    train = prepare_analysis(frame, config)
    propensity = fit_propensity(train, config, config['feature_names'], logger=logger)
    portable = fit_propensity(train, config, config['support_features'],
                              {'kind': 'logistic', 'C': .2}, logger, 'final_support', require_oof=True)
    m0 = fit_outcome(train, config, 0, propensity['oof'], logger=logger)
    m1 = fit_outcome(train, config, 1, propensity['oof'], logger=logger)
    domain = select_domain(train, portable['oof'], propensity['oof'], config)
    development_rows = train.copy()
    development_rows['e'] = propensity['oof']
    development_rows['portable_e'] = portable['oof']
    development_rows['m0'], development_rows['m1'] = m0['oof'], m1['oof']
    development_rows['m0_clipped'] = development_rows['m1_clipped'] = False
    development_rows['support'] = domain_mask(train, portable['oof'], domain)
    # These OOF candidate predictions support final Euro selection diagnostics;
    # a global Euro-selected specification is not held out from every Euro match.
    selection = dict(rows=development_rows,
                     support_diagnostics=support_diagnostics(development_rows, config),
                     tuning=dict(e=propensity['tuning'], m0=m0['tuning'], m1=m1['tuning']),
                     provenance=propensity['provenance'] + m0['provenance'] + m1['provenance'],
                     role='all_Euro_development_selection_not_confirmation')
    return dict(development_selection=selection, portable_model=portable, domain=domain, config=config,
                caliper_width=.2 * float(np.std(np.log(portable['oof'] / (1-portable['oof'])), ddof=1)),
                model_specs=dict(e=propensity['spec'], m0=m0['spec'], m1=m1['spec']),
                training_match_ids=sorted(train.original_match_id.astype(str).unique()),
                source_commit=SOURCE_COMMIT)


def outcome_validation(rows):
    summaries, calibration = [], []
    for arm, column in [(0, 'm0'), (1, 'm1')]:
        if column not in rows or rows[column].isna().all():
            continue
        arm_rows = rows.loc[rows.Treatment == arm].copy()
        arm_rows['weight'] = arm_rows.e / (1 - arm_rows.e) if arm == 0 else 1.
        for name, mask in [('all', np.ones(len(arm_rows), bool)),
                           ('supported', arm_rows.support.to_numpy()),
                           ('influential_top_1pct', arm_rows.weight >= arm_rows.weight.quantile(.99))]:
            sample = arm_rows.loc[mask]
            if not len(sample):
                continue
            residual = sample.Y - sample[column]
            summaries.append(dict(arm=arm, population=name, n=len(sample),
                                  prediction_min=sample[column].min(), prediction_max=sample[column].max(),
                                  mse=np.mean(residual ** 2), weighted_mse=np.average(residual ** 2, weights=sample.weight),
                                  residual_mean=residual.mean(), weighted_residual=np.average(residual, weights=sample.weight),
                                  clipped_fraction=sample[f'{column}_clipped'].mean()))
        arm_rows['prediction_bin'] = pd.qcut(arm_rows[column], q=10, duplicates='drop')
        if arm_rows['prediction_bin'].isna().all():
            arm_rows['prediction_bin'] = 'constant_prediction'
        for interval, sample in arm_rows.groupby('prediction_bin', observed=True):
            calibration.append(dict(arm=arm, bin=str(interval), n=len(sample),
                                    predicted=np.average(sample[column], weights=sample.weight),
                                    observed=np.average(sample.Y, weights=sample.weight)))
    return pd.DataFrame(summaries), pd.DataFrame(calibration)


def refit_on_fixed_population(baseline_rows, config, outcome_spec=None, logger=None, frozen_specs=None):
    """Ablation/model sensitivity without moving the primary target IDs."""
    rows = baseline_rows.copy()
    provenance = []
    frozen_specs = frozen_specs or {}
    for fold in sorted(rows.outer_fold.unique()):
        test_mask = rows.outer_fold == fold
        train, test = rows.loc[~test_mask].copy(), rows.loc[test_mask].copy()
        if outcome_spec is None:
            e_fit = fit_propensity(train, config, config['feature_names'], frozen_spec=frozen_specs.get('e'), logger=logger)
            e_predict = predict_propensity(e_fit, test, config['feature_names'], logger, 'fixed_population_e_prediction')
            e_oof = e_fit['oof']
            provenance.extend(e_fit['provenance'])
        else:
            # Outcome-only sensitivity preserves every original propensity.
            e_predict, e_oof = test.e.to_numpy(), train.e.to_numpy()
        m0 = fit_outcome(train, config, 0, e_oof, frozen_spec=outcome_spec or frozen_specs.get('m0'), logger=logger)
        provenance.extend(m0['provenance'])
        rows.loc[test_mask, 'e'] = e_predict
        raw = predict_checked(m0['model'], test[config['feature_names']], logger=logger, stage='fixed_population_m0_prediction')
        rows.loc[test_mask, 'm0'] = np.clip(raw, -1, 1)
        rows.loc[test_mask, 'm0_clipped'] = (raw < -1) | (raw > 1)
        provenance.append(provenance_record(train, test, 'fixed_population_m0_outer', fold, m0['spec']))
    assert rows.loc[rows.support, 'decision_row_id'].tolist() == baseline_rows.loc[baseline_rows.support, 'decision_row_id'].tolist()
    return rows, provenance


## Row-level ATT, uncertainty, sensitivity and controlled simulations

The count-normalized ATT score is `sum[T*(Y-m0) - (1-T)*e/(1-e)*(Y-m0)] / n_shots` on the shared domain. Row contributions and residual correction are reproducible. Raw, regression, IPW, AIPW and actual pair/reuse-weighted matching compare the same target IDs.

Match-cluster refits preserve original groups even when bootstrap draws repeat. Finite extreme draws are retained; failed attempts are logged. Development repeats learned selection; confirmation holds the Euro domain fixed. Hard support selection can be nonregular, so simulation coverage is required rather than assumed.

ATT DVDS uses the general-outcome construction for {-1,0,+1}, quantiles plus independently learned tails, Gamma=1 equivalence and oracle atom/tie checks. Identification bounds and statistical endpoint intervals are different objects. Published iid theory is not a guarantee for clustered ternary rewards. Source: https://jacobdorn.info/files/DVDS.pdf (Appendix A).

Controlled simulations call these production functions and compare against truth in the actual selected treated population. Bias, RMSE, coverage, false-positive rates, overlap retention, donor ESS and failure rates are reported with Monte Carlo error. Unexecuted scenarios are never presented as findings.


In [ ]:
"""Notebook-owned inference definitions; embedded into the tagged scientific core."""
import hashlib
import json
import math
import os
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.stats import norm, t as student_t
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def _inference_rows(rows):
    out = rows.copy()
    aliases = {"T": ["Z"], "Y": ["Y15", "Y_reward_15"], "e": ["e_hat"],
               "m0": ["mu0_hat"], "m1": ["mu1_hat"], "support": ["supported"],
               "original_match_id": ["match_id"]}
    for name, candidates in aliases.items():
        if name not in out:
            for candidate in candidates:
                if candidate in out:
                    out[name] = out[candidate]
                    break
    needed = {"T", "Y", "e", "m0", "support", "original_match_id"}
    missing = needed - set(out.columns)
    if missing:
        raise ValueError(f"Inference rows missing {sorted(missing)}")
    for col in ["T", "Y", "e", "m0"]:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["support"] = out["support"].fillna(False).astype(bool)
    # Unknown outcomes cannot enter the estimator; retain them in upstream flow audits.
    sample = out.loc[out["support"] & out["T"].isin([0, 1]) & out["Y"].notna()].copy()
    if not np.isfinite(sample[["Y", "e", "m0"]].to_numpy(float)).all():
        raise ValueError("Nonfinite supported nuisance/outcome predictions")
    if not sample["e"].between(0, 1, inclusive="neither").all():
        raise ValueError("Supported propensity must be strictly between zero and one")
    if sample["original_match_id"].isna().any():
        raise ValueError("Missing original match ID")
    return sample


def _weight_ess(weights):
    weights = np.asarray(weights, dtype=float)
    return float(weights.sum() ** 2 / np.dot(weights, weights)) if np.dot(weights, weights) > 0 else 0.0


def att_estimate(rows):
    """Count-normalized ATT with auditable row contributions and match scores."""
    sample = _inference_rows(rows)
    t = sample["T"].to_numpy(int)
    n1, n0 = int(t.sum()), int((t == 0).sum())
    if not n1 or not n0:
        raise ValueError("Supported ATT requires both arms")
    sample["att_weight"] = np.where(t == 0, sample["e"] / (1 - sample["e"]), 0.0)
    sample["residual"] = sample["Y"] - sample["m0"]
    sample["att_numerator"] = np.where(t == 1, sample["residual"], -sample["att_weight"] * sample["residual"])
    sample["att_contribution"] = sample["att_numerator"] / n1
    estimate = float(sample["att_contribution"].sum())
    treated = sample.loc[sample["T"].eq(1)]
    controls = sample.loc[sample["T"].eq(0)]
    w = controls["att_weight"].to_numpy(float)
    sum_w = float(w.sum())
    correction = float((controls["att_weight"] * controls["residual"]).sum() / n1)
    observed = float(treated["Y"].mean())
    predicted = float(treated["m0"].mean())
    assert np.isclose(estimate, observed - predicted - correction, rtol=1e-12, atol=1e-12)
    sample["ratio_score"] = sample["att_numerator"] - sample["T"] * estimate
    # The sampling registry includes eligible matches with no supported score.
    # Unsupported rows contribute zero, not a smaller sampling population.
    registry_rows = rows.loc[rows['eligible'].fillna(False).astype(bool)] if 'eligible' in rows else rows
    match_column = 'original_match_id' if 'original_match_id' in registry_rows else 'match_id'
    if registry_rows[match_column].isna().any():
        raise ValueError('Missing original match ID in eligible registry')
    registry = pd.Index(registry_rows[match_column].unique(), name='original_match_id')
    if not set(sample.original_match_id).issubset(set(registry)):
        raise ValueError('Supported sample contains ineligible original matches')
    cluster_scores = sample.groupby('original_match_id', sort=False)['ratio_score'].sum().reindex(registry, fill_value=0.)
    g = len(cluster_scores)
    if g <= 1:
        raise ValueError('Cluster inference requires at least two eligible original matches')
    se = float(np.sqrt(g / (g - 1) * np.square(cluster_scores).sum()) / n1)
    critical = float(student_t.ppf(.975, g - 1))
    interval = [estimate - critical * se, estimate + critical * se]
    match_weights = controls.groupby("original_match_id")["att_weight"].sum()
    return {"att": estimate, "rows": sample, "n_shots": n1, "n_controls": n0,
            "n_matches": g, "eligible_matches": g,
            "support_bearing_matches": int(sample.original_match_id.nunique()),
            "treated_bearing_matches": int(treated.original_match_id.nunique()),
            "observed_treated_mean": observed,
            "predicted_continuation_mean": predicted, "residual_correction": correction,
            "implied_continuation_mean": predicted + correction,
            "sum_control_weights": sum_w, "control_ess": _weight_ess(w),
            "max_donor_weight_share": float(w.max() / sum_w),
            "max_match_weight_share": float(match_weights.max() / sum_w),
            "cluster_score_se": se, "cluster_scores": cluster_scores.to_frame("score"),
            "cluster_score_df": g - 1, "cluster_score_critical": critical,
            "score_interval": interval,
            "score_interval_kind": "conditional_approximate_match_cluster_t",
            "sampling_registry": "all_eligible_original_matches",
            "hajek_aipw_att": observed - predicted - float(np.dot(w, controls["residual"]) / sum_w),
            "normalization": "treated_count", "diagnostic_interval_only": [estimate - 1.96 * se, estimate + 1.96 * se]}


def _inference_preprocessor(frame, features):
    features = [c for c in features if c in frame]
    numeric = [c for c in features if pd.api.types.is_numeric_dtype(frame[c])]
    categorical = [c for c in features if c not in numeric]
    blocks = []
    if numeric:
        blocks.append(("numeric", Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                                            ("scale", StandardScaler())]), numeric))
    if categorical:
        blocks.append(("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                                                ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical))
    if not blocks:
        raise ValueError("At least one inference feature is required")
    return ColumnTransformer(blocks, remainder="drop", sparse_threshold=0)


def _external_match_pairs(treated, controls, features, k):
    prep = _inference_preprocessor(controls, features)
    xc = prep.fit_transform(controls[features])
    xt = prep.transform(treated[features])
    if not len(controls) or int(k) < 1:
        raise ValueError("Matching requires donors")
    # k plus the largest within-match donor count is sufficient to find k
    # external candidates, when they exist. No same-match trajectory donors.
    search_count = min(len(controls), int(k) + int(controls.original_match_id.value_counts().max()))
    distance, indices = NearestNeighbors(n_neighbors=search_count).fit(xc).kneighbors(xt)
    shot_ids = treated.get("decision_row_id", pd.Series(treated.index, index=treated.index)).to_numpy()
    donor_ids = controls.get("decision_row_id", pd.Series(controls.index, index=controls.index)).to_numpy()
    donor_matches = controls.original_match_id.to_numpy()
    shot_matches = treated.original_match_id.to_numpy()
    records = []
    for query in range(len(treated)):
        allowed = donor_matches[indices[query]] != shot_matches[query]
        selected = indices[query][allowed][:int(k)]
        selected_distances = distance[query][allowed][:int(k)]
        if not len(selected):
            raise ValueError('No external-match donor for a primary target; matching cannot silently drop it')
        records.extend(dict(shot_id=shot_ids[query], donor_id=donor_ids[j],
                            shot_original_match_id=shot_matches[query], donor_original_match_id=donor_matches[j],
                            distance=float(d), pair_weight=1/len(selected), donor_outcome=float(controls.Y.iloc[j]))
                       for j, d in zip(selected, selected_distances))
    pairs = pd.DataFrame(records)
    reuse = pairs.groupby("donor_id", sort=False).agg(reuse_count=("pair_weight", "size"),
                                                      matching_weight=("pair_weight", "sum"),
                                                      donor_outcome=("donor_outcome", "first")).reset_index()
    matching = float(treated["Y"].mean() - (reuse["matching_weight"] * reuse["donor_outcome"]).sum() / len(treated))
    assert set(pairs.shot_id) == set(shot_ids)
    assert (pairs.shot_original_match_id != pairs.donor_original_match_id).all()
    assert np.isclose(reuse.matching_weight.sum(), len(treated))
    return matching, pairs, reuse


In [ ]:


def att_comparators(rows, feature_names=None, k=3):
    """Same-target comparators; actual external pair/reuse weights are exported."""
    sample = _inference_rows(rows)
    primary = att_estimate(sample)
    treated = sample.loc[sample['T'].eq(1)].copy()
    controls = sample.loc[sample['T'].eq(0)].copy()
    features = feature_names or [c for c in ['x', 'y', 'distance', 'nearest_defender'] if c in sample]
    matching, pairs, reuse = _external_match_pairs(treated, controls, features, k)
    spatial_features = [c for c in ['x', 'y'] if c in sample]
    spatial, spatial_pairs, spatial_reuse = _external_match_pairs(treated, controls, spatial_features or features, k)
    w = controls["e"] / (1 - controls["e"])
    summaries = [("raw_difference", treated["Y"].mean() - controls["Y"].mean()),
                 ("outcome_regression_ATT", treated["Y"].mean() - treated["m0"].mean()),
                 ("IPW_Hajek_ATT", treated["Y"].mean() - np.average(controls["Y"], weights=w)),
                 ("AIPW_count_ATT", primary["att"]),
                 ("AIPW_Hajek_ATT", primary["hajek_aipw_att"]), ("NN_matching_ATT", matching),
                 ('NN_spatial_ATT', spatial)]
    return {"summary": pd.DataFrame([{"estimator": name, "att": float(value), "n_shots": len(treated),
                                       "n_controls": len(controls), "same_treated_ids": True,
                                       "inference": "descriptive_only" if name.startswith("NN") else "cluster_inference"}
                                      for name, value in summaries]), "pairs": pairs, "reuse": reuse,
            'spatial_pairs': spatial_pairs, 'spatial_reuse': spatial_reuse}


def _inference_log(logger, name, **fields):
    if logger is not None:
        logger.event(name, **fields)


def _inference_digest(frame, config, frozen_design=None):
    content = pd.util.hash_pandas_object(frame.astype(str), index=True).values.tobytes()
    import importlib.metadata
    import joblib
    versions = {}
    for package in ["numpy", "pandas", "scipy", "scikit-learn", "joblib"]:
        versions[package] = importlib.metadata.version(package)
    payload = json.dumps({"config": config, "design_hash": joblib.hash(frozen_design),
                          "scientific": globals().get("CORE_SOURCE_HASH", "unextracted_source"),
                          "provenance_hash": config.get("provenance_hash"),
                          "versions": versions}, sort_keys=True, default=str)
    return hashlib.sha256(content + payload.encode()).hexdigest()[:20]


def _atomic_inference_json(path, payload):
    from PublicCausal.notebook_runtime import atomic_write_json, json_safe, stable_hash as envelope_hash
    normalized = json_safe(payload)
    atomic_write_json(Path(path), {"schema": 1, "payload": normalized, "payload_hash": envelope_hash(normalized)})


def _load_inference_checkpoint(path):
    from PublicCausal.notebook_runtime import stable_hash as envelope_hash
    saved = json.loads(Path(path).read_text())
    if not isinstance(saved, dict) or saved.get("schema") != 1 or "payload" not in saved:
        raise ValueError("Inference checkpoint envelope is missing or incompatible")
    if saved.get("payload_hash") != envelope_hash(saved["payload"]):
        raise ValueError("Inference checkpoint payload checksum mismatch")
    return saved["payload"]


def resample_original_matches(frame, seed, stratify="cohort"):
    """Draw whole matches; duplicates never become independent validation groups."""
    out = frame.copy()
    if "original_match_id" not in out:
        out["original_match_id"] = out["match_id"]
    if "original_decision_row_id" not in out:
        out["original_decision_row_id"] = out.get("decision_row_id", pd.Series(out.index, index=out.index))
    rng = np.random.default_rng(seed)
    matches = out[["original_match_id"] + ([stratify] if stratify in out else [])].drop_duplicates()
    if matches["original_match_id"].duplicated().any():
        raise ValueError("A match cannot belong to multiple bootstrap strata")
    strata = list(matches.groupby(stratify, dropna=False)) if stratify in matches else [(None, matches)]
    parts = []
    draw = 0
    for _, group in strata:
        ids = group["original_match_id"].to_numpy()
        for match in rng.choice(ids, size=len(ids), replace=True):
            part = out.loc[out["original_match_id"].eq(match)].copy()
            part["bootstrap_draw_id"] = draw
            part["match_id"] = f"{match}__draw_{draw}"
            parts.append(part)
            draw += 1
    return pd.concat(parts, ignore_index=True)


def _compact_inference_diagnostics(diagnostics):
    return {name: value for name, value in diagnostics.items()
            if isinstance(value, (bool, int, float, str, np.number)) or value is None}


def _worker_progress(config, task_id):
    if not config.get('log_run_dir'):
        return None
    from PublicCausal.notebook_runtime import WorkerProgressLogger
    return WorkerProgressLogger(config['log_run_dir'], config.get('run_id', 'public-study'), task_id)


def _bootstrap_attempt(frame, config, fit_fn, frozen_design, stats_fn, attempt):
    replicate_seed = int(np.random.SeedSequence([int(config.get("seed", 42)), 1907, attempt]).generate_state(1)[0])
    started = time.monotonic()
    record = {"attempt": attempt, "seed": replicate_seed}
    progress = _worker_progress(config, f'bootstrap_{attempt}_seed_{replicate_seed}')
    try:
        draw = resample_original_matches(frame, replicate_seed)
        arguments = dict(frozen_design=frozen_design)
        if progress is not None:
            arguments['logger'] = progress
        fitted = fit_fn(draw, dict(config, seed=replicate_seed), **arguments)
        estimate = att_estimate(fitted["rows"])
        record.update({"status": "ok", "att": estimate["att"], "n_shots": estimate["n_shots"],
                       "control_ess": estimate["control_ess"], "n_matches": estimate["n_matches"],
                       "diagnostics": _compact_inference_diagnostics(fitted.get("diagnostics", {}))})
        if stats_fn is not None:
            try:
                record["statistics"] = stats_fn(fitted["rows"])
                record["statistics_status"] = "ok"
            except Exception as exc:
                # Endpoint/subgroup failure cannot erase an otherwise valid primary ATT draw.
                record.update({"statistics_status": "failed", "statistics_error": repr(exc),
                               "statistics_traceback": traceback.format_exc()})
    except Exception as exc:
        record.update({"status": "failed", "error": repr(exc), "traceback": traceback.format_exc()})
    record["elapsed_seconds"] = time.monotonic() - started
    if progress is not None:
        progress.close('completed' if record['status'] == 'ok' else 'failed', attempt=attempt,
                       result_status=record['status'], error=record.get('error'))
    return record


In [ ]:


def _inference_disk_guard(config):
    if config.get("study_root"):
        from PublicCausal.notebook_runtime import check_disk_budget
        check_disk_budget(config["study_root"])


def cluster_bootstrap(frame, config, fit_fn, frozen_design=None, n_success=1000,
                      logger=None, checkpoint_dir=None, stats_fn=None, max_attempts=None):
    """Refit the actual study. Finite extreme/diagnostically poor draws are retained."""
    digest = _inference_digest(frame, config, frozen_design)
    checkpoint = Path(checkpoint_dir) / f"bootstrap_{digest}.json" if checkpoint_dir is not None else None
    records = _load_inference_checkpoint(checkpoint) if checkpoint is not None and checkpoint.exists() else []
    max_attempts = max_attempts or config.get("bootstrap_max_attempts") or max(2 * int(n_success), int(n_success) + 100)
    successes = sum(r["status"] == "ok" for r in records)
    workers = max(1, int(config.get("workers", 4)))
    from joblib import Parallel, delayed, parallel_config
    started_run = time.monotonic()
    while successes < n_success and len(records) < max_attempts:
        if len(records) % 50 < workers:
            _inference_disk_guard(config)
        batch_size = min(workers, int(n_success) - successes, int(max_attempts) - len(records))
        indices = list(range(len(records), len(records) + batch_size))
        if workers == 1:
            batch = [_bootstrap_attempt(frame, config, fit_fn, frozen_design, stats_fn, attempt) for attempt in indices]
        else:
            with parallel_config(backend="loky", inner_max_num_threads=1):
                batch = Parallel(n_jobs=workers)(delayed(_bootstrap_attempt)(frame, config, fit_fn, frozen_design, stats_fn, attempt)
                                                 for attempt in indices)
        for record in batch:
            records.append(record)
            successes += int(record["status"] == "ok")
            _inference_log(logger, "bootstrap_attempt", **record)
        if checkpoint is not None:
            _atomic_inference_json(checkpoint, records)
        if logger is not None:
            elapsed = time.monotonic() - started_run
            logger.progress("bootstrap", successes, int(n_success), attempts=len(records),
                            failures=len(records) - successes, workers=workers, elapsed_seconds=elapsed)
    attempts = pd.DataFrame(records)
    valid = attempts.loc[attempts["status"].eq("ok")].copy() if len(attempts) else pd.DataFrame()
    estimates = valid["att"].to_numpy(float) if len(valid) else np.array([])
    summary = {"status": "complete" if len(valid) >= n_success else "incomplete", "n_requested": int(n_success),
               "n_success": len(valid), "n_attempts": len(attempts), "n_failed": len(attempts) - len(valid),
               "ci_low": float(np.quantile(estimates, .025)) if len(estimates) else np.nan,
               "ci_high": float(np.quantile(estimates, .975)) if len(estimates) else np.nan,
               "bootstrap_se": float(np.std(estimates, ddof=1)) if len(estimates) > 1 else np.nan,
               "domain": "frozen" if frozen_design is not None else "refitted_procedure",
               "cache_digest": digest, "extreme_finite_draws_retained": True}
    return {"summary": summary, "attempts": attempts, "estimates": valid, **summary}


def leave_one_match_out(frame, config, fit_fn, frozen_design=None, logger=None, checkpoint_dir=None):
    key = "original_match_id" if "original_match_id" in frame else "match_id"
    point = att_estimate(fit_fn(frame, config, frozen_design=frozen_design)["rows"])["att"]
    digest = _inference_digest(frame, config, frozen_design)
    checkpoint = Path(checkpoint_dir) / f"lomo_{digest}.json" if checkpoint_dir is not None else None
    records = _load_inference_checkpoint(checkpoint) if checkpoint is not None and checkpoint.exists() else []
    complete = {r["removed_match"] for r in records}
    for match in frame[key].drop_duplicates():
        if str(match) in complete:
            continue
        record = {"removed_match": str(match), "original_att": point}
        try:
            fitted = fit_fn(frame.loc[~frame[key].eq(match)].copy(), config, frozen_design=frozen_design)
            estimate = att_estimate(fitted["rows"])
            record.update({"status": "ok", "att": estimate["att"], "delta": estimate["att"] - point,
                           "n_shots": estimate["n_shots"], "control_ess": estimate["control_ess"]})
        except Exception as exc:
            record.update({"status": "failed", "error": repr(exc), "traceback": traceback.format_exc()})
        records.append(record)
        _inference_log(logger, "lomo_refit", **record)
        if checkpoint is not None:
            _atomic_inference_json(checkpoint, records)
    return pd.DataFrame(records)


def ternary_quantile(probabilities, level):
    """Left generalized quantile, including atoms and exact CDF-boundary ties."""
    p = np.asarray(probabilities, dtype=float)
    if p.ndim == 1:
        p = p[None, :]
    if p.shape[1] != 3 or not np.isfinite(p).all() or (p < 0).any() or not np.allclose(p.sum(axis=1), 1):
        raise ValueError("Three probabilities in (-1,0,+1) order must sum to one")
    if not 0 < level < 1:
        raise ValueError("Quantile level must be strictly inside (0,1)")
    return np.array([-1., 0., 1.])[np.argmax(np.cumsum(p, axis=1) >= float(level), axis=1)]


def ternary_oracle_bounds(probabilities, gamma):
    """Exact three-point LP and CVaR references; not fitted uncertainty claims."""
    p = np.asarray(probabilities, dtype=float)
    if p.shape != (3,) or (p < 0).any() or not np.isclose(p.sum(), 1) or gamma < 1:
        raise ValueError("Invalid probability vector or Gamma")
    y = np.array([-1., 0., 1.])
    beta = 1 / (gamma + 1)
    lower_q = float(ternary_quantile(p, beta)[0])
    upper_q = float(ternary_quantile(p, 1 - beta)[0])
    mu = float(p @ y)
    cv_lower = lower_q + float(p @ np.minimum(y - lower_q, 0)) / beta
    cv_upper = upper_q + float(p @ np.maximum(y - upper_q, 0)) / beta
    lo = linprog(p * y, A_eq=p[None, :], b_eq=[1.], bounds=[(1 / gamma, gamma)] * 3, method="highs")
    hi = linprog(-p * y, A_eq=p[None, :], b_eq=[1.], bounds=[(1 / gamma, gamma)] * 3, method="highs")
    if not lo.success or not hi.success:
        raise RuntimeError("Oracle density-ratio LP failed")
    return {"lower": float(lo.fun), "upper": float(-hi.fun),
            "cvar_lower": mu / gamma + (1 - 1 / gamma) * cv_lower,
            "cvar_upper": mu / gamma + (1 - 1 / gamma) * cv_upper,
            "lower_ratios": lo.x, "upper_ratios": hi.x,
            "q_lower": lower_q, "q_upper": upper_q}


In [ ]:


def _ternary_probability_fit(train, evaluate, features, seed):
    classes = sorted(train["Y"].astype(int).unique())
    probabilities = np.zeros((len(evaluate), 3), dtype=float)
    if len(classes) == 1:
        probabilities[:, classes[0] + 1] = 1.0
        return probabilities
    pipeline = Pipeline([("prep", _inference_preprocessor(train, features)),
                         ("model", LogisticRegression(C=.2, max_iter=2000, random_state=seed))])
    pipeline.fit(train[features], train["Y"].astype(int))
    prediction = pipeline.predict_proba(evaluate[features])
    for j, value in enumerate(pipeline.named_steps["model"].classes_):
        probabilities[:, int(value) + 1] = prediction[:, j]
    return probabilities


def _dvds_fold_stages(frame, features, seed, folds=5):
    """Quantile training matches and tail-label matches are strictly disjoint."""
    data = frame.copy().reset_index(drop=True)
    if "original_match_id" not in data:
        data["original_match_id"] = data["match_id"]
    data["Y"] = pd.to_numeric(data["Y"], errors="coerce")
    if not data.loc[data["Y"].notna(), "Y"].isin([-1, 0, 1]).all():
        raise ValueError("DVDS ternary learner requires signed first-goal outcomes")
    if "outer_fold" in data:
        membership = data.groupby("original_match_id")["outer_fold"].nunique()
        if not membership.eq(1).all():
            raise ValueError("An original match appears in multiple outer folds")
        split_iter = [(np.flatnonzero(data["outer_fold"].ne(f)), np.flatnonzero(data["outer_fold"].eq(f)))
                      for f in sorted(data["outer_fold"].unique())]
    else:
        groups = data["original_match_id"]
        count = min(int(folds), groups.nunique())
        if count < 3:
            raise ValueError("DVDS separated stages need at least three original matches")
        split_iter = GroupKFold(n_splits=count).split(data, groups=groups)
    stages, provenance = [], []
    for fold, (train_pos, test_pos) in enumerate(split_iter):
        training = data.iloc[train_pos]
        controls = training.loc[training["T"].eq(0) & training["Y"].notna()]
        ids = controls["original_match_id"].drop_duplicates().to_numpy()
        if len(ids) < 2:
            raise ValueError("Each DVDS outer training partition needs two control-bearing matches")
        rng = np.random.default_rng(np.random.SeedSequence([seed, 881, fold]))
        ids = rng.permutation(ids)
        n_quantile = max(1, len(ids) // 2)
        quantile_ids, tail_ids = set(ids[:n_quantile]), set(ids[n_quantile:])
        quantile = controls.loc[controls["original_match_id"].isin(quantile_ids)]
        tail = controls.loc[controls["original_match_id"].isin(tail_ids)]
        test = data.iloc[test_pos]
        combined = pd.concat([tail, test], ignore_index=True)
        probabilities = _ternary_probability_fit(quantile, combined, features, seed + fold)
        preprocessor = _inference_preprocessor(tail, features)
        tail_x = preprocessor.fit_transform(tail[features])
        test_x = preprocessor.transform(test[features])
        stages.append({"fold": fold, "test_pos": test_pos, "tail_y": tail["Y"].to_numpy(float),
                       "tail_x": tail_x, "test_x": test_x, "tail_prob": probabilities[:len(tail)],
                       "test_prob": probabilities[len(tail):], "tail_cache": {}})
        q_ids = sorted(map(str, quantile_ids))
        r_ids = sorted(map(str, tail_ids))
        test_ids = sorted(test["original_match_id"].astype(str).unique())
        assert not (set(q_ids) & set(r_ids) or set(q_ids) & set(test_ids) or set(r_ids) & set(test_ids))
        provenance.append({"fold": fold, "quantile_train_matches": q_ids,
                           "tail_label_train_matches": r_ids, "evaluation_matches": test_ids,
                           "tail_preprocess_matches": r_ids, "quantile_n": len(quantile), "tail_n": len(tail)})
    return data, stages, provenance


def _dvds_predictions(data, stages, gamma, seed):
    n = len(data)
    rho = {"upper": np.empty(n), "lower": np.empty(n)}
    transformed = {"upper": np.empty(n), "lower": np.empty(n)}
    margins = []
    beta = 1 / (gamma + 1)
    for stage in stages:
        pos = stage["test_pos"]
        for side, level in [("upper", 1 - beta), ("lower", beta)]:
            q_train = ternary_quantile(stage["tail_prob"], level)
            q_test = ternary_quantile(stage["test_prob"], level)
            # Fit the actual transformed-tail response, not a tail from possibly wrong probabilities.
            gap_train = stage["tail_y"] - q_train
            # q(X) is already known from disjoint quantile-stage matches.
            # E[h|X] = q(X) + E[(Y-q(X))_+/-|X]/beta. Fit the actual
            # observed gap, without clipping it, and reuse identical quantile
            # patterns across Gamma values. This avoids refitting a scaled
            # version of the same tail-response model seven times per fold.
            tail_response = np.maximum(gap_train, 0) if side == "upper" else np.minimum(gap_train, 0)
            signature = side + hashlib.sha256(q_train.tobytes() + q_test.tobytes() + tail_response.tobytes()).hexdigest()
            if signature not in stage["tail_cache"]:
                model = HistGradientBoostingRegressor(max_leaf_nodes=7, max_iter=100,
                                                      learning_rate=.05, l2_regularization=1,
                                                      early_stopping=False, random_state=seed + stage["fold"])
                model.fit(stage["tail_x"], tail_response)
                stage["tail_cache"][signature] = model.predict(stage["test_x"])
            expected_tail = q_test + stage["tail_cache"][signature] / beta
            rho[side][pos] = data.iloc[pos]["m0"].to_numpy(float) / gamma + (1 - 1 / gamma) * expected_tail
            gap_test = data.iloc[pos]["Y"].to_numpy(float) - q_test
            h_test = q_test + (np.maximum(gap_test, 0) if side == "upper" else np.minimum(gap_test, 0)) / beta
            transformed[side][pos] = data.iloc[pos]["Y"].to_numpy(float) / gamma + (1 - 1 / gamma) * h_test
            cdf = np.cumsum(stage["test_prob"], axis=1)[:, :2]
            margins.extend(np.min(np.abs(cdf - level), axis=1).tolist())
    return rho, transformed, margins


def dvds_sensitivity(rows, feature_names, gammas=(1, 1.25, 1.5, 2, 3, 5, 10),
                     fit_rows=None, seed=42, tipping_tolerance=.01):
    """Supported ATT DVDS scores; clustered ternary inference requires validation."""
    # All eligible rows, including those outside the support domain, train nuisances.
    data = (fit_rows if fit_rows is not None else rows).copy().reset_index(drop=True)
    if "T" not in data:
        data["T"] = data["Z"]
    if "Y" not in data:
        data["Y"] = data["Y15"]
    if "m0" not in data:
        data["m0"] = data["mu0_hat"]
    if "e" not in data:
        data["e"] = data["e_hat"]
    if "support" not in data:
        data["support"] = data["supported"]
    if "original_match_id" not in data:
        data["original_match_id"] = data["match_id"]
    primary = att_estimate(data)
    valid = data["support"].fillna(False).astype(bool) & data["T"].isin([0, 1]) & data["Y"].notna()
    t = data["T"].eq(1).to_numpy()
    mask = valid.to_numpy()
    n1 = int((mask & t).sum())
    observed = float(data.loc[valid & data["T"].eq(1), "Y"].mean())
    values = sorted(set(float(g) for g in gammas))
    if min(values) < 1:
        raise ValueError("Gamma must be at least one")
    if any(g > 1 for g in values):
        data, stages, provenance = _dvds_fold_stages(data, feature_names, seed)
    else:
        stages, provenance = [], []
    result_rows, score_rows, lookup = [], [], {}

    def evaluate(gamma, save_scores=True):
        if gamma in lookup:
            return lookup[gamma]
        if gamma == 1:
            rho = {side: data["m0"].to_numpy(float) for side in ["upper", "lower"]}
            transformed = {side: data["Y"].to_numpy(float) for side in ["upper", "lower"]}
            margins = []
        else:
            rho, transformed, margins = _dvds_predictions(data, stages, gamma, seed)
        estimates, cluster_se = {}, {}
        for side in ["upper", "lower"]:
            weights = data["e"].to_numpy(float) / (1 - data["e"].to_numpy(float))
            contribution = np.where(t, rho[side], weights * (transformed[side] - rho[side]))
            theta = float(contribution[mask].sum() / n1)
            estimates[side] = observed - theta
            # Ratio score corresponds to ATT endpoint: y - phi0 - T*endpoint.
            numerator = np.where(t, data["Y"].to_numpy(float) - rho[side],
                                 -weights * (transformed[side] - rho[side]))
            score = numerator - t * estimates[side]
            cluster = pd.Series(score[mask]).groupby(data.loc[mask, "original_match_id"].to_numpy()).sum()
            count = len(cluster)
            cluster_se[side] = float(np.sqrt(count / (count - 1) * np.square(cluster).sum()) / n1) if count > 1 else np.nan
            if save_scores:
                scores = data.loc[mask, [c for c in ["decision_row_id", "original_match_id", "T", "Y", "e"] if c in data]].copy()
                scores["gamma"] = gamma
                scores["counterfactual_endpoint"] = side
                scores["rho"] = rho[side][mask]
                scores["transformed_outcome"] = transformed[side][mask]
                scores["att_endpoint_numerator"] = numerator[mask]
                scores["ratio_score"] = score[mask]
                score_rows.append(scores)
        answer = {"gamma": gamma, "att_lower": estimates["upper"], "att_upper": estimates["lower"],
                  "lower_cluster_score_se": cluster_se["upper"], "upper_cluster_score_se": cluster_se["lower"],
                  "min_quantile_boundary_margin": float(min(margins)) if margins else np.nan,
                  "fraction_margin_below_01": float(np.mean(np.asarray(margins) < .01)) if margins else 0.0,
                  "n_shots": n1, "finite_sample_crossing": estimates["upper"] > estimates["lower"],
                  "uncertainty": "cluster_score_diagnostic; bootstrap_coverage_not_assumed"}
        if gamma == 1:
            assert np.isclose(answer["att_lower"], primary["att"], rtol=1e-12, atol=1e-12)
            assert np.isclose(answer["att_upper"], primary["att"], rtol=1e-12, atol=1e-12)
        lookup[gamma] = answer
        return answer

    for gamma in values:
        result_rows.append(evaluate(gamma))
    grid_lower = np.array([row["att_lower"] for row in result_rows])
    grid_upper = np.array([row["att_upper"] for row in result_rows])
    monotonicity = {"lower_nonincreasing": bool(np.all(np.diff(grid_lower) <= 1e-10)),
                    "upper_nondecreasing": bool(np.all(np.diff(grid_upper) >= -1e-10)),
                    "any_endpoint_crossing": any(row["finite_sample_crossing"] for row in result_rows)}
    tipping = {"status": "already_includes_zero" if primary["att"] == 0 else "not_reached", "tolerance": tipping_tolerance}
    if primary["att"] != 0 and tipping_tolerance is not None:
        endpoint = "att_lower" if primary["att"] > 0 else "att_upper"
        direction = 1 if primary["att"] > 0 else -1
        for left, right in zip(values[:-1], values[1:]):
            if direction * lookup[right][endpoint] <= 0:
                lo, hi = left, right
                while hi - lo > tipping_tolerance:
                    middle = (lo + hi) / 2
                    if direction * evaluate(middle, save_scores=False)[endpoint] <= 0:
                        hi = middle
                    else:
                        lo = middle
                tipping = {"status": "bracketed", "gamma_low": lo, "gamma_high": hi,
                           "tolerance": tipping_tolerance, "meaning": "estimated identification endpoint; not significance",
                           "assumes_local_monotonicity": True,
                           "grid_monotonicity_passed": monotonicity["lower_nonincreasing"] and monotonicity["upper_nondecreasing"]}
                break
    return {"bounds": pd.DataFrame(result_rows), "rows": pd.concat(score_rows, ignore_index=True),
            "provenance": pd.DataFrame(provenance), "tipping": tipping,
            "monotonicity": monotonicity,
            "primary_att": primary["att"], "assumption": "marginal treatment-odds sensitivity inside frozen common domain",
            "limitations": "General DVDS identification; published iid binary/continuous efficiency does not establish clustered ternary coverage."}


In [ ]:


def dvds_cluster_bootstrap(frame, config, fit_fn, frozen_design=None, n_success=1000,
                           gammas=(1, 1.25, 1.5, 2, 3, 5, 10), logger=None, checkpoint_dir=None):
    def statistic(fitted_rows):
        result = dvds_sensitivity(fitted_rows, config["feature_names"], gammas=gammas,
                                  fit_rows=fitted_rows, seed=int(config.get("seed", 42)), tipping_tolerance=None)
        return result["bounds"][["gamma", "att_lower", "att_upper"]].to_dict("records")
    tagged_config = dict(config, inference_statistic="dvds", sensitivity_gammas=list(gammas))
    boot = cluster_bootstrap(frame, tagged_config, fit_fn, frozen_design=frozen_design,
                             n_success=n_success, logger=logger, checkpoint_dir=checkpoint_dir, stats_fn=statistic)
    endpoint_rows = []
    for record in boot["attempts"].to_dict("records"):
        if record["status"] == "ok" and record.get("statistics_status") == "ok":
            endpoint_rows.extend(dict(row, attempt=record["attempt"]) for row in record["statistics"])
    endpoint_frame = pd.DataFrame(endpoint_rows)
    intervals = []
    if len(endpoint_frame):
        for gamma, group in endpoint_frame.groupby("gamma"):
            intervals.append({"gamma": gamma, "lower_endpoint_ci_low": group.att_lower.quantile(.025),
                              "lower_endpoint_ci_high": group.att_lower.quantile(.975),
                              "upper_endpoint_ci_low": group.att_upper.quantile(.025),
                              "upper_endpoint_ci_high": group.att_upper.quantile(.975),
                              "identification_set_outer_ci_low": group.att_lower.quantile(.025),
                              "identification_set_outer_ci_high": group.att_upper.quantile(.975),
                              "n_success": len(group), "coverage": "simulation_required"})
    boot.update({"endpoint_draws": endpoint_frame, "endpoint_intervals": pd.DataFrame(intervals)})
    return boot


def simulation_scenarios(include_stresses=True):
    scenarios = []
    for sign in [-1, 0, 1]:
        for overlap in ["adequate", "weak"]:
            for e_correct, m_correct in [(True, True), (True, False), (False, True), (False, False)]:
                name = f"effect_{sign:+d}_{overlap}_e{int(e_correct)}m{int(m_correct)}"
                scenarios.append({"name": name, "effect_sign": sign, "overlap": overlap,
                                  "e_correct": e_correct, "m_correct": m_correct,
                                  "stress": None, "family": "base24"})
    if include_stresses:
        scenarios.extend([
            {"name": "visibility_selection", "effect_sign": 1, "overlap": "adequate", "e_correct": True,
             "m_correct": True, "stress": "visibility", "family": "stress", "identification_failure_expected": True},
            {"name": "quantile_boundary", "effect_sign": 0, "overlap": "adequate", "e_correct": True,
             "m_correct": True, "stress": "quantile_boundary", "family": "stress"}])
    return scenarios


def simulate_causal_matches(scenario, seed, n_matches=48, decisions_per_match=250):
    """Known potential means; correlated outcomes within independent match clusters.

    'Correctness' denotes data-generating membership in the prespecified logistic/
    linear classes, not a claim that finite-sample fitted nuisances equal truth.
    Visibility stress deliberately creates action-dependent observation selection.
    """
    rng = np.random.default_rng(seed)
    n = int(n_matches) * int(decisions_per_match)
    match = np.repeat(np.arange(int(n_matches)), int(decisions_per_match))
    u, v = rng.uniform(-1, 1, (2, n))
    score = rng.choice([-1., 0., 1.], n, p=[.25, .5, .25])
    strength = 1.2 if scenario["overlap"] == "adequate" else 4.5
    logit_e = -2.3 + strength * u + .5 * v + .25 * score
    if not scenario["e_correct"]:
        logit_e += 1.6 * np.sin(3 * u) + 1.2 * u * v
    e_true = 1 / (1 + np.exp(-logit_e))
    treatment = rng.binomial(1, e_true)
    positive0 = .06 + .005 * u + .005 * v
    negative0 = .008 + .002 * v
    if not scenario["m_correct"]:
        positive0 += .01 * np.sin(3 * u) + .005 * u * v
    if scenario.get("stress") == "quantile_boundary":
        # Gamma=2 beta=1/3: substantial conditional mass sits exactly at a quantile boundary.
        positive0 = np.where(u > 0, 1 / 3, .04)
        negative0 = np.full(n, .015)
    effect = float(scenario["effect_sign"]) * .02 * (1 + .25 * u)
    positive1 = positive0 + effect
    negative1 = negative0.copy()
    if ((positive0 < 0) | (positive1 < 0) | (negative0 < 0)
            | (positive0 + negative0 > 1) | (positive1 + negative1 > 1)).any():
        raise ValueError("Simulation outcome probabilities left the simplex")
    # A Gaussian copula preserves marginal potential means while inducing match dependence.
    cluster_noise = rng.normal(size=int(n_matches))
    potential_uniform = norm.cdf(.3 * cluster_noise[match] + np.sqrt(1 - .3 ** 2) * rng.normal(size=n))
    y0 = np.where(potential_uniform < negative0, -1, np.where(potential_uniform > 1 - positive0, 1, 0))
    y1 = np.where(potential_uniform < negative1, -1, np.where(potential_uniform > 1 - positive1, 1, 0))
    observed = np.where(treatment == 1, y1, y0)
    visibility_probability = np.full(n, .95)
    if scenario.get("stress") == "visibility":
        # Deliberately outcome/assignment-dependent selection: a test of the stated limitation.
        visibility_probability = np.clip(.85 - .25 * treatment - .3 * (potential_uniform > .9), .1, .95)
    visible = rng.uniform(size=n) < visibility_probability
    original_mean0, original_mean1 = positive0 - negative0, positive1 - negative1
    control_positive, control_negative = positive0.copy(), negative0.copy()
    target_mean0, target_mean1 = original_mean0.copy(), original_mean1.copy()
    observed_e = e_true.copy()
    if scenario.get("stress") == "visibility":
        # Conditional potential means for the actual selected treated population
        # integrate U under T=1,V=1, not the original unconditional uniform law.
        def visible_positive(probability, below, above):
            return below * np.maximum(probability - .1, 0) + above * np.minimum(probability, .1)
        treated_visibility = .9 * .60 + .1 * .30
        control_visibility = .9 * .85 + .1 * .55
        target_mean0 = (visible_positive(positive0, .60, .30) - .60 * negative0) / treated_visibility
        target_mean1 = (visible_positive(positive1, .60, .30) - .60 * negative1) / treated_visibility
        control_positive = visible_positive(positive0, .85, .55) / control_visibility
        control_negative = .85 * negative0 / control_visibility
        observed_e = treated_visibility * e_true / (treated_visibility * e_true + control_visibility * (1 - e_true))
    frame = pd.DataFrame({"x": 34 + 13 * u, "y": 8 * v,
                          "distance": np.sqrt((52.5 - (34 + 13 * u)) ** 2 + (8 * v) ** 2),
                          "angle": 20 + 12 * u + 2 * v,
                          "defenders_3m": rng.binomial(2, .5, n),
                          "nearest_defender": rng.uniform(.2, 3, n),
                          "second_defender": rng.uniform(1.5, 3, n),
                          "score_difference": score, "period": rng.choice([1, 2], n),
                          "time_seconds": rng.uniform(0, 2700, n),
                          "time_since_prev_action": rng.uniform(0, 8, n),
                          "prior_10s_actions": rng.poisson(3, n),
                          "prior_10s_progress": rng.normal(5, 3, n),
                          "position_role": rng.choice(["defender", "midfielder", "forward"], n),
                          "team_id": [f"team_{i % 12}" for i in match],
                          "opponent_id": [f"team_{(i + 1) % 12}" for i in match],
                          "cohort": np.where(match < n_matches // 2, "euro2020", "euro2024"),
                          "original_match_id": [f"sim_{i}" for i in match],
                          "match_id": [f"sim_{i}" for i in match],
                          "decision_row_id": [f"sim_{int(match[i])}:occasion_{i}" for i in range(n)],
                          "event_id": [f"sim_uuid_{i}" for i in range(n)],
                          "T": treatment, "Y": observed, "eligible": visible,
                          "true_y0": target_mean0, "true_y1": target_mean1,
                          "original_mean_y0": original_mean0, "original_mean_y1": original_mean1,
                          "true_control_p_minus": control_negative,
                          "true_control_p_zero": 1 - control_positive - control_negative,
                          "true_control_p_plus": control_positive,
                          "true_propensity": observed_e, "original_propensity": e_true,
                          "potential_y0": y0, "potential_y1": y1,
                          "visibility_probability": visibility_probability})
    return frame


In [ ]:


def _simulation_dvds_statistic(rows, features, seed):
    result = dvds_sensitivity(rows, features, gammas=[1, 2, 5], seed=seed, tipping_tolerance=None)
    return result["bounds"][["gamma", "att_lower", "att_upper"]].to_dict("records")


def _simulation_oracle_dvds(selected, gamma):
    p = selected[["true_control_p_minus", "true_control_p_zero", "true_control_p_plus"]].to_numpy(float)
    beta = 1 / (gamma + 1)
    values = np.array([-1., 0., 1.])
    mean = p @ values
    q_plus, q_minus = ternary_quantile(p, 1 - beta), ternary_quantile(p, beta)
    tail_plus = q_plus + (p * np.maximum(values[None, :] - q_plus[:, None], 0)).sum(axis=1) / beta
    tail_minus = q_minus + (p * np.minimum(values[None, :] - q_minus[:, None], 0)).sum(axis=1) / beta
    rho_plus = mean / gamma + (1 - 1 / gamma) * tail_plus
    rho_minus = mean / gamma + (1 - 1 / gamma) * tail_minus
    observed_mean = float(selected["true_y1"].mean())
    return observed_mean - float(rho_plus.mean()), observed_mean - float(rho_minus.mean())


def _simulation_one(scenario, replicate, config, fit_fn, checkpoint_dir,
                    n_matches, decisions_per_match, bootstrap_reps, learned_procedure, dvds=False):
    dataset_seed = int(np.random.SeedSequence([int(config.get("seed", 42)), 7123,
                                             int(hashlib.sha256(scenario["name"].encode()).hexdigest()[:8], 16),
                                             replicate]).generate_state(1)[0])
    record = {"scenario": scenario["name"], "family": scenario["family"],
              "replicate": replicate, "seed": dataset_seed,
              "effect_sign": scenario["effect_sign"], "overlap": scenario["overlap"],
              "e_correct_class": scenario["e_correct"], "m_correct_class": scenario["m_correct"],
              "learned_procedure": learned_procedure,
              "identification_failure_expected": bool(scenario.get("identification_failure_expected", False))}
    began = time.monotonic()
    progress = _worker_progress(config, f"simulation_{scenario['name']}_{replicate}")
    try:
        frame = simulate_causal_matches(scenario, dataset_seed, n_matches, decisions_per_match)
        fitting_config = dict(config, seed=dataset_seed, workers=1)
        if not learned_procedure:
            # Restrict classes only for controlled correctness experiments. The actual
            # production fitting/cross-fitting/support functions still execute.
            fitting_config.update(e_candidates=[{"kind": "logistic", "C": 1}],
                                  outcome_candidates=[{"kind": "ridge", "alpha": .1}])
        arguments = dict(frozen_design=None)
        if progress is not None:
            arguments['logger'] = progress
        fitted = fit_fn(frame, fitting_config, **arguments)
        estimate = att_estimate(fitted["rows"])
        fitted_rows = fitted["rows"]
        selected = fitted_rows.loc[fitted_rows["support"].astype(bool) & fitted_rows["T"].eq(1)]
        truth = float((selected["true_y1"] - selected["true_y0"]).mean())
        realized_truth = float((selected["potential_y1"] - selected["potential_y0"]).mean())
        record.update({"status": "ok", "att": estimate["att"], "truth": truth,
                       "realized_selected_truth": realized_truth, "error": estimate["att"] - truth,
                       "n_shots": estimate["n_shots"], "control_ess": estimate["control_ess"],
                       "retention": len(selected) / max(1, int((frame["T"].eq(1) & frame["eligible"]).sum())),
                       "diagnostics_passed": bool(fitted.get("diagnostics", {}).get("passed", False))})
        if bootstrap_reps:
            from functools import partial
            statistics = partial(_simulation_dvds_statistic, features=fitting_config["feature_names"], seed=dataset_seed) if dvds else None
            boot_dir = Path(checkpoint_dir) / f"bootstrap_{scenario['name']}_{replicate}"
            bootstrap = cluster_bootstrap(frame, fitting_config, fit_fn, n_success=bootstrap_reps,
                                          checkpoint_dir=boot_dir, logger=None, stats_fn=statistics)
            low, high = bootstrap["ci_low"], bootstrap["ci_high"]
            record.update({"ci_kind": "actual_refitted_procedure_bootstrap", "ci_low": low, "ci_high": high,
                           "ci_complete": bootstrap["status"] == "complete",
                           "bootstrap_success": bootstrap["n_success"],
                           "coverage": bool(low <= truth <= high), "reject_zero": bool(low > 0 or high < 0)})
        else:
            low, high = estimate["diagnostic_interval_only"]
            record.update({"ci_kind": "match_score_diagnostic_not_production_CI", "ci_low": low, "ci_high": high,
                           "ci_complete": True, "coverage": bool(low <= truth <= high),
                           "reject_zero": bool(low > 0 or high < 0)})
        if dvds:
            sensitivity = dvds_sensitivity(fitted_rows, fitting_config["feature_names"], gammas=[1, 2, 5],
                                           seed=dataset_seed, tipping_tolerance=None)
            endpoint_rows = []
            for point in sensitivity["bounds"].to_dict("records"):
                gamma = point["gamma"]
                oracle_lower, oracle_upper = _simulation_oracle_dvds(selected, gamma)
                endpoint = {"gamma": gamma, "estimated_lower": point["att_lower"], "estimated_upper": point["att_upper"],
                            "oracle_lower": oracle_lower, "oracle_upper": oracle_upper,
                            "lower_error": point["att_lower"] - oracle_lower,
                            "upper_error": point["att_upper"] - oracle_upper,
                            "truth_inside_oracle": oracle_lower <= truth <= oracle_upper,
                            "min_quantile_boundary_margin": point["min_quantile_boundary_margin"]}
                if bootstrap_reps:
                    endpoint_draws = []
                    for attempted in bootstrap["attempts"].to_dict("records"):
                        if attempted.get("statistics_status") == "ok":
                            endpoint_draws.extend(s for s in attempted["statistics"] if s["gamma"] == gamma)
                    count = len(endpoint_draws)
                    endpoint["endpoint_bootstrap_success"] = count
                    endpoint["endpoint_ci_complete"] = count >= bootstrap_reps
                    if count:
                        lower_low = float(np.quantile([d["att_lower"] for d in endpoint_draws], .025))
                        lower_high = float(np.quantile([d["att_lower"] for d in endpoint_draws], .975))
                        upper_low = float(np.quantile([d["att_upper"] for d in endpoint_draws], .025))
                        upper_high = float(np.quantile([d["att_upper"] for d in endpoint_draws], .975))
                        endpoint.update({"lower_endpoint_coverage": lower_low <= oracle_lower <= lower_high,
                                         "upper_endpoint_coverage": upper_low <= oracle_upper <= upper_high,
                                         "oracle_set_outer_coverage": lower_low <= oracle_lower and oracle_upper <= upper_high,
                                         "true_att_outer_coverage": lower_low <= truth <= upper_high})
                endpoint_rows.append(endpoint)
            record["dvds_bound_rows"] = endpoint_rows
    except Exception as exc:
        record.update({"status": "failed", "failure": repr(exc), "traceback": traceback.format_exc()})
    record["elapsed_seconds"] = time.monotonic() - began
    if progress is not None:
        progress.close('completed' if record['status'] == 'ok' else 'failed',
                       replicate=replicate, scenario=scenario['name'], error=record.get('failure'))
    return record


In [ ]:


def summarize_simulations(records):
    data = pd.DataFrame(records)
    summaries = []
    if data.empty:
        return data, pd.DataFrame()
    for scenario, part in data.groupby("scenario", sort=False):
        valid = part.loc[part.status.eq("ok")]
        successful = len(valid)
        n_coverage = int(valid.get("ci_complete", pd.Series(dtype=bool)).sum())
        coverage = float(valid.loc[valid.ci_complete, "coverage"].mean()) if n_coverage else np.nan
        zero = bool(part.effect_sign.iloc[0] == 0)
        summaries.append({"scenario": scenario, "n_datasets": len(part), "n_success": successful,
                          "identification_failure_expected": bool(part.identification_failure_expected.iloc[0]),
                          "n_failed": len(part) - successful, "n_complete_intervals": n_coverage,
                          "bias": float(valid.error.mean()) if successful else np.nan,
                          "rmse": float(np.sqrt(np.square(valid.error).mean())) if successful else np.nan,
                          "bias_mc_se": float(valid.error.std(ddof=1) / np.sqrt(successful)) if successful > 1 else np.nan,
                          "coverage": coverage,
                          "coverage_mc_se": float(np.sqrt(coverage * (1 - coverage) / n_coverage)) if n_coverage else np.nan,
                          "false_positive_rate": float(valid.reject_zero.mean()) if successful and zero else np.nan,
                          "fpr_mc_se": float(np.sqrt(valid.reject_zero.mean() * (1 - valid.reject_zero.mean()) / successful)) if successful and zero else np.nan,
                          "mean_retention": float(valid.retention.mean()) if successful else np.nan,
                          "mean_ess": float(valid.control_ess.mean()) if successful else np.nan,
                          "ci_kind": part.get("ci_kind", pd.Series(["none"])).iloc[0]})
    return data, pd.DataFrame(summaries)


def summarize_dvds_simulations(records):
    endpoint_records = []
    for record in records:
        if record["status"] == "ok":
            endpoint_records.extend(dict(row, scenario=record["scenario"], replicate=record["replicate"])
                                    for row in record.get("dvds_bound_rows", []))
    endpoints = pd.DataFrame(endpoint_records)
    summaries = []
    if len(endpoints):
        for (scenario, gamma), group in endpoints.groupby(["scenario", "gamma"]):
            n = len(group)
            summary = {"scenario": scenario, "gamma": gamma, "n": n,
                       "lower_bias": float(group.lower_error.mean()), "upper_bias": float(group.upper_error.mean()),
                       "lower_bias_mc_se": float(group.lower_error.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan,
                       "upper_bias_mc_se": float(group.upper_error.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan}
            complete = group.loc[group.get("endpoint_ci_complete", pd.Series(False, index=group.index)).eq(True)]
            summary["n_complete_endpoint_intervals"] = len(complete)
            for field in ["lower_endpoint_coverage", "upper_endpoint_coverage", "oracle_set_outer_coverage", "true_att_outer_coverage"]:
                probability = float(complete[field].mean()) if len(complete) else np.nan
                summary[field] = probability
                summary[field + "_mc_se"] = float(np.sqrt(probability * (1 - probability) / len(complete))) if len(complete) else np.nan
            summaries.append(summary)
    return endpoints, pd.DataFrame(summaries)


def run_simulation_harness(config, fit_fn, checkpoint_dir, logger=None, repetitions=200,
                           coverage_cases=False, bootstrap_reps=1000, workers=4,
                           n_matches=None, decisions_per_match=None, scenarios=None, dvds=False):
    """Resumable production-function simulations, with explicit honest completion.

    The base 24-class experiments use cluster-score diagnostic intervals. The six
    sign/overlap learned-procedure cases execute the full bootstrap CI pipeline.
    These are distinct outputs and must not be substituted for one another.
    """
    directory = Path(checkpoint_dir)
    directory.mkdir(parents=True, exist_ok=True)
    n_matches = int(n_matches if n_matches is not None else config.get("simulation_n_matches", 48))
    decisions_per_match = int(decisions_per_match if decisions_per_match is not None else config.get("simulation_decisions_per_match", 250))
    chosen = scenarios if scenarios is not None else simulation_scenarios()
    if coverage_cases and scenarios is None:
        chosen = [s for s in chosen if s["family"] == "base24" and s["e_correct"] and s["m_correct"]]
    import joblib
    provenance = {"core": globals().get("CORE_SOURCE_HASH", "unextracted"), "config": config,
                  "repetitions": repetitions, "coverage_cases": coverage_cases,
                  "bootstrap_reps": bootstrap_reps if coverage_cases else 0,
                  "n_matches": n_matches, "occasions": decisions_per_match,
                  "scenarios": chosen, "dvds": dvds}
    signature = hashlib.sha256(json.dumps(provenance, sort_keys=True, default=str).encode()).hexdigest()[:20]
    checkpoint = directory / f"simulation_{signature}.json"
    records = _load_inference_checkpoint(checkpoint) if checkpoint.exists() else []
    completed = {(r["scenario"], r["replicate"]) for r in records}
    pending = [(scenario, rep) for scenario in chosen for rep in range(int(repetitions))
               if (scenario["name"], rep) not in completed]
    total = len(chosen) * int(repetitions)
    worker_count = max(1, int(workers))
    for begin in range(0, len(pending), worker_count):
        _inference_disk_guard(config)
        batch = pending[begin:begin + worker_count]
        args = [(s, r, config, fit_fn, directory, n_matches, decisions_per_match,
                 bootstrap_reps if coverage_cases else 0, coverage_cases, dvds) for s, r in batch]
        if worker_count == 1:
            results = [_simulation_one(*argument) for argument in args]
        else:
            with joblib.parallel_config(backend="loky", inner_max_num_threads=1):
                results = joblib.Parallel(n_jobs=worker_count)(joblib.delayed(_simulation_one)(*argument) for argument in args)
        for result in results:
            records.append(result)
            _inference_log(logger, "simulation_dataset", **result)
        _atomic_inference_json(checkpoint, records)
        if logger is not None:
            logger.progress("production_simulation" if coverage_cases else "base_simulation",
                            len(records), total, failures=sum(r["status"] != "ok" for r in records))
    frame, summary = summarize_simulations(records)
    dvds_records, dvds_summary = summarize_dvds_simulations(records)
    complete = len(records) == total
    successful = frame.loc[frame.status.eq("ok")] if len(frame) else frame
    # Prespecified validation tolerances allow Monte Carlo variation but cannot
    # silently count failed datasets as evidence of correct coverage.
    base_complete = complete and len(successful) == total
    checks = {"all_datasets_executed": complete, "no_computational_failures": len(successful) == total}
    if coverage_cases:
        checks["all_bootstraps_complete"] = bool(len(successful) and successful.ci_complete.all())
        identified_summary = summary.loc[~summary.identification_failure_expected] if len(summary) else summary
        checks["coverage_with_mc_tolerance"] = bool(len(identified_summary) and
            (identified_summary.coverage + 2 * identified_summary.coverage_mc_se >= .95).all())
        zero_summary = summary.loc[summary.scenario.isin([s["name"] for s in chosen if s["effect_sign"] == 0])]
        checks["null_fpr_with_mc_tolerance"] = bool(len(zero_summary) and
            (zero_summary.false_positive_rate - 2 * zero_summary.fpr_mc_se <= .05).all())
        if dvds:
            checks["all_endpoint_bootstraps_complete"] = bool(len(dvds_summary) and
                dvds_summary.n_complete_endpoint_intervals.eq(int(repetitions)).all())
            checks["endpoint_coverage_with_mc_tolerance"] = bool(len(dvds_summary) and
                (dvds_summary.oracle_set_outer_coverage + 2 * dvds_summary.oracle_set_outer_coverage_mc_se >= .95).all())
    else:
        base_summary = summary.loc[summary.scenario.isin([s["name"] for s in chosen if s["family"] == "base24"])]
        # Correctly specified arm(s) are expected to recover the selected-domain
        # mean within sampling error; both-misspecified stress is descriptive.
        names = [s["name"] for s in chosen if s["family"] == "base24" and (s["e_correct"] or s["m_correct"])]
        identified = base_summary.loc[base_summary.scenario.isin(names)]
        checks["recoverable_class_bias"] = bool(len(identified) and
            (identified.bias.abs() <= .005 + 2 * identified.bias_mc_se.fillna(0)).all())
    passed = base_complete and all(checks.values())
    return {"records": frame, "summary": summary, "dvds_records": dvds_records, "dvds_summary": dvds_summary,
            "checks": checks, "passed": passed,
            "complete": complete, "completion": complete, "status": "complete" if complete else "incomplete",
            "coverage_cases": coverage_cases, "cache_signature": signature,
            "limitations": "Class membership is not oracle finite-sample fitting; match-score intervals do not replace production bootstrap coverage. Visibility selection deliberately breaks exchangeability: its primary ATT coverage is descriptive, not a recovery gate; DVDS gates target the oracle observed-law endpoints."}


## Null experiments, robustness, tactical insight and durable execution

Fresh frames/models prevent the verified DoWhy 0.14 stale-score placebo bug. Global/within-match randomized treatment and independent dummy outcomes are software checks, not proof of exchangeability. A negative control is omitted without a defensible causal justification.

Model/feature ablations first hold the primary population fixed; reselected support, excluded carries, larger visible disks and full-horizon-before-end subsets are different populations. Horizons 5/10/30 seconds and goal-occurrence fallbacks are declared sensitivities, not effect-optimized alternatives.

Subgroup summaries show counts, goals, matches, balance, ESS and coverage; low-support strata remain insufficient. Cases and real donors support interpretation, not claims that an individual player made a mistake. A blinded rating template does not imply an analyst assessment has occurred.

Every stage writes structured logs, status/heartbeats, hash-verified artifacts and deterministic resume checkpoints. Confirmation access is programmatically locked. Saved exports and executed notebooks carry exact source/config/data/environment provenance. Run headlessly in tmux; see `../../docs/public_360_execution.md`.


In [ ]:
# %% Null experiments, exports and execution workflow
import os
import time
import traceback
import tempfile
from functools import partial

import joblib
from joblib import Parallel, delayed
from PublicCausal.data import acquire_cohort, build_decisions
from PublicCausal.notebook_runtime import (
    RunLogger, CheckpointStore, atomic_write_json, atomic_write_bytes,
    file_hash, check_disk_budget, freeze_design_lock, validate_design_lock,
    environment_provenance, json_safe,
)


def atomic_parquet(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix='.' + path.name, suffix='.parquet', dir=path.parent)
    os.close(fd)
    try:
        frame.to_parquet(name, index=False)
        os.replace(name, path)
    finally:
        Path(name).unlink(missing_ok=True)
    return path


def atomic_joblib(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, name = tempfile.mkstemp(prefix='.' + path.name, dir=path.parent)
    os.close(fd)
    try:
        joblib.dump(value, name, compress=3)
        os.replace(name, path)
    finally:
        Path(name).unlink(missing_ok=True)
    return path


def cluster_score_interval(rows):
    """Shared denominator-aware conditional approximate match-cluster t CI."""
    result = att_estimate(rows)
    return dict(att=result['att'], diagnostic_se=result['cluster_score_se'],
                diagnostic_lower=result['score_interval'][0],
                diagnostic_upper=result['score_interval'][1], clusters=result['n_matches'],
                df=result['cluster_score_df'], interval_kind=result['score_interval_kind'])


def summary_att(rows):
    result = att_estimate(rows)
    return {key: value for key, value in result.items() if not isinstance(value, (pd.DataFrame, pd.Series))}


def null_replicate(frame, config, kind, branch, replicate, seed, fixed_design=None):
    started = time.monotonic()
    rng = np.random.default_rng(seed)
    sample = frame.copy()
    progress = _worker_progress(config, f'null_{kind}_{branch}_{replicate}')
    known_probability = None
    if kind == 'global_treatment':
        known_probability = np.full(len(sample), sample['T'].mean())
        sample['T'] = rng.permutation(sample['T'].to_numpy())
    elif kind == 'within_match_treatment':
        known_probability = sample.groupby('original_match_id')['T'].transform('mean').to_numpy()
        sample['T'] = sample.groupby('original_match_id', sort=True)['T'].transform(lambda column: rng.permutation(column.to_numpy()))
    elif kind == 'independent_dummy':
        sample['Y'] = rng.choice([-1, 0, 1], len(sample), p=[.01, .98, .01])
    else:
        raise ValueError(kind)
    # Fresh functions and frames: no stale propensity_score from DoWhy.
    sample = sample.drop(columns=['propensity_score', 'e', 'm0', 'm1', 'support'], errors='ignore')
    try:
        result = fit_study(sample, config, fixed_design if branch == 'fixed_domain' else None, logger=progress)
        stats = cluster_score_interval(result['rows'])
        if known_probability is not None and np.all((known_probability > 0) & (known_probability < 1)):
            oracle_rows = result['rows'].copy()
            oracle_rows['e'] = known_probability
            oracle_score = cluster_score_interval(oracle_rows)
            stats['known_randomization_att'] = oracle_score['att']
            stats['known_randomization_diagnostic_se'] = oracle_score['diagnostic_se']
        elif known_probability is not None:
            stats['known_randomization_status'] = 'Assignment has a nonpositive original-match stratum; no oracle score is claimed'
        stats.update(status='success', diagnostics_passed=result['diagnostics']['passed'],
                     propensity_hash=stable_hash(result['rows'].e.tolist()))
    except Exception as exc:
        stats = dict(status='failure', exception_type=type(exc).__name__, error=str(exc), traceback=traceback.format_exc())
    if progress is not None:
        progress.close('completed' if stats['status'] == 'success' else 'failed', replicate=replicate,
                       error=stats.get('error'))
    return dict(kind=kind, branch=branch, replicate=replicate, seed=int(seed),
                elapsed_seconds=time.monotonic() - started, **stats)


def run_null_checks(frame, config, fixed_design, logger, store):
    results = []
    target = int(config.get('null_reps', 200))
    for kind in ['global_treatment', 'within_match_treatment', 'independent_dummy']:
        for branch in ['fixed_domain', 'rebuilt_domain']:
            stage = f'null_{kind}_{branch}'
            pending = []
            logger.progress(stage, 0, target)
            for replicate in range(target):
                seed = int(np.random.SeedSequence([config['seed'], 991, replicate,
                    ['global_treatment', 'within_match_treatment', 'independent_dummy'].index(kind),
                    int(branch == 'rebuilt_domain')]).generate_state(1)[0])
                key = store.key(stage, replicate=replicate, seed=seed)
                cached = store.load(key)
                if cached is not None:
                    results.append(cached)
                else:
                    pending.append((replicate, seed, key))
            jobs = Parallel(n_jobs=config.get('workers', 4), return_as='generator_unordered',
                            pre_dispatch=config.get('workers', 4))(
                delayed(null_replicate)(frame, config, kind, branch, replicate, seed, fixed_design)
                for replicate, seed, _ in pending)
            keys = {replicate: key for replicate, _, key in pending}
            complete = target - len(pending)
            for record in jobs:
                store.save(keys[record['replicate']], record)
                results.append(record)
                complete += 1
                logger.progress(stage, complete, target, replicate=record['replicate'],
                                result_status=record['status'], seed=record['seed'])
    table = pd.DataFrame(results)
    summaries = []
    for (kind, branch), group in table.groupby(['kind', 'branch']):
        valid = group.loc[group.status == 'success']
        mean = float(valid.att.mean()) if len(valid) else float('nan')
        mcse = float(valid.att.std(ddof=1) / np.sqrt(len(valid))) if len(valid) > 1 else float('inf')
        passed = len(valid) >= .95 * target and target >= 200 and abs(mean) <= max(.005, 4 * mcse)
        summaries.append(dict(kind=kind, branch=branch, requested=target, successful=len(valid),
                              failed=len(group) - len(valid), mean_null_att=mean, monte_carlo_se=mcse,
                              score_interval_false_positive=float(((valid.diagnostic_lower > 0) | (valid.diagnostic_upper < 0)).mean()) if len(valid) else None,
                              known_randomization_mean=float(valid.known_randomization_att.mean()) if 'known_randomization_att' in valid else None,
                              diagnostic_explanation='Known-assignment scores help separate nuisance approximation from software behavior; neither establishes observational exchangeability.',
                              passed=bool(passed)))
    return dict(passed=all(item['passed'] for item in summaries), summary=summaries, attempts=table,
                interpretation='Software null checks, not tests of exchangeability. Score intervals are diagnostic, not primary uncertainty.',
                negative_control='No pre-action outcome with a defensible exclusion restriction was identified; none is presented as identification evidence.')


In [ ]:


def subgroup_points(rows, config):
    families = [('distance', 18., 'm'), ('angle', 30., 'degrees'), ('nearest_defender', 1.5, 'm')]
    records = []
    for feature, boundary, unit in families:
        for label, mask in [('low', rows[feature] <= boundary), ('high', rows[feature] > boundary)]:
            subset = rows.loc[mask].copy()
            diagnostic = support_diagnostics(subset, config)
            valid = (diagnostic['n_shots'] >= 50 and diagnostic['control_ess'] >= 50 and diagnostic['n_treated_matches'] >= 10
                     and diagnostic['key_max_smd'] <= config['key_smd_max']
                     and diagnostic['other_max_smd'] <= config['other_smd_max']
                     and diagnostic['max_weight_share'] <= config['max_weight_share']
                     and diagnostic['max_match_weight_share'] <= config['max_match_weight_share'])
            try:
                estimate = att_estimate(subset)
                point = estimate.get('att', estimate.get('estimate'))
            except ValueError:
                point = None
            records.append(dict(subgroup=f'{feature}_{label}', feature=feature, boundary=boundary,
                                unit=unit, att=point, sufficient=valid,
                                treated_positive_goals=int(((subset.Treatment == 1) & subset.support & (subset.Y == 1)).sum()),
                                treated_negative_goals=int(((subset.Treatment == 1) & subset.support & (subset.Y == -1)).sum()),
                                **diagnostic))
    return records


def bootstrap_statistics(rows, config):
    bounds = dvds_sensitivity(rows, config['feature_names'], fit_rows=rows,
                              seed=config['seed'], tipping_tolerance=None)
    groups = {}
    for feature, threshold in [('distance', 18), ('angle', 30), ('nearest_defender', 1.5)]:
        for label, mask in [('low', rows[feature] <= threshold), ('high', rows[feature] > threshold)]:
            try:
                groups[f'{feature}_{label}'] = summary_att(rows.loc[mask])['att']
            except ValueError:
                groups[f'{feature}_{label}'] = None
    return dict(sensitivity=bounds['bounds'][['gamma', 'att_lower', 'att_upper']].to_dict('records'), subgroups=groups)


def summarize_bootstrap_extras(bootstrap, point_rows, config):
    endpoint_rows, group_rows = [], []
    for record in bootstrap['attempts'].to_dict('records'):
        if record['status'] != 'ok' or not isinstance(record.get('statistics'), dict):
            continue
        statistics = record['statistics']
        endpoint_rows.extend(dict(row, attempt=record['attempt']) for row in statistics['sensitivity'])
        group_rows.append(dict(attempt=record['attempt'], **statistics['subgroups']))
    endpoints = pd.DataFrame(endpoint_rows)
    intervals = []
    if len(endpoints):
        for gamma, group in endpoints.groupby('gamma'):
            intervals.append(dict(gamma=gamma, successful=len(group),
                lower_endpoint_ci_low=group.att_lower.quantile(.025), lower_endpoint_ci_high=group.att_lower.quantile(.975),
                upper_endpoint_ci_low=group.att_upper.quantile(.025), upper_endpoint_ci_high=group.att_upper.quantile(.975),
                outer_uncertainty_low=group.att_lower.quantile(.025), outer_uncertainty_high=group.att_upper.quantile(.975),
                qualification='Refitted cluster endpoint uncertainty; clustered ternary coverage must be assessed in simulation.'))
    group_draws = pd.DataFrame(group_rows)
    points = pd.DataFrame(subgroup_points(point_rows, config))
    names = points.subgroup.tolist()
    if len(group_draws) and set(names).issubset(group_draws):
        complete = group_draws[names].dropna()
        standard_error = complete.std(ddof=1)
        point = points.set_index('subgroup').att.reindex(names).astype(float)
        positive_variance = standard_error > 1e-12
        if len(complete) >= 500 and positive_variance.all() and point.notna().all():
            maximum = np.abs((complete - point) / standard_error).max(axis=1)
            critical = maximum.quantile(.95)
            points['simultaneous_ci_low'] = [point[n] - critical * standard_error[n] for n in names]
            points['simultaneous_ci_high'] = [point[n] + critical * standard_error[n] for n in names]
            points['simultaneous_method'] = 'Bootstrap maximum standardized absolute deviation across six prespecified strata'
            points['joint_successful_draws'] = len(complete)
        else:
            points['interval_status'] = 'Insufficient joint draws or degenerate bootstrap variance; no simultaneous interval claim'
    return dict(endpoint_draws=endpoints, endpoint_intervals=pd.DataFrame(intervals),
                subgroup_draws=group_draws, subgroup_intervals=points)


def run_simulation_smoke(config, logger, checkpoint_dir):
    pilot = dict(config, simulation_n_matches=12, simulation_decisions_per_match=40,
                 workers=config.get('workers', 4))
    results = run_simulation_harness(pilot, fit_study, checkpoint_dir, logger=logger,
                                     repetitions=2, coverage_cases=False, bootstrap_reps=10,
                                     n_matches=12, decisions_per_match=40,
                                     workers=pilot['workers'])
    results['interpretation'] = 'Engineering smoke only; two datasets per scenario do not validate bias or coverage.'
    save_complex_result(results, Path(checkpoint_dir) / 'exports')
    if not results.get('complete') or not results.get('checks', {}).get('no_computational_failures'):
        raise RuntimeError('Controlled simulation engineering smoke did not complete without computational failures')
    return results


def run_production_simulations(config, logger, checkpoint_dir, runtime_config):
    repetitions = int(runtime_config.get('simulation_datasets', 200))
    bootstrap_reps = int(runtime_config.get('simulation_bootstrap_target', 1000))
    base = run_simulation_harness(config, fit_study, Path(checkpoint_dir) / 'base', logger=logger,
                                 repetitions=repetitions, coverage_cases=False,
                                 bootstrap_reps=bootstrap_reps, workers=config.get('workers', 4))
    save_complex_result(base, Path(checkpoint_dir) / 'base_exports')
    coverage = run_simulation_harness(config, fit_study, Path(checkpoint_dir) / 'actual_ci', logger=logger,
                                     repetitions=repetitions, coverage_cases=True,
                                     bootstrap_reps=bootstrap_reps, workers=config.get('workers', 4))
    save_complex_result(coverage, Path(checkpoint_dir) / 'coverage_exports')
    stress = run_simulation_harness(config, fit_study, Path(checkpoint_dir) / 'dvds_stress', logger=logger,
                                   repetitions=repetitions, coverage_cases=True,
                                   bootstrap_reps=bootstrap_reps, workers=config.get('workers', 4),
                                   scenarios=[s for s in simulation_scenarios() if s['family'] == 'stress'], dvds=True)
    save_complex_result(stress, Path(checkpoint_dir) / 'dvds_stress_exports')
    enough = repetitions >= 200 and bootstrap_reps >= 1000
    base_summary = base.get('summary')
    coverage_summary = coverage.get('summary')
    stress_summary = stress.get('summary')
    if isinstance(base_summary, pd.DataFrame):
        base_summary = base_summary.to_dict('records')
    if isinstance(coverage_summary, pd.DataFrame):
        coverage_summary = coverage_summary.to_dict('records')
    if isinstance(stress_summary, pd.DataFrame):
        stress_summary = stress_summary.to_dict('records')
    return dict(passed=bool(enough and base.get('passed') and coverage.get('passed') and stress.get('passed')),
                completion=base.get('complete') and coverage.get('complete') and stress.get('complete'),
                summary=dict(base=base_summary, actual_ci=coverage_summary, dvds_stress=stress_summary,
                             requested_datasets_per_scenario=repetitions, requested_cluster_draws=bootstrap_reps,
                             interpretation='Targets are design commitments; passing operating-characteristic gates is measured, not assumed.'))


In [ ]:


def team_removal_influence(frame, config, frozen_design, logger, store):
    records = []
    for team in sorted(set(frame.team_id.astype(str)) | set(frame.opponent_id.astype(str))):
        key = store.key('team_removal', team=team)
        record = store.load(key)
        if record is None:
            excluded = frame.loc[(frame.team_id.astype(str) == team) | (frame.opponent_id.astype(str) == team), 'original_match_id'].unique()
            kept = frame.loc[~frame.original_match_id.isin(excluded)].copy()
            try:
                fitted = fit_study(kept, config, frozen_design=frozen_design)
                record = dict(team=team, status='success', removed_matches=len(excluded),
                              estimate=summary_att(fitted['rows']), diagnostics=fitted['diagnostics'])
            except Exception as exc:
                record = dict(team=team, status='failure', removed_matches=len(excluded),
                              error=str(exc), traceback=traceback.format_exc())
            store.save(key, record)
        records.append(record)
        logger.progress('team_removal', len(records), len(set(frame.team_id.astype(str)) | set(frame.opponent_id.astype(str))),
                        removed_team=team, result_status=record['status'])
    return records


def joint_domain_draw(development_frame, confirmation_frame, config, replicate, seed):
    progress = _worker_progress(config, f'joint_domain_{replicate}')
    try:
        development = resample_original_matches(development_frame, seed)
        confirmation = resample_original_matches(confirmation_frame, seed + 1)
        design = fit_frozen_design(development, dict(config, seed=seed, workers=1), logger=progress)
        result = fit_study(confirmation, dict(config, seed=seed + 1, workers=1), frozen_design=design, logger=progress)
        if progress is not None:
            progress.close('completed', replicate=replicate)
        return dict(replicate=replicate, seed=seed, status='success', estimate=summary_att(result['rows']),
                    domain_threshold=design['domain']['threshold'], domain_cells=design['domain']['cells'],
                    diagnostics=result['diagnostics'])
    except Exception as exc:
        if progress is not None:
            progress.close('failed', replicate=replicate, error=str(exc))
        return dict(replicate=replicate, seed=seed, status='failure', error=str(exc), traceback=traceback.format_exc())


def joint_domain_sensitivity(development_frame, confirmation_frame, config, logger, store, n_reps=200):
    records, pending = [], []
    for replicate in range(n_reps):
        seed = int(np.random.SeedSequence([config['seed'], 88712, replicate]).generate_state(1)[0])
        key = store.key('joint_domain_draw', replicate=replicate, seed=seed)
        record = store.load(key)
        if record is not None:
            records.append(record)
        else:
            pending.append((replicate, seed, key))
    keys = {replicate: key for replicate, _, key in pending}
    jobs = Parallel(n_jobs=config['workers'], return_as='generator_unordered', pre_dispatch=config['workers'])(
        delayed(joint_domain_draw)(development_frame, confirmation_frame, config, replicate, seed)
        for replicate, seed, _ in pending)
    for record in jobs:
        store.save(keys[record['replicate']], record)
        records.append(record)
        logger.progress('joint_domain_sensitivity', len(records), n_reps, result_status=record['status'], replicate=record['replicate'])
        check_disk_budget(config['study_root'])
    values = [r['estimate']['att'] for r in records if r['status'] == 'success']
    return dict(attempts=pd.DataFrame(records), summary=dict(requested=n_reps, successful=len(values), failed=n_reps-len(values),
        percentile_low=float(np.quantile(values, .025)) if values else None,
        percentile_high=float(np.quantile(values, .975)) if values else None,
        interpretation='Joint Euro/WC refits vary the support domain and selected treated target. This is separate from the fixed-Euro-domain confirmation CI.'))


def run_robustness(frame, baseline, config, logger, output_dir, frozen_design=None):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    records = []
    common = baseline['rows'].copy()
    common['support'] &= common.e.between(.05, .95, inclusive='neither')
    if frozen_design is None:
        try:
            records.append(dict(kind='support_sensitivity', variant='full_propensity_common_overlap_005_095',
                                population='different_selected_population', estimate=summary_att(common),
                                diagnostics=support_diagnostics(common, config),
                                qualification='Exploratory additional OOF-score selection, not the frozen confirmation domain.'))
        except ValueError as exc:
            records.append(dict(kind='support_sensitivity', variant='common_overlap', status='insufficient', reason=str(exc)))
    calipers = football_caliper_matching(baseline['rows'], frozen_design.get('caliper_width') if frozen_design else None)
    save_complex_result(calipers, output_dir / 'football_calipers')
    for spec in [{'kind': 'ridge', 'alpha': 1}, {'kind': 'gam', 'knots': 5, 'alpha': 1},
                 {'kind': 'hgb', 'leaves': 15, 'iterations': 200}]:
        rows, provenance = refit_on_fixed_population(baseline['rows'], config, outcome_spec=spec, logger=logger)
        records.append(dict(kind='outcome_model', spec=spec, population='fixed_primary',
                            estimate=summary_att(rows), diagnostics=support_diagnostics(rows, config)))
        atomic_write_json(output_dir / ('outcome_' + spec['kind'] + '_provenance.json'), provenance)
    primary_ids = baseline['rows'].loc[baseline['rows'].support, 'decision_row_id'].tolist()
    for horizon in [5, 10, 30]:
        sample = frame.copy()
        sample['Y'] = sample[f'Y{horizon}']
        result = fit_study(sample, config, frozen_design=frozen_design, logger=logger)
        ids = result['rows'].loc[result['rows'].support, 'decision_row_id'].tolist()
        if ids != primary_ids:
            raise AssertionError('Changing reward horizon changed treatment-only support construction')
        records.append(dict(kind='horizon', horizon=horizon, population='fixed_primary',
                            estimate=summary_att(result['rows']), diagnostics=result['diagnostics']))
    if 'Y15_shot_start' in frame:
        sample = frame.copy(); sample['Y'] = sample.Y15_shot_start
        result = fit_study(sample, config, frozen_design=frozen_design, logger=logger)
        assert result['rows'].loc[result['rows'].support, 'decision_row_id'].tolist() == primary_ids
        records.append(dict(kind='goal_timing', population='fixed_primary',
                            timing='shot_start_fallback', estimate=summary_att(result['rows'])))
    feature_sets = {
        'spatial': ['x', 'y', 'distance', 'angle'],
        'without_history': [x for x in config['feature_names'] if x not in ['time_since_prev_action', 'prior_10s_actions', 'prior_10s_progress']],
        'without_360_pressure': [x for x in config['feature_names'] if x not in ['defenders_3m', 'nearest_defender', 'second_defender']],
        'without_context': [x for x in config['feature_names'] if x not in ['score_difference', 'period', 'time_seconds', 'team_id', 'opponent_id', 'cohort', 'position_role']],
        'without_geometry': [x for x in config['feature_names'] if x not in ['x', 'y', 'distance', 'angle']],
    }
    for name, features in feature_sets.items():
        altered = dict(config, feature_names=features)
        fixed_rows, provenance = refit_on_fixed_population(baseline['rows'], altered, logger=logger,
                                                         frozen_specs=frozen_design['model_specs'] if frozen_design else None)
        selected = baseline['rows'].support
        full_log_odds = np.log(baseline['rows'].loc[selected, 'e'] / (1 - baseline['rows'].loc[selected, 'e']))
        reduced_log_odds = np.log(fixed_rows.loc[selected, 'e'] / (1 - fixed_rows.loc[selected, 'e']))
        absolute_shift = np.abs(full_log_odds - reduced_log_odds)
        records.append(dict(kind='feature_ablation', ablation=name, population='fixed_primary',
                            estimate=summary_att(fixed_rows), diagnostics=support_diagnostics(fixed_rows, config),
                            omitted_group_odds_multiplier_95pct=float(np.exp(np.quantile(absolute_shift, .95))),
                            omitted_group_odds_multiplier_max=float(np.exp(absolute_shift.max())),
                            benchmarking_caveat='Descriptive measured-group odds shifts, not a proof or calibrated magnitude of hidden confounding.'))
        if frozen_design is None:
            reselected = fit_study(frame, altered, logger=logger)
            records.append(dict(kind='feature_ablation', ablation=name, population='reselected_support_different_estimand',
                                estimate=summary_att(reselected['rows']), diagnostics=reselected['diagnostics']))
        atomic_write_json(output_dir / (name + '_provenance.json'), provenance)
    for name, mask in [('carry_excluded', ~frame.contains_carry),
                       ('full15_before_end', frame.full_horizon_15),
                       ('visible_disk5', frame.visible_disk_5m),
                       ('visible_disk10', frame.visible_disk_10m)]:
        altered = frame.loc[mask].copy()
        try:
            result = fit_study(altered, config, frozen_design=frozen_design, logger=logger)
            records.append(dict(kind='selection_sensitivity', variant=name,
                                population='different_selected_population', estimate=summary_att(result['rows']),
                                diagnostics=result['diagnostics']))
        except ValueError as exc:
            records.append(dict(kind='selection_sensitivity', variant=name, status='insufficient', reason=str(exc)))
    atomic_write_json(output_dir / 'summary.json', records)
    return records


In [ ]:


def football_caliper_matching(rows, caliper_width=None):
    """Real pair/reuse-weighted matching, never union-donor AIPW.

    Prespecified legacy-inspired 3m goal-distance and .2 SD portable-logit
    calipers, same side zone and local density, external-match donors only.
    Targets without donors are explicitly a different selected population.
    """
    supported = rows.loc[rows.support].copy()
    supported['zone'] = np.digitize(supported.y, [-9.16, 9.16])
    supported['density'] = np.minimum(supported.defenders_3m, 2)
    supported['logit'] = np.log(supported.portable_e / (1 - supported.portable_e))
    width = float(caliper_width) if caliper_width is not None else .2 * float(supported.logit.std(ddof=1))
    donors = supported.loc[supported.Treatment == 0]
    shots = supported.loc[supported.Treatment == 1]
    pairs, retained = [], []
    for _, shot in shots.iterrows():
        candidates = donors.loc[(donors.original_match_id != shot.original_match_id) &
                                (donors.zone == shot.zone) & (donors.density == shot.density) &
                                ((donors.distance - shot.distance).abs() <= 3) &
                                ((donors.logit - shot.logit).abs() <= width)].copy()
        if not len(candidates):
            continue
        candidates['metric_distance'] = np.hypot(candidates.x - shot.x, candidates.y - shot.y)
        selected = candidates.sort_values('metric_distance').head(3)
        retained.append(shot)
        pairs.extend(dict(shot_id=shot.decision_row_id, donor_id=donor.decision_row_id,
                          donor_original_match_id=donor.original_match_id,
                          pair_weight=1/len(selected), donor_outcome=donor.Y,
                          distance=donor.metric_distance) for _, donor in selected.iterrows())
    pair_frame = pd.DataFrame(pairs)
    if not len(pair_frame):
        return dict(summary=dict(status='insufficient', original_shots=len(shots), retained_shots=0), pairs=pair_frame)
    target = pd.DataFrame(retained)
    reuse = pair_frame.groupby('donor_id', as_index=False).agg(matching_weight=('pair_weight', 'sum'),
                                                            reuse_count=('pair_weight', 'size'), donor_outcome=('donor_outcome', 'first'))
    contrast = target.Y.mean() - (reuse.matching_weight * reuse.donor_outcome).sum() / len(target)
    return dict(summary=dict(status='descriptive', att=float(contrast), original_shots=len(shots), retained_shots=len(target),
                    retention=len(target)/max(1,len(shots)), effective_matched_donors=float(reuse.matching_weight.sum()**2/(reuse.matching_weight**2).sum()),
                    logit_caliper=width, normalization='actual_pair_and_reuse_weights',
                    inference='No ordinary bootstrap CI asserted for nonsmooth NN matching; narrower selected target.'), pairs=pair_frame, reuse=reuse)


def export_case_packet(rows, comparisons, output_dir):
    """States and real donor pairs, not individual counterfactual mistake labels."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cases = []
    shots = rows.loc[rows.Treatment == 1].sort_values(['original_match_id', 'decision_row_id'])
    for supported in [True, False]:
        cases.append(shots.loc[shots.support == supported].head(3))
    packet = pd.concat(cases, ignore_index=True)
    atomic_parquet(packet, output_dir / 'cases.parquet')
    if 'pairs' in comparisons:
        pairs = comparisons['pairs']
        if isinstance(pairs, pd.DataFrame):
            atomic_parquet(pairs, output_dir / 'all_comparator_pairs.parquet')
            atomic_parquet(pairs.loc[pairs.shot_id.isin(packet.decision_row_id)], output_dir / 'case_donor_pairs.parquet')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from PublicCausal.data import metric_xy, visible_polygon
    data_root = next((p for p in output_dir.parents if (p / 'raw/data/three-sixty').is_dir()), None)
    frame_cache = {}
    for i, (_, row) in enumerate(packet.iterrows()):
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.set_xlim(-52.5, 52.5); ax.set_ylim(-34, 34); ax.set_aspect('equal')
        ax.plot([-52.5, 52.5, 52.5, -52.5, -52.5], [-34, -34, 34, 34, -34], color='black')
        ax.scatter([row.x], [row.y], c='red', label='decision ball')
        ax.add_patch(plt.Circle((row.x, row.y), 3, fill=False, color='orange'))
        pairs = comparisons.get('pairs')
        if isinstance(pairs, pd.DataFrame):
            donor_ids = pairs.loc[pairs.shot_id == row.decision_row_id, 'donor_id']
            donors = rows.loc[rows.decision_row_id.isin(donor_ids)]
            if len(donors):
                ax.scatter(donors.x, donors.y, marker='s', facecolors='none', edgecolors='green',
                           label='matched donor ball states (separate occasions)')
        if data_root is not None:
            match = str(row.match_id)
            if match not in frame_cache:
                source = data_root / 'raw/data/three-sixty' / (match + '.json')
                frame_cache[match] = {f['event_uuid']: f for f in json.loads(source.read_text())}
            snapshot = frame_cache[match].get(row.event_id)
            if snapshot:
                polygon = visible_polygon(snapshot['visible_area'])
                polygon = np.asarray(polygon)
                ax.fill(polygon[:, 0], polygon[:, 1], color='skyblue', alpha=.15, label='visible area')
                for teammate, color in [(True, 'blue'), (False, 'black')]:
                    locations = [metric_xy(p['location']) for p in snapshot['freeze_frame'] if p['teammate'] == teammate and not p.get('actor')]
                    if locations:
                        positions = np.asarray(locations)
                        ax.scatter(positions[:, 0], positions[:, 1], c=color, marker='o' if teammate else 'x',
                                   label='anonymous teammates' if teammate else 'anonymous opponents')
        ax.set_title(f'Case {i+1}: support={row.support}; distance={row.distance:.1f}m; local opponents={row.defenders_3m}')
        ax.text(.01, .01, str(row.get('support_reason', '')) + '\nSource: public StatsBomb/Hudl data',
                transform=ax.transAxes, fontsize=7)
        ax.legend(); fig.tight_layout()
        for extension in ['svg', 'pdf', 'png']:
            fig.savefig(output_dir / f'case_{i+1}.{extension}', dpi=220)
        plt.close(fig)
    template = ('case_id,blind_rater_id,comparison_plausibility_1to5,visibility_concern,donor_context_concern,notes\n' +
                ''.join(f'{i+1},,,,,\n' for i in range(len(packet))))
    atomic_write_bytes(output_dir / 'blinded_rating_template.csv', template.encode())


def export_study(result, output_dir, config):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rows = result['rows']
    try:
        point_bundle = att_estimate(rows)
        save_complex_result(point_bundle, output_dir / 'att_contributions')
        point = {key: value for key, value in point_bundle.items() if not isinstance(value, (pd.DataFrame, pd.Series))}
        point['units'] = 'signed first-goal reward per supported decision'
        point['att_per_100_supported_decisions'] = 100 * point['att']
    except ValueError as exc:
        point = dict(status='insufficient', reason=str(exc))
    atomic_parquet(rows, output_dir / 'nuisance_rows.parquet')
    atomic_write_json(output_dir / 'support_diagnostics.json', result['diagnostics'])
    atomic_write_json(output_dir / 'fold_provenance.json', result['provenance'])
    atomic_write_json(output_dir / 'model_validation.json', result['model_validation'])
    # All subsequent tables/figures use persisted rows, not hand-entered values.
    rows = pd.read_parquet(output_dir / 'nuisance_rows.parquet')
    atomic_write_json(output_dir / 'att.json', point)
    atomic_write_json(output_dir / 'reward_components_observed.json', {
        'treated_positive_fraction': float(((rows.Treatment == 1) & rows.support & (rows.Y == 1)).sum() / max(1, ((rows.Treatment == 1) & rows.support).sum())),
        'treated_negative_fraction': float(((rows.Treatment == 1) & rows.support & (rows.Y == -1)).sum() / max(1, ((rows.Treatment == 1) & rows.support).sum())),
        'interpretation': 'Observed reward frequencies only; signed causal reward is not automatically a scoring-probability difference.'})
    summaries, calibration = outcome_validation(rows)
    summaries.to_csv(output_dir / 'outcome_validation.csv', index=False)
    calibration.to_csv(output_dir / 'weighted_calibration.csv', index=False)
    atomic_write_json(output_dir / 'subgroups.json', subgroup_points(rows, config))
    try:
        comparisons = att_comparators(rows, config['support_features'])
    except ValueError as exc:
        comparisons = dict(status='insufficient', reason=str(exc))
    for name, value in comparisons.items():
        if isinstance(value, pd.DataFrame):
            atomic_parquet(value, output_dir / f'comparator_{name}.parquet')
        else:
            atomic_write_json(output_dir / f'comparator_{name}.json', value)
    export_case_packet(rows, comparisons, output_dir / 'cases')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for arm, group in rows.loc[rows.support].groupby('Treatment'):
        axes[0].hist(group.e, bins=30, alpha=.5, label=f'arm {arm}')
    axes[0].set(xlabel='Full-adjustment held-out propensity', ylabel='Supported occasions'); axes[0].legend()
    balance = result['diagnostics']['balance']
    keys = [key for key in sorted(balance, key=balance.get, reverse=True) if np.isfinite(balance[key])][:20]
    axes[1].barh(keys[::-1], [balance[k] for k in keys[::-1]])
    axes[1].axvline(.10, color='red', linestyle='--'); axes[1].set(xlabel='Absolute weighted SMD')
    fig.tight_layout()
    for extension in ['svg', 'pdf', 'png']:
        fig.savefig(output_dir / f'support.{extension}', dpi=220)
    plt.close(fig)
    return point


In [ ]:


def cached_fit(frame, config, run_dir, provenance, logger, frozen_design=None):
    path = Path(run_dir) / 'fit.joblib'
    hash_path = path.with_suffix('.sha256.json')
    key = stable_hash(provenance)
    if path.exists() and hash_path.exists():
        saved = json.loads(hash_path.read_text())
        if saved['key'] != key or saved['sha256'] != file_hash(path):
            raise ValueError('Fit cache provenance/content changed; use a new run ID')
        logger.event('checkpoint_reused', stage='nuisances', path=str(path))
        return joblib.load(path)
    result = fit_study(frame, config, frozen_design, logger)
    atomic_joblib(result, path)
    atomic_write_json(hash_path, dict(key=key, sha256=file_hash(path)))
    return result


def development_timing_gate(frame):
    """Measured observation audit, not a claim event snapshots precede action."""
    eligible = frame.loc[frame.eligible]
    return dict(passed=bool(len(eligible) and eligible.Y15.notna().all() and
                           eligible.original_match_id.nunique() >= 30),
                eligible=len(eligible), matches=eligible.original_match_id.nunique(),
                exclusions=frame.drop_reason.value_counts(dropna=False).to_dict(),
                outcome='signed first goal within15 seconds or natural half termination',
                limitation='Start-labelled snapshot/action timing is not provably strictly pre-action; visibility may depend on action.')


def execute_public_study(runtime_config):
    if runtime_config.get('profile', 'extended') == 'submission':
        return execute_submission_study(runtime_config)
    """Run phases with durable artifacts; a failed scientific gate is a result.

    No failure silently becomes a passing design lock or authorizes holdout.
    Expensive bootstrap/simulation jobs resume from deterministic checkpoints.
    """
    runtime_config = copy.deepcopy(runtime_config)
    root = Path(runtime_config['root']).resolve()
    study_root = Path(runtime_config.get('study_root', root / 'data/public_statsbomb_causal'))
    run_dir = Path(runtime_config['run_dir'])
    run_dir.mkdir(parents=True, exist_ok=True)
    phase = runtime_config.get('phase', 'smoke')
    if phase == 'full':
        development_config = dict(runtime_config, phase='development', run_dir=str(run_dir / 'development'))
        development = execute_public_study(development_config)
        if development['status'] != 'completed':
            atomic_write_json(run_dir / 'phase_result.json', development)
            return development
        confirmation_config = dict(runtime_config, phase='confirmation', run_dir=str(run_dir / 'confirmation'),
                                   design_lock_path=development['design_lock_path'])
        confirmation = execute_public_study(confirmation_config)
        atomic_write_json(run_dir / 'phase_result.json', confirmation)
        return confirmation
    logger = RunLogger(run_dir, runtime_config.get('run_id', run_dir.name))
    config = scientific_config(runtime_config.get('scientific_overrides'))
    for key in ['workers', 'bootstrap_target', 'null_reps']:
        if key in runtime_config:
            config[key] = runtime_config[key]
    config['study_root'] = str(study_root)
    config['log_run_dir'] = str(run_dir)
    config['run_id'] = runtime_config.get('run_id', run_dir.name)
    config['bootstrap_max_attempts'] = runtime_config.get('bootstrap_max_attempts', max(1500, config['bootstrap_target']))
    if phase == 'smoke':
        config.update(outer_folds=3, inner_folds=2, min_shots=5, min_matches=2,
                      min_ess=5, donor_min=2, donor_match_min=2,
                      key_smd_max=10, other_smd_max=10,
                      max_weight_share=1, max_match_weight_share=1,
                      e_candidates=[{'kind': 'logistic', 'C': .2}],
                      outcome_candidates=[{'kind': 'ridge', 'alpha': 1},
                                          {'kind': 'gam', 'knots': 3, 'alpha': 1},
                                          {'kind': 'hgb', 'leaves': 7, 'iterations': 100}])
    atomic_write_json(run_dir / 'config.json', dict(runtime=runtime_config, science=config))
    atomic_write_json(run_dir / 'public_release_manifest.json', {
        'source_commit': SOURCE_COMMIT,
        'allowed_project_files': ['results/models/public_statsbomb_causal_study.ipynb',
            'src/PublicCausal/', 'scripts/run_public_causal.py', 'scripts/validate_public_causal.py',
            'tests/public_causal/', 'requirements_public_causal.txt',
            'docs/public_360_approved_plan.md', 'docs/public_360_execution.md'],
        'public_data': 'Only data/public_statsbomb_causal/raw and canonical artifacts derived from the pinned source, subject to source agreement/attribution.',
        'excluded': ['Private PFF data, artifacts, model weights and row-level audit inputs', 'Existing unrelated project code'],
        'private_appendix_status': 'The DoWhy stale-score mechanism was reproduced synthetically; full attribution of the historical football placebo failure remains unresolved until exact matching inputs are rerun. No other positive estimator resolves it.',
        'publication': 'No automatic external publication. Check pinned license and attribution before distributing data or figures.'})
    environment = environment_provenance()
    environment_fingerprint = stable_hash({key: environment[key] for key in ['python', 'executable', 'packages']})
    environment_path = run_dir / 'environment.json'
    if environment_path.exists():
        old_environment = json.loads(environment_path.read_text())
        if stable_hash({key: old_environment[key] for key in ['python', 'executable', 'packages']}) != environment_fingerprint:
            raise ValueError('Environment changed on resume; use a new run ID')
    else:
        atomic_write_json(environment_path, environment)
    helper_hashes = {str(path.relative_to(root)): file_hash(path) for path in (root / 'src/PublicCausal').glob('*.py')}
    core_hash = runtime_config.get('core_source_hash', globals().get('CORE_SOURCE_HASH'))
    frozen_design = None
    lock_path = Path(runtime_config.get('design_lock_path') or (run_dir / 'design_lock.json'))
    cohort = 'confirmation' if phase == 'confirmation' else 'development'
    try:
        check_disk_budget(study_root)
        if phase == 'simulation':
            with logger.stage('simulation_validation'):
                simulation = run_production_simulations(config, logger, run_dir / 'simulations', runtime_config)
                save_complex_result(simulation, run_dir / 'exports/simulations')
            outcome = dict(status='completed' if simulation.get('passed') else 'scientific_failure',
                           phase='simulation', confirmation_opened=False, simulation=simulation.get('summary'))
            atomic_write_json(run_dir / 'phase_result.json', outcome)
            logger.close(outcome['status'])
            return outcome
        if phase == 'confirmation':
            lock = validate_design_lock(lock_path, expected_core_hash=core_hash)
            if lock['helper_hashes'] != helper_hashes:
                raise PermissionError('Helpers changed after design lock')
            frozen_path = Path(lock['frozen_design_path'])
            if file_hash(frozen_path) != lock['frozen_design_hash']:
                raise PermissionError('Frozen support design checksum changed')
            frozen_design = joblib.load(frozen_path)
            frozen_config = copy.deepcopy(frozen_design['config'])
            for key in ['workers', 'bootstrap_target', 'null_reps']:
                frozen_config[key] = config[key]
            config = frozen_config
            config['log_run_dir'] = str(run_dir)
            config['run_id'] = runtime_config.get('run_id', run_dir.name)
        with logger.stage('acquisition'):
            manifest = acquire_cohort(study_root, cohort, logger,
                                      limit_matches=6 if phase == 'smoke' else None,
                                      design_lock=lock_path if phase == 'confirmation' else None)
        with logger.stage('canonicalization'):
            candidates = build_decisions(study_root, cohort, logger,
                                         design_lock=lock_path if phase == 'confirmation' else None)
            atomic_parquet(candidates, run_dir / 'candidate_decisions.parquet')
            timing = development_timing_gate(candidates)
            atomic_write_json(run_dir / 'timing_audit.json', timing)
            frame = prepare_analysis(candidates, config)
            data_hash = stable_hash(dict(source_commit=manifest['source_commit'],
                                         raw_data_hash=manifest['data_sha256'],
                                         candidates=file_hash(run_dir / 'candidate_decisions.parquet')))
        provenance = dict(core_hash=core_hash, helper_hashes=helper_hashes, config=copy.deepcopy(config),
                          environment_hash=environment_fingerprint,
                          data_hash=data_hash, design_hash=file_hash(lock_path) if phase == 'confirmation' else None)
        atomic_write_json(run_dir / 'provenance.json', provenance)
        store = CheckpointStore(run_dir / 'checkpoints', provenance)
        config['provenance_hash'] = stable_hash(provenance)
        with logger.stage('nuisances'):
            result = cached_fit(frame, config, run_dir, provenance, logger, frozen_design)
        with logger.stage('exports'):
            point = export_study(result, run_dir / 'exports', config)
        atomic_write_json(run_dir / 'primary_status.json', dict(status='supported' if result['diagnostics']['passed'] else 'unsupported',
                          exploratory_att=point, diagnostics=result['diagnostics'],
                          warning='No causal headline without passing design/uncertainty/sensitivity checks.'))
        if phase not in ['smoke', 'simulation'] and not result['diagnostics']['passed']:
            outcome = dict(status='scientific_failure', stage='support', confirmation_opened=phase == 'confirmation',
                           reason='No diagnostics-passing supported ATT. Thresholds are not relaxed after seeing effects.',
                           diagnostics=result['diagnostics'])
            atomic_write_json(run_dir / 'phase_result.json', outcome)
            logger.close('scientific_failure', reason=outcome['reason'])
            return outcome
        with logger.stage('bootstrap', total=config['bootstrap_target']):
            bootstrap = cluster_bootstrap(frame, config, fit_study, frozen_design=frozen_design,
                                           n_success=config['bootstrap_target'], logger=logger,
                                           checkpoint_dir=run_dir / 'bootstrap',
                                           stats_fn=partial(bootstrap_statistics, config=config) if phase != 'smoke' else None)
            save_complex_result(bootstrap, run_dir / 'exports/bootstrap')
            if phase != 'smoke':
                extra = summarize_bootstrap_extras(bootstrap, result['rows'], config)
                save_complex_result(extra, run_dir / 'exports/bootstrap_extra')
        bootstrap_complete = bootstrap.get('summary', {}).get('status') == 'complete'
        if phase != 'smoke':
            intervals = extra['endpoint_intervals']
            endpoint_complete = bool(len(intervals) and intervals.successful.ge(config['bootstrap_target']).all())
            bootstrap_complete = bootstrap_complete and endpoint_complete
        if not bootstrap_complete:
            outcome = dict(status='scientific_failure', phase=phase, stage='bootstrap',
                           confirmation_opened=phase == 'confirmation',
                           reason='Requested successful primary or sensitivity-endpoint cluster-refit bootstrap count was not reached.',
                           bootstrap_summary=bootstrap.get('summary'))
            atomic_write_json(run_dir / 'phase_result.json', outcome)
            logger.close('scientific_failure', reason=outcome['reason'])
            return outcome
        if phase == 'smoke':
            with logger.stage('simulation_smoke'):
                simulation = run_simulation_smoke(config, logger, run_dir / 'simulation_smoke')
            outcome = dict(status='completed', phase='smoke', primary_claim=False,
                           message='Engineering smoke only: relaxed size/balance gates cannot authorize confirmation.',
                           diagnostics=result['diagnostics'], bootstrap_target=config['bootstrap_target'])
            atomic_write_json(run_dir / 'phase_result.json', outcome)
            logger.close('completed', phase='smoke')
            return outcome
        with logger.stage('sensitivity'):
            sensitivity = dvds_sensitivity(result['rows'], config['feature_names'], fit_rows=result['rows'], seed=config['seed'])
            save_complex_result(sensitivity, run_dir / 'exports/sensitivity')
        with logger.stage('robustness'):
            run_robustness(frame, result, config, logger, run_dir / 'exports/robustness', frozen_design=frozen_design)
        with logger.stage('influence'):
            influence = leave_one_match_out(frame, config, fit_study, frozen_design=frozen_design,
                                            logger=logger, checkpoint_dir=run_dir / 'influence')
            save_complex_result(influence, run_dir / 'exports/influence')
            teams = team_removal_influence(frame, config, frozen_design, logger, store)
            atomic_write_json(run_dir / 'exports/team_removal.json', teams)
        if phase == 'development':
            with logger.stage('freeze_candidate'):
                frozen_design = fit_frozen_design(frame, config, logger)
                frozen_path = atomic_joblib(frozen_design, run_dir / 'frozen_design.joblib')
            with logger.stage('null_checks'):
                nulls = run_null_checks(frame, config, frozen_design, logger, store)
                save_complex_result(nulls, run_dir / 'exports/null_checks')
            with logger.stage('simulation_validation'):
                simulations = run_production_simulations(config, logger, run_dir / 'simulations', runtime_config)
                save_complex_result(simulations, run_dir / 'exports/simulations')
            test_result_path = root / 'data/public_statsbomb_causal/manifests/implementation_tests.json'
            test_result = json.loads(test_result_path.read_text()) if test_result_path.exists() else {}
            gates = dict(timing_audit=timing['passed'],
                         leakage_tests=test_result.get('passed') is True and test_result.get('core_source_hash') == core_hash and test_result.get('helper_hashes') == helper_hashes,
                         development_diagnostics=result['diagnostics']['passed'] and frozen_design['domain']['selection_passed'] and bootstrap_complete,
                         null_checks=nulls['passed'], simulation_validation=simulations.get('passed') is True)
            atomic_write_json(run_dir / 'development_gates.json', gates)
            if all(gates.values()):
                freeze_design_lock(lock_path, dict(source_commit=SOURCE_COMMIT, core_source_hash=core_hash,
                    data_manifest_hash=data_hash, config_hash=stable_hash(config), helper_hashes=helper_hashes,
                    development_gates=gates, frozen_design_path=str(frozen_path), frozen_design_hash=file_hash(frozen_path),
                    development_candidates_sha256=file_hash(run_dir / 'candidate_decisions.parquet'),
                    development_run_dir=str(run_dir), development_match_ids=frozen_design['training_match_ids']))
                outcome = dict(status='completed', phase=phase, design_lock_path=str(lock_path), confirmation_opened=False)
            else:
                outcome = dict(status='scientific_failure', phase=phase, development_gates=gates,
                               confirmation_opened=False, reason='Unpassed development gate; no design lock generated.')
        else:
            if phase == 'confirmation':
                with logger.stage('joint_domain_sensitivity', total=200):
                    development_candidates = Path(lock['development_run_dir']) / 'candidate_decisions.parquet'
                    if file_hash(development_candidates) != lock['development_candidates_sha256']:
                        raise PermissionError('Development decisions changed after lock')
                    development_frame = prepare_analysis(pd.read_parquet(development_candidates), config)
                    joint = joint_domain_sensitivity(development_frame, frame, config, logger, store, n_reps=200)
                    save_complex_result(joint, run_dir / 'exports/joint_domain_sensitivity')
            outcome = dict(status='completed', phase=phase, diagnostics=result['diagnostics'],
                           warning='Report actual uncertainty/confounding sensitivity before causal conclusion.')
        atomic_write_json(run_dir / 'phase_result.json', outcome)
        check_disk_budget(study_root)
        logger.close(outcome['status'])
        return outcome
    except BaseException as exc:
        atomic_write_json(run_dir / 'phase_result.json', dict(status='failed', phase=phase,
                          exception_type=type(exc).__name__, error=str(exc), traceback=traceback.format_exc()))
        logger.close('failed', error=str(exc), traceback=traceback.format_exc())
        raise


In [ ]:


def save_complex_result(result, path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    if isinstance(result, pd.DataFrame):
        atomic_parquet(result, path / 'rows.parquet')
        return
    summary = {}
    for key, value in result.items():
        if isinstance(value, pd.DataFrame):
            if value.index.name:
                value = value.reset_index()
            atomic_parquet(value, path / f'{key}.parquet')
        elif isinstance(value, np.ndarray):
            atomic_parquet(pd.DataFrame(value), path / f'{key}.parquet')
        else:
            summary[key] = value
    atomic_write_json(path / 'summary.json', summary)


In [ ]:

# %% Submission artifact lineage and bounded durable execution
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from joblib.externals.loky import ProcessPoolExecutor
from threadpoolctl import threadpool_limits
import signal
from PublicCausal.notebook_runtime import utc_now


def _submission_task_call(worker, spec):
    started=time.monotonic()
    try:
        # pragmatic: all independent sklearn/XGBoost tasks use one inner thread.
        with threadpool_limits(limits=1):
            record=worker(spec)
        if not isinstance(record,dict):
            raise TypeError('Submission task must return a compact result dictionary')
    except Exception as exc:
        record=dict(status='failed', error=repr(exc), failure_category='computational',
                    traceback=traceback.format_exc())
    return json_safe(dict(record, task_id=spec['task_id'], seed=spec.get('seed'),
                          elapsed_seconds=record.get('elapsed_seconds',time.monotonic()-started)))


def _submission_close_progress(progress,record):
    """Close a worker logger; logging problems never discard a computed record."""
    if progress is None: return
    try:
        progress.close('completed' if record['status']=='ok' else 'failed',result_status=record['status'],
                       task_seconds=record.get('elapsed_seconds'),replicate=record.get('replicate'),
                       error=record.get('failure'))
    except Exception as exc:
        record['logging_error']=repr(exc)


class SubmissionTaskPool:
    """One persistent global budget; durable records as each worker returns.

    Stop tokens/signals drain admitted tasks. Resume never adopts another source,
    configuration, dataset, procedure or seed namespace. Production processes are
    persistent across sequential stages; threads are available for tiny tests.
    """
    def __init__(self,root,identity,workers=1,executor_kind='process',stop_path=None,
                 logger=None,study_root=None,handle_signals=False,progress_path=None):
        self.workers=int(workers)
        self.progress_path=Path(progress_path) if progress_path else None
        self.plans={}; self.seen={}; self.started_monotonic=None
        if self.workers<1 or self.workers>4:
            raise ValueError('Global worker budget must be between 1 and 4')
        if executor_kind not in {'process','thread'}:
            raise ValueError('Unknown executor kind')
        self.root=Path(root); self.root.mkdir(parents=True,exist_ok=True)
        self.identity=json_safe(dict(identity)); self.identity_hash=stable_hash(self.identity)
        self.logger=logger; self.study_root=study_root
        self.executor_kind=executor_kind; self.executor=None
        self.stop_path=Path(stop_path) if stop_path else self.root.parent/'STOP_REQUESTED'
        self.handle_signals=handle_signals; self.previous_handlers={}
        path=self.root/'identity.json'
        payload=dict(schema=1,identity=self.identity,identity_hash=self.identity_hash,
                     workers=self.workers,inner_threads=1)
        if path.exists():
            old=json.loads(path.read_text())
            if old!=payload:
                raise ValueError('Incompatible submission task identity/provenance or resource budget')
        else:
            atomic_write_json(path,payload)

    @property
    def stopped(self): return self.stop_path.exists()

    def request_stop(self,reason='signal'):
        atomic_write_json(self.stop_path,dict(reason=reason,identity_hash=self.identity_hash,
                                            checkpoint_policy='drain admitted tasks, no new admission'))
        log_event(self.logger,'stop_requested',reason=reason)

    def plan(self,stage,required,kind,target_success=False):
        """Declare required records (successes when target_success) for the global ETA."""
        self.plans[stage]=dict(required=int(required),kind=str(kind),target_success=bool(target_success))
        self.seen.setdefault(stage,{})
        self._write_progress()

    def _record_progress(self,stage,spec,value,fresh):
        # Keyed by task hash: reloading a stage (e.g. a precision extension) never double counts.
        entries=self.seen.setdefault(stage,{})
        key=stable_hash(spec)
        if fresh or key not in entries:
            entries[key]=dict(ok=value.get('status')=='ok',
                              fresh_seconds=float(value.get('elapsed_seconds') or 0.) if fresh else None)

    def _stage_state(self,stage):
        entries=self.seen.get(stage,{}).values()
        fresh=[e['fresh_seconds'] for e in entries if e['fresh_seconds'] is not None]
        return dict(completed=len(entries),successful=sum(e['ok'] for e in entries),
                    cached=sum(e['fresh_seconds'] is None for e in entries),fresh=len(fresh),fresh_seconds=float(sum(fresh)))

    def _write_progress(self):
        if self.progress_path is None: return
        wall=time.monotonic()-self.started_monotonic if self.started_monotonic else 0.
        states={stage:self._stage_state(stage) for stage in self.seen}
        fresh_total=sum(state['fresh_seconds'] for state in states.values())
        # Measured concurrency: fresh task seconds per wall second, capped at the worker budget.
        concurrency=min(float(self.workers),fresh_total/wall) if wall>0 and fresh_total>0 else None
        kinds={}
        for stage,state in states.items():
            kind=self.plans.get(stage,{}).get('kind',stage)
            entry=kinds.setdefault(kind,[0,0.]); entry[0]+=state['fresh']; entry[1]+=state['fresh_seconds']
        means={kind:seconds/count for kind,(count,seconds) in kinds.items() if count}
        stages={}; remaining_tasks=0; remaining_seconds=0.; unmeasured=[]
        for stage,state in states.items():
            plan=self.plans.get(stage,{})
            required=plan.get('required',state['completed'])
            remaining=max(0,required-(state['successful'] if plan.get('target_success') else state['completed']))
            remaining_tasks+=remaining
            mean=means.get(plan.get('kind',stage))
            if remaining and mean is None: unmeasured.append(stage)
            elif remaining: remaining_seconds+=remaining*mean
            stages[stage]=dict(state,required=required,remaining=remaining,kind=plan.get('kind'),
                               mean_fresh_task_seconds=state['fresh_seconds']/state['fresh'] if state['fresh'] else None)
        eta=None
        if not remaining_tasks: eta=0.
        elif not unmeasured and concurrency: eta=remaining_seconds/concurrency
        atomic_write_json(self.progress_path,dict(utc=utc_now(),identity_hash=self.identity_hash,workers=self.workers,
            wall_seconds=wall,fresh_task_seconds=fresh_total,measured_concurrency=concurrency,
            remaining_tasks=remaining_tasks,eta_seconds=eta,unmeasured_stages=unmeasured,stages=stages,
            method='sum(remaining required records x measured mean fresh task seconds of that kind) / measured concurrency; '
                   'cached completions excluded from throughput; failures/retries and unplanned stages not forecast'))

    def __enter__(self):
        self.started_monotonic=time.monotonic()
        cls=ThreadPoolExecutor if self.executor_kind=='thread' else ProcessPoolExecutor
        self.executor=cls(max_workers=self.workers)
        if self.handle_signals:
            for sig in (signal.SIGINT,signal.SIGTERM):
                self.previous_handlers[sig]=signal.getsignal(sig)
                signal.signal(sig,lambda number,frame:self.request_stop(signal.Signals(number).name))
        return self

    def __exit__(self,*args):
        if self.executor is not None:
            self.executor.shutdown(wait=True)
        for sig,previous in self.previous_handlers.items(): signal.signal(sig,previous)

    def _path(self,stage,spec):
        if not stage or any(ch not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for ch in stage):
            raise ValueError('Unsafe task stage name')
        return self.root/stage/(stable_hash(spec)+'.json')

    def _load(self,stage,spec):
        path=self._path(stage,spec)
        if not path.exists(): return None
        value=json.loads(path.read_text())
        if (value.get('identity_hash')!=self.identity_hash or value.get('task_hash')!=stable_hash(spec)
                or stable_hash(value.get('result'))!=value.get('result_hash')):
            raise ValueError('Corrupt task checksum or incompatible provenance: '+str(path))
        return value['result']

    def _save(self,stage,spec,result):
        return atomic_write_json(self._path(stage,spec),dict(schema=1,identity_hash=self.identity_hash,
                    task_hash=stable_hash(spec),task=spec,result=result,result_hash=stable_hash(result)))

    def run(self,stage,tasks,worker,target_success=None):
        if self.executor is None: raise RuntimeError('Task pool must be used as a context manager')
        tasks=list(tasks)
        if len({str(t['task_id']) for t in tasks})!=len(tasks):
            raise ValueError('Duplicate task IDs in stage')
        records={}; remaining=[]
        for index,spec in enumerate(tasks):
            value=self._load(stage,spec)
            if value is None: remaining.append((index,spec))
            else:
                records[index]=value; self._record_progress(stage,spec,value,fresh=False)
        cached=len(records); successful=sum(r.get('status')=='ok' for r in records.values())
        cached_successful=successful
        self._write_progress()
        def report():
            # Targeted stages report progress/ETA against required successes, not attempt capacity.
            if self.logger is None: return
            if target_success is None:
                self.logger.progress(stage,len(records),len(tasks),cached_completed=cached,
                                     successful=successful,attempted=len(records))
            else:
                self.logger.progress(stage,min(successful,int(target_success)),int(target_success),
                                     cached_completed=min(cached_successful,int(target_success)),
                                     successful=successful,attempted=len(records),attempt_capacity=len(tasks))
        report()
        cursor=0; admitted={}
        while admitted or cursor<len(remaining):
            need=self.workers-len(admitted)
            if target_success is not None: need=min(need,int(target_success)-successful-len(admitted))
            while need>0 and cursor<len(remaining) and not self.stopped:
                index,spec=remaining[cursor]; cursor+=1; need-=1
                admitted[self.executor.submit(_submission_task_call,worker,spec)]=(index,spec)
                log_event(self.logger,'task_admitted',stage=stage,task_id=spec['task_id'],seed=spec.get('seed'))
            if not admitted: break
            done,_=wait(admitted,return_when=FIRST_COMPLETED)
            for future in done:
                index,spec=admitted.pop(future)
                persist=True
                try: value=future.result()
                except BaseException as exc:
                    # Dead worker (OOM/segfault): not a scientific result, so never persisted; resume retries it.
                    value=dict(status='failed',task_id=spec['task_id'],seed=spec.get('seed'),error=repr(exc),
                               failure_category='computational',worker_crash=True,elapsed_seconds=None)
                    persist=False
                if persist: self._save(stage,spec,value)  # durable BEFORE logging/admitting more work
                records[index]=value; successful+=value.get('status')=='ok'
                self._record_progress(stage,spec,value,fresh=True); self._write_progress()
                log_event(self.logger,'task_checkpoint',stage=stage,task_id=spec['task_id'],
                          result_status=value.get('status'),elapsed_seconds=value.get('elapsed_seconds'),
                          diagnostics=value.get('diagnostics'),successful=successful)
                report()
                if self.study_root: check_disk_budget(self.study_root)
        self._write_progress()
        log_event(self.logger,'task_stage_checkpoint',stage=stage,completed=len(records),
                  cached_completed=cached,successful=successful,stopped=self.stopped)
        return [records[index] for index in sorted(records)]


def _submission_original_inventory(directory):
    origin=next((p for p in Path(directory).resolve().parents if (p/'src/PublicCausal').is_dir()),None)
    if origin is None: raise ValueError('Cannot verify original source; supply original_source_inventory')
    notebook=json.loads((origin/'results/models/public_statsbomb_causal_study.ipynb').read_text())
    definitions='\n\n'.join((''.join(c['source']) if isinstance(c['source'],list) else c['source']).rstrip()
                            for c in notebook['cells'] if c.get('cell_type')=='code' and
                            'public-causal-core' in c.get('metadata',{}).get('tags',[]))+'\n'
    return dict(origin=str(origin),core_hash=stable_hash(dict(export_schema=1,definitions=definitions)),
                helper_hashes={str(p.relative_to(origin)):file_hash(p) for p in (origin/'src/PublicCausal').glob('*.py')})


def verify_development_import(directory,original_source_inventory=None):
    """Explicit baseline import; never forge a new fit-cache/resume key."""
    directory=Path(directory).resolve()
    paths={name:directory/name for name in ['fit.joblib','fit.sha256.json','provenance.json',
                          'candidate_decisions.parquet','exports/nuisance_rows.parquet']}
    for path in paths.values():
        if not path.is_file(): raise ValueError('Missing development import artifact '+str(path))
    metadata=json.loads(paths['fit.sha256.json'].read_text())
    if file_hash(paths['fit.joblib'])!=metadata.get('sha256'):
        raise ValueError('Development fit checksum mismatch')
    provenance=json.loads(paths['provenance.json'].read_text())
    if stable_hash(provenance)!=metadata.get('key'):
        raise ValueError('Development fit provenance checksum mismatch')
    inventory=original_source_inventory or _submission_original_inventory(directory)
    if 'core_hash' not in inventory:
        inventory=_submission_original_inventory(Path(inventory['origin'])/'data/public_statsbomb_causal/runs/import')
    if inventory['core_hash']!=provenance.get('core_hash'):
        raise ValueError('Original scientific source identity differs from saved provenance')
    for name,expected in provenance.get('helper_hashes',{}).items():
        if inventory.get('helper_hashes',{}).get(name)!=expected:
            raise ValueError('Original helper source identity mismatch: '+name)
    fit=joblib.load(paths['fit.joblib'])
    rows=fit['rows'].reset_index(drop=True)
    saved=pd.read_parquet(paths['exports/nuisance_rows.parquet']).reset_index(drop=True)
    features=provenance.get('config',{}).get('feature_names',fit.get('design',{}).get('config',{}).get('feature_names',[]))
    columns=list(dict.fromkeys(['original_match_id','decision_row_id','Treatment','Y','e','m0',
                                'm1','portable_e','support','outer_fold']+features))
    if len(rows)!=len(saved) or set(columns)-set(rows) or set(columns)-set(saved):
        raise ValueError('Development scalar prediction schema/row identity mismatch')
    for name in columns:
        left,right=rows[name],saved[name]
        if pd.api.types.is_numeric_dtype(left):
            equal=np.array_equal(pd.to_numeric(left).to_numpy(float),pd.to_numeric(right).to_numpy(float),equal_nan=True)
        else:
            equal=left.fillna('__missing__').astype(str).equals(right.fillna('__missing__').astype(str))
        if not equal: raise ValueError('Development scalar prediction identity mismatch: '+name)
    if rows.decision_row_id.duplicated().any() or not rows.groupby('original_match_id').outer_fold.nunique().eq(1).all():
        raise ValueError('Development row/fold identity invalid')
    for record in fit.get('provenance',[]):
        if set(map(str,record.get('train_match_ids',[]))) & set(map(str,record.get('test_match_ids',[]))):
            raise ValueError('Development import records match leakage')
    environment_path=directory/'environment.json'
    environment=json.loads(environment_path.read_text()) if environment_path.exists() else None
    if environment is not None:
        fingerprint=stable_hash({key:environment[key] for key in ['python','executable','packages']})
        if fingerprint!=provenance.get('environment_hash'):
            raise ValueError('Original environment provenance checksum mismatch')
    manifest=dict(schema=1,purpose='baseline_development_evidence',original_run=str(directory),
                  artifacts={name:dict(path=str(path),sha256=file_hash(path)) for name,path in paths.items()},
                  original_provenance=provenance,original_source_inventory=inventory,
                  original_environment=environment,original_fold_provenance=fit.get('provenance',[]),
                  original_design=fit.get('design'),scalar_columns=columns,row_count=len(rows),
                  normalized_scalar_identity_verified=True,
                  nested_json_identity_verified=rows.to_json(orient='split')==saved.to_json(orient='split'),
                  eligible_definition='L1/full3m/event-start proxies; original configuration retained',
                  bootstrap_reuse='Old adaptive development draws are supplementary only; never pooled or resumed here')
    return fit,json_safe(manifest)


def submission_analysis_specs(config):
    return ([dict(kind='outcome_model',name=spec['kind'],spec=spec) for spec in
             [{'kind':'ridge','alpha':10},{'kind':'gam','knots':3,'alpha':10},
              {'kind':'hgb','leaves':7,'iterations':100}]]+
            [dict(kind='horizon',name=f'Y{h}',outcome=f'Y{h}') for h in [5,10,30]]+
            [dict(kind='timing',name='shot_start',outcome='Y15_shot_start')]+
            [dict(kind='population',name=name) for name in ['carry_excluded','visible_disk5']]+
            [dict(kind='ablation',name=name,exclude=features) for name,features in
             [('local_pressure',['defenders_3m','nearest_defender','second_defender']),
              ('recent_history',['time_since_prev_action','prior_10s_actions','prior_10s_progress'])]])


def submission_outcome_refit(rows,config,frozen,new_outcome=None,outcome_spec=None,logger=None):
    changed=rows.copy()
    if new_outcome is not None: changed['Y']=np.asarray(new_outcome)
    answer,provenance=refit_on_fixed_population(changed,config,
                   outcome_spec=outcome_spec or frozen['model_specs']['m0'],logger=logger)
    for name in ['e','portable_e','support','outer_fold','decision_row_id']:
        if not answer[name].equals(changed[name]):
            raise AssertionError('Outcome-only refit changed treatment/support/fold identity: '+name)
    return answer,provenance


def submission_manifest(config,frozen):
    cfg=scientific_config(config)
    return json_safe(dict(
        horizon=dict(primary='Y15',seconds=15,termination='natural_half',sensitivity=[5,10,30],
                     timing='linked_goal_occurrence; shot_start sensitivity'),
        model=dict(specifications=frozen['model_specs'],feature_names=cfg['feature_names'],
                   candidates=dict(e=cfg['e_candidates'],m0=cfg['outcome_candidates']),
                   outer_folds=cfg['outer_folds'],inner_folds=cfg['inner_folds'],seed=cfg['seed'],
                   main_refits='fixed e/m0, no search or m1; grouped calibration retained',
                   propensity_selection='lowest grouped log-loss; submission+cuda: CUDA candidate within one SE preferred'),
        backend=dict(candidate_device=cfg.get('candidate_device','cpu'),
                     selected_propensity=frozen['model_specs']['e'].get('device','cpu'),
                     implementation=frozen['model_specs']['e']['kind'],threads=1),
        support=dict(domain=frozen['domain'],support_features=cfg['support_features'],
                     training_matches=frozen['training_match_ids'],radius_m=3,region='L1',
                     gates={key:cfg[key] for key in ['min_shots','min_matches','min_ess','ess_shot_fraction',
                            'key_smd_max','other_smd_max','max_weight_share','max_match_weight_share']},
                     outcome_independent_evaluation_domain=True),
        subgroup=dict(strata=[['distance',18],['angle',30],['nearest_defender',1.5]],
                      inference='joint original-match multiplier max-statistic including contrasts',
                      numerical_draws=10000,qualification='existing count/ESS/match/balance/concentration gates'),
        inference=dict(primary='conditional approximate match-cluster t score CI',
                       denominator='supported treated decision count',registry='all eligible original matches',
                       bootstrap='fixed-design fixed-spec original-match refits',
                       precision_rule=dict(numerical_draws=1000,endpoint_mcse_width_fraction=.05,
                                           optional_extension=bool(cfg.get('bootstrap_precision_extension',False)),
                                           maximum_successes=500,independent_of_significance=True),
                       assumptions='adequate overlap, small nuisance remainder, independent match clusters; repeated teams limitation'),
        simulation=dict(scenarios=[s['name'] for s in submission_scenarios()],scenario_forms=submission_scenarios(),
                        independent_design_confirmation=True,n_matches=48,decisions_per_match=250,
                        truth='selected supported treated potential-mean difference',nested_bootstrap=0),
        budget=dict(bootstrap_successes=int(cfg.get('bootstrap_target',200)),
                    bootstrap_max_attempts=int(cfg.get('bootstrap_max_attempts',300)),
                    null_per_kind=int(cfg.get('null_reps',50)),
                    simulation_per_scenario=int(cfg.get('simulation_reps',50)),
                    workers=int(cfg.get('workers',1)),inner_threads=1,
                    targeted_match_refits=5,targeted_team_refits=2,
                    analyses=submission_analysis_specs(cfg),gamma=[1,1.25,1.5,2,3,5,10])))


def submission_lock_payload(runtime,config,frozen,gates,evidence):
    if runtime.get('phase') in {'smoke','pilot'}:
        raise PermissionError('Engineering smoke/pilot cannot authorize confirmation')
    required=['timing_audit','leakage_tests','development_diagnostics','weighted_validation','design_sanity']
    if not all(gates.get(key) is True for key in required):
        raise PermissionError('Unpassed submission development gates')
    manifest=submission_manifest(config,frozen)
    return dict(evidence,profile='submission',development_gates=gates,
                specification_manifest=manifest,specification_manifest_hash=stable_hash(manifest))


def _submission_task_design(frozen):
    """Locked design without bulky all-Euro selection rows; tasks need specs/portable/domain only."""
    return {key:value for key,value in frozen.items() if key!='development_selection'}


def _submission_estimate_record(rows,diagnostics):
    estimate=att_estimate(rows)
    return dict(att=estimate['att'],att_per_100=100*estimate['att'],score_interval=list(estimate['score_interval']),
                score_se=estimate['cluster_score_se'],score_df=estimate['cluster_score_df'],n_shots=estimate['n_shots'],
                eligible_matches=estimate['eligible_matches'],control_ess=estimate['control_ess'],
                diagnostics=_compact_inference_diagnostics(diagnostics or {}))


_SUBMISSION_POPULATIONS={'carry_excluded':lambda rows:~rows.contains_carry.astype(bool),
                         'visible_disk5':lambda rows:rows.visible_disk_5m.astype(bool)}


def submission_analysis_task(spec,rows,config,frozen,logger=None):
    """One bounded robustness task; refits only the component the variation changes.

    Outcome models/horizons/timing: fixed e/support/folds, fresh m0. Ablations: fixed
    support/folds, fresh e and m0 without the group. Population changes and targeted
    removals: frozen Euro domain/specs, fresh WC nuisances on a changed sample.
    """
    started=time.monotonic(); kind=spec['kind']
    cfg=scientific_config(dict(config,att_only=True,diagnostic_m1=False,workers=1))
    record=dict(kind=kind,name=spec.get('name'),seed=spec.get('seed'),status='running')
    progress=logger or _worker_progress(cfg,'analysis_'+str(spec.get('task_id',kind)))
    try:
        if kind in {'outcome_model','horizon','timing'}:
            if kind=='outcome_model':
                answer,_=submission_outcome_refit(rows,cfg,frozen,outcome_spec=spec['spec'],logger=progress)
                validation=_submission_weighted_validation(answer,cfg)
                record['weighted_validation']={k:validation.get(k) for k in
                    ['passed','checks','donor_ess','weighted_mse','weighted_residual','weighted_residual_se']}
                record['outcome_spec']=spec['spec']
            else:
                name=spec['outcome']
                outcome=rows[name] if name in rows else None
                if outcome is None or outcome.isna().any() or not np.isin(outcome,[-1,0,1]).all():
                    raise ValueError('Alternative outcome unavailable or invalid on fixed primary rows: '+str(name))
                answer,_=submission_outcome_refit(rows,cfg,frozen,new_outcome=outcome.to_numpy(),logger=progress)
                record['outcome']=name
            record.update(population='fixed_primary',reuse_policy='reuse e/support/folds; fresh m0',
                          **_submission_estimate_record(answer,support_diagnostics(answer,cfg)))
        elif kind=='ablation':
            excluded=set(spec['exclude'])
            kept=[name for name in cfg['feature_names'] if name not in excluded]
            answer,_=refit_on_fixed_population(rows,dict(cfg,feature_names=kept),logger=progress,
                                               frozen_specs=frozen['model_specs'])
            selected=rows.support.astype(bool).to_numpy()
            full=np.log(rows.e.to_numpy(float)[selected]/(1-rows.e.to_numpy(float)[selected]))
            reduced=np.log(answer.e.to_numpy(float)[selected]/(1-answer.e.to_numpy(float)[selected]))
            shift=np.abs(full-reduced)
            record.update(population='fixed_primary',excluded_features=sorted(excluded),
                          reuse_policy='fixed support/folds; fresh e and m0 without the group',
                          omitted_group_odds_multiplier_95pct=float(np.exp(np.quantile(shift,.95))) if len(shift) else None,
                          omitted_group_odds_multiplier_max=float(np.exp(shift.max())) if len(shift) else None,
                          benchmarking_caveat='Descriptive measured-group odds shifts, not a calibrated magnitude of hidden confounding.',
                          **_submission_estimate_record(answer,support_diagnostics(answer,cfg)))
        elif kind in {'population','targeted_match','targeted_team'}:
            if kind=='population':
                if spec['name'] not in _SUBMISSION_POPULATIONS:
                    raise ValueError('Unknown submission population variation: '+str(spec['name']))
                mask=_SUBMISSION_POPULATIONS[spec['name']](rows)
            elif kind=='targeted_match':
                mask=~rows.original_match_id.astype(str).isin(set(map(str,spec['remove_matches'])))
            else:
                team=str(spec['team'])
                mask=(rows.team_id.astype(str)!=team)&(rows.opponent_id.astype(str)!=team)
            subset=rows.loc[np.asarray(mask,bool)].copy()
            fitted=fit_study(subset,cfg,frozen_design=frozen,logger=progress)
            record.update(population='changed_selected_population',retained_rows=len(subset),
                          removed_rows=len(rows)-len(subset),
                          removed_matches=int(rows.original_match_id.nunique()-subset.original_match_id.nunique()),
                          reuse_policy='frozen Euro domain/specs; fresh WC e/m0 on changed sample',
                          **_submission_estimate_record(fitted['rows'],fitted['diagnostics']))
            if kind!='population':
                record.update(removed=spec.get('remove_matches',spec.get('team')),
                              qualification='Targeted refit of a top-ranked score-screen unit; not complete leave-one-out')
        else:
            raise ValueError('Unknown submission analysis kind: '+str(kind))
        record['status']='ok'
    except Exception as exc:
        _submission_science_failure(record,exc)
    record['elapsed_seconds']=time.monotonic()-started
    if logger is None: _submission_close_progress(progress,record)
    return record


def submission_score_influence(rows,top_matches=5,top_teams=2):
    """Exact fixed-nuisance removal screen for every match and team (no refits).

    Removing unit set M gives (sum numerators - M numerators)/(N1 - M treated).
    Top-ranked units are then refit; neither step is complete leave-one-out.
    """
    estimate=att_estimate(rows); sample=estimate['rows']; base=estimate['att']
    # String keys throughout: real match IDs are integers, screen/team lookups use strings.
    by_match=sample.groupby(sample.original_match_id.astype(str),sort=True).agg(numerator=('att_numerator','sum'),treated=('T','sum'))
    total_numerator=float(sample.att_numerator.sum()); total_treated=int(estimate['n_shots'])
    def removal(matches):
        part=by_match.reindex(sorted(set(map(str,matches))&set(by_match.index)))
        treated=total_treated-int(part.treated.sum())
        value=(total_numerator-float(part.numerator.sum()))/treated if treated>0 else float('nan')
        return value,int(part.treated.sum())
    matches=[]
    for match in sorted(rows.original_match_id.astype(str).unique()):
        value,treated=removal([match])
        matches.append(dict(original_match_id=match,att_without=value,delta=value-base,removed_treated=treated))
    team_matches={}
    for column in ['team_id','opponent_id']:
        if column in rows:
            for team,group in rows.groupby(rows[column].astype(str)):
                team_matches.setdefault(team,set()).update(group.original_match_id.astype(str))
    teams=[]
    for team,ids in sorted(team_matches.items()):
        value,treated=removal(ids)
        teams.append(dict(team=team,removed_matches=len(ids),att_without=value,delta=value-base,removed_treated=treated))
    order=lambda record:(-abs(record['delta']) if np.isfinite(record['delta']) else -np.inf)
    matches=sorted(matches,key=order); teams=sorted(teams,key=order)
    return json_safe(dict(att=base,matches=matches,teams=teams,
        targeted_matches=[r['original_match_id'] for r in matches[:int(top_matches)]],
        targeted_teams=[r['team'] for r in teams[:int(top_teams)]],
        interpretation='Fixed-nuisance score screen over all matches/teams; targeted refits of top-ranked units are not complete leave-one-out'))


def submission_subgroups(rows,config,numerical_draws=10000,seed=None):
    """Six prespecified strata; joint multiplier max-t intervals from saved match scores.

    Only strata passing existing count/ESS/match/balance/concentration gates enter the
    family (plus low-high contrasts when both sides qualify). No models are refit.
    """
    cfg=scientific_config(config)
    registry_rows=rows.loc[rows['eligible'].fillna(False).astype(bool)] if 'eligible' in rows else rows
    registry=pd.Index(registry_rows.original_match_id.unique(),name='original_match_id'); g=len(registry)
    strata,members=[],{}
    for feature,boundary,unit in [('distance',18.,'m'),('angle',30.,'degrees'),('nearest_defender',1.5,'m')]:
        for label,mask in [('low',rows[feature]<=boundary),('high',rows[feature]>boundary)]:
            subset=rows.loc[mask].copy(); name=f'{feature}_{label}'
            record=dict(subgroup=name,feature=feature,boundary=boundary,unit=unit,att=None,sufficient=False,
                        marginal_interval=None,simultaneous_interval=None,
                        treated_positive_goals=int(((subset.Treatment==1)&subset.support&(subset.Y==1)).sum()),
                        treated_negative_goals=int(((subset.Treatment==1)&subset.support&(subset.Y==-1)).sum()))
            sufficient=False
            try:
                d=support_diagnostics(subset,cfg)
                record.update({k:d[k] for k in ['n_shots','n_controls','n_treated_matches','control_ess','key_max_smd',
                                                'other_max_smd','max_weight_share','max_match_weight_share']})
                sufficient=(d['n_shots']>=50 and d['control_ess']>=50 and d['n_treated_matches']>=10
                            and d['key_max_smd']<=cfg['key_smd_max'] and d['other_max_smd']<=cfg['other_smd_max']
                            and d['max_weight_share']<=cfg['max_weight_share']
                            and d['max_match_weight_share']<=cfg['max_match_weight_share'])
            except Exception as exc:
                record['diagnostic_error']=repr(exc)
            try:
                estimate=att_estimate(subset)
                record.update(att=estimate['att'],score_se=estimate['cluster_score_se'],
                              marginal_interval=list(estimate['score_interval']))
                psi=estimate['cluster_scores']['score'].reindex(registry,fill_value=0.).to_numpy(float)/estimate['n_shots']
            except ValueError as exc:
                record['estimate_status']='insufficient: '+str(exc); sufficient=False
            record['sufficient']=bool(sufficient)
            if sufficient: members[name]=(record['att'],psi)
            strata.append(record)
    contrasts=[]
    for feature in ['distance','angle','nearest_defender']:
        low,high=f'{feature}_low',f'{feature}_high'
        record=dict(contrast=f'{low}-{high}',sufficient=low in members and high in members,estimate=None,simultaneous_interval=None)
        if record['sufficient']:
            record['estimate']=members[low][0]-members[high][0]
            members[record['contrast']]=(record['estimate'],members[low][1]-members[high][1])
        contrasts.append(record)
    critical=None; family=[]
    if members and g>1:
        names=[name for name,(_,psi) in members.items() if np.square(psi).sum()>1e-24]
        if names:
            psis=np.column_stack([members[name][1] for name in names])
            scale=np.sqrt(np.square(psis).sum(axis=0))
            rng=np.random.default_rng(int(cfg['seed'] if seed is None else seed))
            maxima=np.empty(int(numerical_draws))
            for start in range(0,int(numerical_draws),1000):
                xi=rng.standard_normal((min(1000,int(numerical_draws)-start),g))
                maxima[start:start+len(xi)]=np.abs(xi@psis/scale).max(axis=1)
            critical=float(np.quantile(maxima,.95)); family=names
            se={name:float(np.sqrt(g/(g-1))*s) for name,s in zip(names,scale)}
            for record in strata+contrasts:
                name=record.get('subgroup',record.get('contrast'))
                if name in se:
                    point=members[name][0]
                    record.update(joint_se=se[name],simultaneous_interval=[point-critical*se[name],point+critical*se[name]])
    return json_safe(dict(strata=strata,contrasts=contrasts,family=family,critical_value=critical,
        numerical_draws=int(numerical_draws),registry_matches=g,
        method='Gaussian original-match multiplier max-|t| over qualifying strata and contrasts; saved scores, no refits',
        interpretation='Insufficient strata stay insufficient; differences are judged by contrasts, not significance here versus there'))


def submission_bootstrap_summary(records,requested,seed,numerical_draws=1000,width_fraction=.05,maximum=500):
    """Fixed-spec refit distribution; finite diagnostic-poor draws retained.

    Endpoint Monte Carlo SEs resample saved ATT numbers (no refits). The precision
    rule compares them with interval width, so it is shift/sign independent.
    """
    valid=[r for r in records if r.get('status')=='ok' and np.isfinite(r.get('att',np.nan))]
    values=np.asarray([r['att'] for r in valid],float); n=len(values)
    result=dict(requested=int(requested),maximum_successes=int(maximum),attempted=len(records),successful=n,
                failed=sum(r.get('status')!='ok' for r in records),
                failure_categories={name:sum(r.get('failure_category')==name for r in records) for name in ['structural','computational']},
                diagnostic_poor=sum(not bool(r.get('diagnostics',{}).get('passed',False)) for r in valid),
                complete=n>=int(requested),mean=None,sd=None,percentile_interval=None,endpoint_mcse=None,
                precision_passed=False,extension_recommended=n<int(maximum),
                precision_rule=f'max endpoint MCSE <= {width_fraction} x percentile width, {numerical_draws} numerical resamples; independent of significance',
                procedure='fixed Euro domain/spec original-match refits of the WC ATT; no endpoint DVDS')
    if n>=2:
        low,high=np.quantile(values,[.025,.975])
        rng=np.random.default_rng(int(seed))
        endpoints=np.empty((int(numerical_draws),2))
        for start in range(0,int(numerical_draws),200):
            index=rng.integers(0,n,(min(200,int(numerical_draws)-start),n))
            endpoints[start:start+len(index)]=np.quantile(values[index],[.025,.975],axis=1).T
        mcse=endpoints.std(axis=0,ddof=1); width=float(high-low)
        precision=bool(width>0 and float(mcse.max())<=width_fraction*width)
        result.update(mean=float(values.mean()),sd=float(values.std(ddof=1)),percentile_interval=[float(low),float(high)],
                      endpoint_mcse=[float(mcse[0]),float(mcse[1])],precision_passed=precision,
                      extension_recommended=bool(not precision and n<int(maximum)))
    return json_safe(result)



def _submission_success_prefix(records,n):
    """Records in attempt order up to the n-th success: resume-invariant draw set."""
    prefix,successes=[],0
    for record in records:
        if successes>=int(n): break
        prefix.append(record); successes+=record.get('status')=='ok' and np.isfinite(record.get('att',np.nan))
    return prefix

def submission_inference_agreement(score_se,bootstrap,low=2/3,high=1.5):
    """Predeclared agreement: refit SD / score SE within [2/3, 3/2]. Not a significance choice."""
    sd=bootstrap.get('sd')
    if sd is None or score_se is None or not np.isfinite([sd,score_se]).all() or score_se<=0:
        return dict(passed=False,status='unavailable',ratio=None,rule=f'{low:.3f}<=bootstrap SD/score SE<={high}')
    ratio=float(sd/score_se)
    return dict(passed=bool(low<=ratio<=high),status='agree' if low<=ratio<=high else 'material_disagreement',
                ratio=ratio,rule=f'{low:.3f}<=bootstrap SD/score SE<={high}',
                interpretation='Agreement is useful, not proof: both can share nuisance bias')


_SUBMISSION_CLAIM_CHECKS=['support','bootstrap','precision','inference_agreement','nulls','simulations','analyses']


def submission_claim_status(checks):
    """Claim completion requires every declared check; launch or partial draws never suffice."""
    values={name:checks.get(name) is True for name in _SUBMISSION_CLAIM_CHECKS}
    failed=[name for name,value in values.items() if not value]
    if not values['support']: status='unsupported'
    elif not failed: status='completed'
    elif failed==['inference_agreement']: status='inconclusive_uncertainty'
    else: status='incomplete'
    return dict(status=status,primary_claim=status=='completed',checks=values,failed_or_missing=failed,
                qualification='Conditional approximate match-cluster inference on the locked supported WC ATT; '
                              'repeated national teams and nuisance remainder remain limitations; no automatic publication')


In [ ]:

# %% Submission development lock, engineering pilot and protected confirmation

def _submission_seed(config,stage,replicate):
    code=int(hashlib.sha256(stage.encode()).hexdigest()[:8],16)
    return int(np.random.SeedSequence([int(config['seed']),code,int(replicate)]).generate_state(1)[0])


def _submission_specs(config,stage,n,**extra):
    return [dict(task_id=f'{stage}_{i}',seed=_submission_seed(config,stage,i),replicate=i,
                 procedure=stage,**extra) for i in range(int(n))]


def submission_refit_task(spec,frame,config,frozen):
    progress=_worker_progress(config,spec['task_id'])
    record=dict(kind='refit',seed=spec.get('seed'),replicate=spec.get('replicate',0),status='running',diagnostics={})
    try:
        sample=resample_original_matches(frame,spec['seed'])
        fitted=fit_study(sample,dict(config,seed=spec['seed'],workers=1,diagnostic_m1=False),
                         frozen_design=frozen,logger=progress)
        point=att_estimate(fitted['rows'])
        if not np.isfinite(point['att']): raise ValueError('Nonfinite fixed ATT draw')
        return dict(status='ok',att=point['att'],score_interval=point['score_interval'],
                    n_shots=point['n_shots'],n_matches=point['n_matches'],control_ess=point['control_ess'],
                    diagnostics=_compact_inference_diagnostics(fitted['diagnostics']),
                    procedure='fixed_Euro_domain_fixed_spec_ATT_only',m1_fitted=False,companion_DVDS=False)
    except Exception as exc:
        _submission_science_failure(record,exc)
        return record
    finally:
        if progress: progress.close()


def _submission_existing_manifest(study_root):
    path=Path(study_root)/'manifests/development.json'
    if not path.is_file(): raise ValueError('Existing complete Euro manifest required; no silent development download')
    manifest=json.loads(path.read_text())
    if manifest.get('status')!='complete' or manifest.get('source_commit')!=SOURCE_COMMIT:
        raise ValueError('Existing Euro input manifest is incomplete or from another pinned revision')
    if stable_hash({key:value for key,value in manifest.items() if key!='manifest_sha256'})!=manifest.get('manifest_sha256'):
        raise ValueError('Existing Euro manifest checksum mismatch')
    for record in manifest['files']:
        file=Path(study_root)/record['path']
        if not file.is_file() or file.stat().st_size!=record['bytes'] or file_hash(file)!=record['sha256']:
            raise ValueError('Existing Euro raw input checksum mismatch: '+str(file))
    if stable_hash({r['source_path']:r['sha256'] for r in manifest['files']})!=manifest['data_sha256']:
        raise ValueError('Existing Euro raw fingerprint mismatch')
    return manifest


def _submission_weighted_validation(rows,config):
    donor=rows.loc[rows.support & rows.Treatment.eq(0)].copy()
    if len(donor)<2: return dict(passed=False,reason='Insufficient supported held-out donors')
    weights=(donor.e/(1-donor.e)).to_numpy(float); residual=(donor.Y-donor.m0).to_numpy(float)
    ess=_weight_ess(weights); residual_mean=float(np.average(residual,weights=weights))
    mse=float(np.average(residual**2,weights=weights))
    residual_se=float(np.sqrt(np.average((residual-residual_mean)**2,weights=weights)/max(ess,1)))
    proper=dict(log_loss=float(log_loss(rows.Treatment,rows.e,labels=[0,1])),
                brier_score=float(brier_score_loss(rows.Treatment,rows.e)))
    donor['bin']=pd.qcut(donor.m0,q=10,duplicates='drop')
    calibration=[]
    for name,part in donor.groupby('bin',observed=True):
        w=part.e/(1-part.e); r=part.Y-part.m0; neff=_weight_ess(w)
        mu=float(np.average(r,weights=w)); se=float(np.sqrt(np.average((r-mu)**2,weights=w)/max(neff,1)))
        calibration.append(dict(bin=str(name),n=len(part),ess=neff,residual=mu,se=se,
                                passed=bool(abs(mu)<=.03+4*se)))
    checks=dict(finite=bool(np.isfinite([mse,residual_mean,residual_se,*proper.values()]).all()),
                weighted_residual=abs(residual_mean)<=.02+4*residual_se,
                weighted_mse=mse<=.25,weighted_calibration=all(r['passed'] for r in calibration))
    return dict(passed=bool(all(checks.values())),checks=checks,donor_ess=ess,weighted_mse=mse,
                weighted_residual=residual_mean,weighted_residual_se=residual_se,
                heldout_propensity_metrics=proper,calibration=calibration,
                predeclared_criteria='abs weighted residual <= .02+4SE; decile residual <= .03+4SE; weighted MSE<=.25',
                role='postselection development diagnostics; final specs selected on all Euro')


def benchmark_submission_workload(frame,config,frozen,pool,n_refits=5,baseline_rows=None):
    """Measure complete Euro refits and representative null/simulation tasks, no WC."""
    if not 5<=int(n_refits)<=10: raise ValueError('Representative benchmark requires five to ten full refits')
    started=time.monotonic()
    stages={}
    design=_submission_task_design(frozen)
    pool.plan('benchmark_fixed_refits',int(n_refits),'fixed_refit',target_success=True)
    for kind in ['within_match_treatment','independent_dummy_y']: pool.plan('benchmark_null_'+kind,1,'null_'+kind)
    pool.plan('benchmark_simulation',1,'simulation')
    refs=_submission_specs(config,'benchmark_fixed_refits',max(int(n_refits)*2,int(n_refits)),kind='refit')
    records=pool.run('benchmark_fixed_refits',refs,partial(submission_refit_task,frame=frame,config=config,frozen=design),
                     target_success=int(n_refits))
    stages['fixed_refit']=records
    if baseline_rows is None:
        baseline_rows=fit_study(frame,dict(config,diagnostic_m1=False),frozen_design=frozen)['rows']
    for kind in ['within_match_treatment','independent_dummy_y']:
        stage='benchmark_null_'+kind
        jobs=_submission_specs(config,stage,1,kind='null',null_kind=kind)
        stages[kind]=pool.run(stage,jobs,partial(submission_null_task,baseline_rows=baseline_rows,config=config,frozen=design))
    scenario=submission_scenarios()[0]
    jobs=_submission_specs(config,'benchmark_simulation',1,kind='simulation',scenario=scenario,n_matches=48,decisions_per_match=250)
    stages['simulation_pair']=pool.run('benchmark_simulation',jobs,partial(submission_simulation_task,config=config,frozen=design))
    summary={name:dict(attempted=len(values),successful=sum(r.get('status')=='ok' for r in values),
                      mean_task_seconds=float(np.mean([r['elapsed_seconds'] for r in values if r.get('elapsed_seconds') is not None])) if values else None,
                      seconds=[r.get('elapsed_seconds') for r in values]) for name,values in stages.items()}
    complete=(summary['fixed_refit']['successful']==int(n_refits) and all(v['successful']==v['attempted'] and v['attempted']>0 for v in summary.values()))
    eta=None
    if complete:
        measured=summary['fixed_refit']['mean_task_seconds']*config['bootstrap_target']
        measured+=sum(summary[k]['mean_task_seconds']*config['null_reps'] for k in ['within_match_treatment','independent_dummy_y'])
        measured+=summary['simulation_pair']['mean_task_seconds']*len(submission_scenarios())*config['simulation_reps']
        measured+=summary['fixed_refit']['mean_task_seconds']*20  # point, targeted refits and bounded alternatives allowance
        eta=dict(compute_seconds=measured/pool.workers,range_seconds=[.8*measured/pool.workers,1.5*measured/pool.workers],
                 method='Measured complete task times / global workers; representative scenario extrapolation with 20fit allowance',
                 excludes='WC network/canonicalization and human review; competition/load/scenario variability remains',
                 planning_budget_used=False)
    report=dict(status='complete' if complete else 'incomplete',confirmation_opened=False,
                workers=pool.workers,inner_threads=1,full_refits_requested=int(n_refits),
                wall_seconds=time.monotonic()-started,stages=summary,eta=eta,records=stages,
                role='Euro-only workload benchmark; not confirmation or nominal coverage proof')
    atomic_write_json(pool.root.parent/'workload_benchmark.json',report)
    return report


def _submission_finish(logger,run_dir,outcome):
    atomic_write_json(Path(run_dir)/'phase_result.json',outcome)
    logger.close(outcome['status'])
    return outcome


def execute_submission_study(runtime_config):
    runtime=copy.deepcopy(runtime_config); phase=runtime.get('phase','smoke')
    run_dir=Path(runtime['run_dir']); run_dir.mkdir(parents=True,exist_ok=True)
    if phase=='full':
        lock_path=Path(runtime.get('design_lock_path') or run_dir/'development/design_lock.json')
        if not lock_path.exists():
            development=execute_submission_study(dict(runtime,phase='development',run_dir=str(run_dir/'development')))
            if development['status']!='completed':
                atomic_write_json(run_dir/'phase_result.json',development); return development
            lock_path=Path(development['design_lock_path'])
        outcome=execute_submission_study(dict(runtime,phase='confirmation',run_dir=str(run_dir/'confirmation'),design_lock_path=str(lock_path)))
        atomic_write_json(run_dir/'phase_result.json',outcome); return outcome
    root=Path(runtime['root']).resolve(); study_root=Path(runtime.get('study_root',root/'data/public_statsbomb_causal'))
    logger=RunLogger(run_dir,runtime.get('run_id',run_dir.name))
    config=scientific_config(dict(runtime.get('scientific_overrides') or {},profile='submission',
                                  candidate_device=runtime.get('candidate_device','cpu')))
    for key in ['workers','bootstrap_target','bootstrap_max_attempts','null_reps']:
        if key in runtime: config[key]=runtime[key]
    config['simulation_reps']=runtime.get('simulation_datasets',config['simulation_reps'])
    config.update(study_root=str(study_root),log_run_dir=str(run_dir),run_id=runtime.get('run_id',run_dir.name),att_only=True)
    core_hash=globals().get('CORE_SOURCE_HASH',runtime.get('core_source_hash'))
    helpers={str(p.relative_to(root)):file_hash(p) for p in (root/'src/PublicCausal').glob('*.py')}
    try:
        check_disk_budget(study_root)
        if phase=='confirmation':
            lock=validate_design_lock(runtime['design_lock_path'],expected_core_hash=core_hash,expected_profile='submission')
            if lock['helper_hashes']!=helpers: raise PermissionError('Helpers changed after submission lock')
            frozen=joblib.load(lock['frozen_design_path'])
            # Models/support plus ALL scientific/count/resource settings are immutable.
            manifest=submission_manifest(config,frozen)
            if manifest!=lock['specification_manifest']:
                raise PermissionError('Runtime specifications/counts/resources differ from submission lock')
            logger.event('protected_confirmation_acquisition',source_commit=SOURCE_COMMIT,
                         message='Pinned WC inputs are required separately from existing Euro cache; existing compatible files are verified and reused')
            raw=acquire_cohort(study_root,'confirmation',logger,design_lock=runtime['design_lock_path'])
            candidates=build_decisions(study_root,'confirmation',logger,design_lock=runtime['design_lock_path'])
            frame=prepare_analysis(candidates,config)
            return _execute_submission_confirmation(runtime,config,frozen,frame,candidates,raw,helpers,core_hash,logger)
        if phase not in {'development','pilot','smoke'}: raise ValueError('Submission phase must be development/pilot/smoke/full/confirmation')
        raw=_submission_existing_manifest(study_root)
        imported=None
        if runtime.get('reuse_development'):
            imported,lineage=verify_development_import(runtime['reuse_development'])
            atomic_write_json(run_dir/'import_manifest.json',lineage)
            old_candidates=Path(runtime['reuse_development'])/'candidate_decisions.parquet'
            expected=stable_hash(dict(source_commit=raw['source_commit'],raw_data_hash=raw['data_sha256'],candidates=file_hash(old_candidates)))
            if expected!=lineage['original_provenance']['data_hash']:
                raise ValueError('Imported candidates/raw input fingerprint differs from original fit provenance')
            candidates=pd.read_parquet(old_candidates)
            logger.event('verified_development_import',rows=len(imported['rows']),purpose=lineage['purpose'],original_run=runtime['reuse_development'])
        else:
            candidates=build_decisions(study_root,'development',logger)
        if phase=='smoke':
            ids=sorted(candidates.original_match_id.astype(str).unique())[:6]
            candidates=candidates.loc[candidates.original_match_id.astype(str).isin(ids)].copy()
            config.update(outer_folds=3,inner_folds=2,min_shots=5,min_matches=2,min_ess=5,
                          donor_min=2,donor_match_min=2,key_smd_max=10,other_smd_max=10,
                          max_weight_share=1,max_match_weight_share=1,
                          e_candidates=[{'kind':'logistic','C':.2}],outcome_candidates=[{'kind':'ridge','alpha':10}])
        atomic_parquet(candidates,run_dir/'candidate_decisions.parquet')
        frame=prepare_analysis(candidates,config)
        data_hash=stable_hash(dict(source_commit=raw['source_commit'],raw_data_hash=raw['data_sha256'],candidates=file_hash(run_dir/'candidate_decisions.parquet')))
        environment=environment_provenance()
        provenance=dict(core_hash=core_hash,helper_hashes=helpers,config=config,data_hash=data_hash,
                        environment_hash=stable_hash({key:environment[key] for key in ['python','executable','packages']}),procedure='submission_development_v1')
        atomic_write_json(run_dir/'config.json',dict(runtime=runtime,science=config))
        atomic_write_json(run_dir/'provenance.json',provenance); atomic_write_json(run_dir/'environment.json',environment)
        design_path=run_dir/'frozen_design.joblib'; design_sidecar=run_dir/'frozen_design.sha256.json'
        if design_path.exists():
            side=json.loads(design_sidecar.read_text())
            if side['key']!=stable_hash(provenance) or side['sha256']!=file_hash(design_path):
                raise ValueError('Frozen design resume identity/checksum changed')
            frozen=joblib.load(design_path)
        else:
            with logger.stage('all_Euro_selection'):
                frozen=fit_frozen_design(frame,config,logger)
                atomic_joblib(frozen,design_path)
                atomic_write_json(design_sidecar,dict(key=stable_hash(provenance),sha256=file_hash(design_path)))
        diag_config=dict(config,diagnostic_m1=True)
        diagnostic=cached_fit(frame,diag_config,run_dir,dict(provenance,role='postselection_final_spec_validation'),logger,frozen)
        weighted=_submission_weighted_validation(diagnostic['rows'],config)
        atomic_write_json(run_dir/'weighted_validation.json',weighted)
        atomic_write_json(run_dir/'final_design_diagnostics.json',dict(domain=frozen['domain'],diagnostics=diagnostic['diagnostics'],role='all_Euro_postselection_development'))
        timing=development_timing_gate(candidates); atomic_write_json(run_dir/'timing_audit.json',timing)
        with logger.stage('early_development_exports'):
            point=export_study(diagnostic,run_dir/'exports',diag_config)
            atomic_write_json(run_dir/'primary_status.json',dict(status='provisional_development',point=point,primary_claim=False,confirmation_opened=False))
        identity=dict(provenance=provenance,frozen_design_hash=file_hash(design_path),procedure='submission_tasks_v1')
        with SubmissionTaskPool(run_dir/'tasks',identity,workers=config['workers'],executor_kind=runtime.get('executor_kind','process'),
                                logger=logger,study_root=study_root,handle_signals=True,
                                progress_path=run_dir/'overall_progress.json') as pool:
            benchmark=benchmark_submission_workload(frame,config,frozen,pool,n_refits=runtime.get('benchmark_refits',5),baseline_rows=diagnostic['rows'])
            if pool.stopped: return _submission_finish(logger,run_dir,dict(status='stopped',phase=phase,confirmation_opened=False))
        if phase in {'smoke','pilot'}:
            return _submission_finish(logger,run_dir,dict(status='completed' if benchmark['status']=='complete' else 'incomplete',phase=phase,
                primary_claim=False,confirmation_opened=False,engineering_only=True,benchmark=benchmark,
                diagnostics=diagnostic['diagnostics'],message='Engineering run cannot generate a confirmation lock'))
        test_path=Path(runtime.get('implementation_tests_path',run_dir.parent/'manifests/implementation_tests.json'))
        tests=json.loads(test_path.read_text()) if test_path.exists() else {}
        leakage=tests.get('passed') is True and tests.get('core_source_hash')==core_hash and tests.get('helper_hashes')==helpers
        sim=benchmark['records'].get('simulation_pair',[])
        nulls=[record for kind in ['within_match_treatment','independent_dummy_y'] for record in benchmark['records'].get(kind,[])]
        sanity=bool(sim and all(r.get('status')=='ok' and r.get('independent_design_confirmation') is True and
                   abs(r['att']-r['truth'])<=.05+4*max(abs(r['score_interval'][1]-r['score_interval'][0])/4,1e-6) for r in sim)
                   and len(nulls)==2 and all(r.get('status')=='ok' and abs(r['att'])<=.10 for r in nulls))
        gates=dict(timing_audit=bool(timing['passed']),leakage_tests=bool(leakage),
                   development_diagnostics=bool(diagnostic['diagnostics']['passed'] and frozen['domain'].get('selection_passed') and
                                                frozen['development_selection']['support_diagnostics']['passed']),
                   weighted_validation=bool(weighted['passed']),design_sanity=sanity)
        atomic_write_json(run_dir/'development_gates.json',gates)
        if not all(gates.values()):
            return _submission_finish(logger,run_dir,dict(status='scientific_failure',phase=phase,confirmation_opened=False,development_gates=gates,
                    reason='Actual final-Euro gates failed; holdout remains protected',benchmark=benchmark))
        lock_path=run_dir/'design_lock.json'
        evidence=dict(source_commit=SOURCE_COMMIT,core_source_hash=core_hash,helper_hashes=helpers,data_manifest_hash=data_hash,
                      config_hash=stable_hash(config),frozen_design_path=str(design_path),frozen_design_hash=file_hash(design_path),
                      development_run_dir=str(run_dir),development_candidates_sha256=file_hash(run_dir/'candidate_decisions.parquet'),
                      development_match_ids=frozen['training_match_ids'])
        freeze_design_lock(lock_path,submission_lock_payload(runtime,config,frozen,gates,evidence))
        return _submission_finish(logger,run_dir,dict(status='completed',phase='development',design_lock_path=str(lock_path),
                  confirmation_opened=False,design_status='locked',claim_status='not_started',benchmark=benchmark))
    except Exception as exc:
        _submission_finish(logger,run_dir,dict(status='failed',phase=phase,error=repr(exc),traceback=traceback.format_exc(),confirmation_opened=phase=='confirmation'))
        raise


def _submission_null_check(summary):
    mcse=summary.get('mean_att_mcse'); mean=summary.get('mean_att')
    mean_ok=mean is not None and (mcse is None or abs(mean)<=max(.005,4*mcse))
    return bool(summary['attempted']==summary['requested'] and not summary['failure_categories']['computational'] and mean_ok)



def _submission_simulation_check(summary):
    """All allocations attempted, no computational failures and every scenario has valid pairs."""
    return bool(summary['attempted']==summary['requested'] and not summary['failure_categories']['computational']
                and all(item['successful']>0 for item in summary['by_scenario'].values()))

def _execute_submission_confirmation(runtime,config,frozen,frame,candidates,raw,helpers,core_hash,logger):
    """Locked WC confirmation: point/score CI exported first, then bounded checks.

    Every task shares one persistent global worker budget and durable per-task
    records. Nulls/simulations can qualify or invalidate the claim but never change
    locked choices. Claim completion is separate from launch or partial draws.
    """
    run_dir=Path(runtime['run_dir']); study_root=Path(config.get('study_root',run_dir))
    stop_path=run_dir/'STOP_REQUESTED'; design=_submission_task_design(frozen)
    lock_path=Path(runtime['design_lock_path']) if runtime.get('design_lock_path') else None
    timing=development_timing_gate(candidates); atomic_write_json(run_dir/'timing_audit.json',timing)
    atomic_parquet(candidates,run_dir/'candidate_decisions.parquet')
    environment=environment_provenance()
    data_hash=stable_hash(dict(source_commit=raw.get('source_commit'),raw_data_hash=raw.get('data_sha256'),
                               candidates=file_hash(run_dir/'candidate_decisions.parquet')))
    provenance=dict(core_hash=core_hash,helper_hashes=helpers,config=config,data_hash=data_hash,
                    environment_hash=stable_hash({key:environment[key] for key in ['python','executable','packages']}),
                    design_lock_hash=file_hash(lock_path) if lock_path and lock_path.exists() else None,
                    procedure='submission_confirmation_v1')
    atomic_write_json(run_dir/'config.json',dict(runtime=runtime,science=config))
    atomic_write_json(run_dir/'provenance.json',provenance); atomic_write_json(run_dir/'environment.json',environment)
    primary_config=dict(config,diagnostic_m1=True)
    with logger.stage('confirmation_primary'):
        primary=cached_fit(frame,primary_config,run_dir,dict(provenance,role='confirmation_primary_m1_diagnostic_once'),logger,design)
        point=export_study(primary,run_dir/'exports',primary_config)
    supported=bool(primary['diagnostics']['passed'])
    atomic_write_json(run_dir/'primary_status.json',dict(status='provisional_confirmation' if supported else 'unsupported',
        point=point,primary_claim=False,confirmation_opened=True,diagnostics=_compact_inference_diagnostics(primary['diagnostics']),
        note='Approximate conditional score CI; claim status requires the declared bounded checks'))
    logger.event('confirmation_primary_exported',supported=supported,att=point.get('att'),
                 score_interval=point.get('score_interval'),n_shots=point.get('n_shots'))
    base=dict(phase='confirmation',confirmation_opened=True,point=point)
    if not supported:
        claim=submission_claim_status({'support':False})
        return _submission_finish(logger,run_dir,dict(base,status='unsupported',claim_status=claim['status'],claim=claim,
            reason='Locked Euro domain gates failed on WC; unsupported confirmation stays unsupported'))
    if stop_path.exists():
        return _submission_finish(logger,run_dir,dict(base,status='stopped',claim_status='not_started',
                                  resume='rerun identical command with --resume'))
    rows=primary['rows']
    with logger.stage('confirmation_point_analyses'):
        sensitivity=dvds_sensitivity(rows,config['feature_names'],fit_rows=rows,seed=config['seed'])
        save_complex_result(sensitivity,run_dir/'exports/sensitivity')
        subgroups=submission_subgroups(rows,config,numerical_draws=10000)
        atomic_write_json(run_dir/'exports/subgroups_joint.json',subgroups)
        influence=submission_score_influence(rows,top_matches=5,top_teams=2)
        atomic_write_json(run_dir/'exports/score_influence.json',influence)
    target=int(config['bootstrap_target']); attempts=int(config.get('bootstrap_max_attempts',math.ceil(1.5*target)))
    maximum=int(config.get('bootstrap_max_target',500))
    analyses=[dict(spec,task_id=f'analysis_{i}_{spec["kind"]}_{spec["name"]}',seed=_submission_seed(config,'confirmation_analysis',i))
              for i,spec in enumerate(submission_analysis_specs(config))]
    analyses+=[dict(kind='targeted_match',name=f'remove_match_{match}',remove_matches=[match],task_id=f'targeted_match_{i}',
                    seed=_submission_seed(config,'targeted_match',i)) for i,match in enumerate(influence['targeted_matches'])]
    analyses+=[dict(kind='targeted_team',name=f'remove_team_{team}',team=team,task_id=f'targeted_team_{i}',
                    seed=_submission_seed(config,'targeted_team',i)) for i,team in enumerate(influence['targeted_teams'])]
    null_kinds=['within_match_treatment','independent_dummy_y']; scenarios=submission_scenarios()
    null_reps=int(config['null_reps']); simulation_reps=int(config['simulation_reps'])
    identity=dict(provenance=provenance,frozen_design_hash=provenance['design_lock_hash'] or joblib.hash(design),
                  procedure='submission_confirmation_tasks_v1')
    def stopped(stage):
        return _submission_finish(logger,run_dir,dict(base,status='stopped',stopped_during=stage,claim_status='not_evaluated',
                                  resume='rerun identical command with --resume; completed task records are reused'))
    with SubmissionTaskPool(run_dir/'tasks',identity,workers=config['workers'],executor_kind=runtime.get('executor_kind','process'),
                            logger=logger,study_root=study_root,handle_signals=True,
                            progress_path=run_dir/'overall_progress.json') as pool:
        pool.plan('confirmation_bootstrap',target,'fixed_refit',target_success=True)
        pool.plan('confirmation_analyses',len(analyses),'analysis')
        for kind in null_kinds: pool.plan('confirmation_null_'+kind,null_reps,'null_'+kind)
        for scenario in scenarios: pool.plan('confirmation_simulation_'+scenario['name'],simulation_reps,'simulation')
        with logger.stage('confirmation_bootstrap',total=target):
            refit=partial(submission_refit_task,frame=frame,config=config,frozen=design)
            draws=pool.run('confirmation_bootstrap',_submission_specs(config,'confirmation_bootstrap',attempts,kind='refit'),
                           refit,target_success=target)
            precision_seed=_submission_seed(config,'bootstrap_precision',0)
            # Precision is judged on the first `target` successes in attempt order, whatever a resume cached.
            bootstrap=submission_bootstrap_summary(_submission_success_prefix(draws,target),target,precision_seed,maximum=maximum)
            extended=False
            if not pool.stopped and bootstrap['complete'] and config.get('bootstrap_precision_extension') and bootstrap['extension_recommended']:
                # Predeclared one-step Monte Carlo precision extension; never driven by significance.
                logger.event('bootstrap_precision_extension',from_successes=bootstrap['successful'],to_successes=maximum,
                             endpoint_mcse=bootstrap['endpoint_mcse'])
                pool.plan('confirmation_bootstrap',maximum,'fixed_refit',target_success=True); extended=True
                draws=pool.run('confirmation_bootstrap',_submission_specs(config,'confirmation_bootstrap',math.ceil(1.5*maximum),kind='refit'),
                               refit,target_success=maximum)
                draws=_submission_success_prefix(draws,maximum)
                bootstrap=submission_bootstrap_summary(draws,target,precision_seed,maximum=maximum)
            else:
                draws=_submission_success_prefix(draws,target)
            bootstrap.update(extension_executed=extended,draws=[{k:r.get(k) for k in ['task_id','seed','status','att','score_interval',
                             'n_shots','n_matches','control_ess','diagnostics','failure_category','error','elapsed_seconds']} for r in draws])
            atomic_write_json(run_dir/'exports/bootstrap_summary.json',bootstrap)
        if pool.stopped: return stopped('confirmation_bootstrap')
        with logger.stage('confirmation_analyses',total=len(analyses)):
            records=pool.run('confirmation_analyses',analyses,partial(submission_analysis_task,rows=rows,config=config,frozen=design))
            analysis=dict(requested=len(analyses),attempted=len(records),successful=sum(r.get('status')=='ok' for r in records),
                          computational_failures=sum(r.get('failure_category')=='computational' for r in records),
                          records=[{k:v for k,v in r.items() if k!='traceback'} for r in records])
            atomic_write_json(run_dir/'exports/analyses.json',analysis)
        if pool.stopped: return stopped('confirmation_analyses')
        nulls={}
        for kind in null_kinds:
            stage='confirmation_null_'+kind
            with logger.stage(stage,total=null_reps):
                records=pool.run(stage,_submission_specs(config,stage,null_reps,kind='null',null_kind=kind),
                                 partial(submission_null_task,baseline_rows=rows,config=config,frozen=design))
                nulls[kind]=dict(submission_science_summary(records,null_reps,'null'),
                                 records=[{k:v for k,v in r.items() if k!='traceback'} for r in records])
            if pool.stopped: return stopped(stage)
        atomic_write_json(run_dir/'exports/nulls.json',nulls)
        simulation_records,by_scenario=[],{}
        for scenario in scenarios:
            stage='confirmation_simulation_'+scenario['name']
            with logger.stage(stage,total=simulation_reps):
                records=pool.run(stage,_submission_specs(config,stage,simulation_reps,kind='simulation',scenario=scenario,
                                 n_matches=int(config.get('simulation_matches',48)),
                                 decisions_per_match=int(config.get('simulation_decisions_per_match',250))),
                                 partial(submission_simulation_task,config=config,frozen=design))
                by_scenario[scenario['name']]=submission_science_summary(records,simulation_reps,'simulation')
                simulation_records.extend(records)
            if pool.stopped: return stopped(stage)
        simulations=dict(submission_science_summary(simulation_records,simulation_reps*len(scenarios),'simulation'),by_scenario=by_scenario)
        atomic_write_json(run_dir/'exports/simulations.json',simulations)
    agreement=submission_inference_agreement(point.get('cluster_score_se'),bootstrap)
    precision=bool(bootstrap['precision_passed'] or (extended and bootstrap['successful']>=maximum))
    checks=dict(support=supported,bootstrap=bool(bootstrap['complete']),precision=precision,
                inference_agreement=bool(agreement['passed']),
                nulls=all(_submission_null_check(nulls[kind]) for kind in null_kinds),
                simulations=_submission_simulation_check(simulations),
                analyses=bool(analysis['attempted']==analysis['requested'] and not analysis['computational_failures']))
    claim=submission_claim_status(checks)
    claim['precision_limited_at_maximum']=bool(precision and not bootstrap['precision_passed'])
    results=dict(point=point,diagnostics=_compact_inference_diagnostics(primary['diagnostics']),bootstrap={k:v for k,v in bootstrap.items() if k!='draws'},
                 inference_agreement=agreement,analyses=analysis,nulls={k:{kk:vv for kk,vv in v.items() if kk!='records'} for k,v in nulls.items()},
                 simulations=simulations,subgroups=subgroups,influence=influence,
                 sensitivity=dict(bounds=sensitivity['bounds'].to_dict('records'),tipping=sensitivity['tipping'],
                                  monotonicity=sensitivity['monotonicity'],limitations=sensitivity['limitations']),
                 timing_audit=timing,claim=claim,provenance_hash=stable_hash(provenance),
                 units='signed first-goal reward per supported decision (x100 per 100 supported decisions)')
    atomic_write_json(run_dir/'submission_results.json',results)
    return _submission_finish(logger,run_dir,dict(base,status=claim['status'],claim_status=claim['status'],claim=claim,
                              results_path=str(run_dir/'submission_results.json')))


## Execute the configured phase

The default below is an engineering smoke run, not a publishable result. The runner injects and saves production settings. Long phases may remain running for days; read `status.json`/`events.jsonl` and the phase result before interpreting completion. A diagnostics failure stops confirmation and exports the limitation instead of silently relaxing support.


In [ ]:
# %% Submission independent simulations and software null contracts
def submission_scenarios():
    """Eight frozen DGP stresses, not correctness claims for finite trees."""
    definitions = [
        ('healthy_zero', 0, 'adequate', True, True, None),
        ('healthy_positive', 1, 'adequate', True, True, None),
        ('healthy_negative', -1, 'adequate', True, True, None),
        ('weak_overlap_zero', 0, 'weak', True, True, None),
        ('nonlinear_propensity', 1, 'adequate', False, True, None),
        ('nonlinear_outcome', 1, 'adequate', True, False, None),
        ('both_nonlinear', 0, 'adequate', False, False, None),
        ('action_visibility', 1, 'adequate', True, True, 'visibility')]
    return [dict(name=name, effect_sign=sign, overlap=overlap, e_correct=e_linear,
                 m_correct=m_linear, stress=stress, family='submission8',
                 dgp_propensity_form='linear_logit' if e_linear else 'nonlinear_logit',
                 dgp_outcome_form='linear_mean' if m_linear else 'nonlinear_mean',
                 identification_failure_expected=stress == 'visibility',
                 correctness_interpretation='DGP forms; adequacy depends on the locked fitted backend, not a finite-tree correctness assertion')
            for name, sign, overlap, e_linear, m_linear, stress in definitions]


def submission_simulation_pair(scenario, seed, n_matches=48, decisions_per_match=250):
    """Independent latent match draws and disjoint identities for both roles."""
    if int(n_matches) < 2 or int(decisions_per_match) < 1:
        raise ValueError('Simulation requires at least two matches and positive decisions')
    result = []
    for role_id, role in enumerate(['design', 'confirmation']):
        role_seed = int(np.random.SeedSequence([int(seed), 3187, role_id]).generate_state(1)[0])
        frame = simulate_causal_matches(scenario, role_seed, n_matches, decisions_per_match)
        prefix = f'{scenario["name"]}:{int(seed)}:{role}:'
        for column in ['original_match_id', 'match_id', 'decision_row_id', 'event_id']:
            frame[column] = prefix + frame[column].astype(str)
        frame.attrs['simulation_role'] = role
        frame.attrs['simulation_seed'] = role_seed
        result.append(frame)
    if not set(result[0].original_match_id).isdisjoint(result[1].original_match_id):
        raise AssertionError('Synthetic design/confirmation match leakage')
    return tuple(result)


def submission_selected_truth(rows):
    """Conditional mean effect for actual supported treated confirmation rows.

    The visibility generator already integrates potential means over the
    action-dependent selected treated distribution. Do not use original means.
    """
    treatment = rows['Treatment'] if 'Treatment' in rows else rows['T']
    selected = rows.loc[rows.support.fillna(False).astype(bool) & treatment.eq(1)]
    if not len(selected):
        raise ValueError('Selected truth has no supported treated targets')
    effects = (selected.true_y1 - selected.true_y0).to_numpy(float)
    if not np.isfinite(effects).all():
        raise ValueError('Selected simulation truth is nonfinite')
    return float(effects.mean())


def submission_null_frame(rows, kind, seed):
    """Treatment null refits nuisances; dummy Y reuses only treatment design."""
    rng = np.random.default_rng(int(seed))
    sample = rows.copy()
    if kind == 'within_match_treatment':
        if 'original_match_id' not in sample:
            raise ValueError('Treatment randomization requires original match IDs')
        sample['T'] = sample.groupby('original_match_id', sort=True)['T'].transform(
            lambda column: rng.permutation(column.to_numpy()))
        sample['Treatment'] = sample['T'].astype(int)
        sample = sample.drop(columns=['e', 'm0', 'm1', 'portable_e', 'support',
            'outer_fold', 'm0_clipped', 'm1_clipped', 'domain_selection_passed',
            'support_reason', 'support_threshold', 'football_cell'], errors='ignore')
    elif kind == 'independent_dummy_y':
        sample['Y'] = rng.choice([-1, 0, 1], len(sample), p=[.01, .98, .01])
        sample = sample.drop(columns=['m0', 'm1', 'm0_clipped', 'm1_clipped'], errors='ignore')
    else:
        raise ValueError(f'Unknown submission null: {kind}')
    return sample


def _submission_science_record(spec):
    return dict(kind=spec.get('kind'), seed=int(spec['seed']), replicate=int(spec.get('replicate', 0)),
                status='running', diagnostics={})


def _submission_science_failure(record, exc):
    record.update(status='failed', exception_type=type(exc).__name__, failure=str(exc),
                  failure_category='structural' if isinstance(exc, ValueError) else 'computational',
                  traceback=traceback.format_exc())


def submission_simulation_task(spec, config, frozen, logger=None):
    """One independent design/confirmation pair, production fitter and score CI.

    Locked nuisance classes/backends are reused. Synthetic design trains only
    portable/full propensity OOF required for domain learning, with no m0/m1
    model selection. Confirmation does direct fixed e/m0 fitting. No bootstrap.
    """
    started = time.monotonic()
    record = _submission_science_record(spec)
    scenario = spec['scenario']
    record.update(scenario=scenario['name'], effect_sign=scenario['effect_sign'],
                  dgp_propensity_form=scenario['dgp_propensity_form'],
                  dgp_outcome_form=scenario['dgp_outcome_form'],
                  identification_failure_expected=bool(scenario.get('identification_failure_expected')),
                  locked_model_specs=copy.deepcopy(frozen['model_specs']),
                  correctness_interpretation=scenario['correctness_interpretation'])
    progress = logger or _worker_progress(config, f'simulation_{scenario["name"]}_{record["replicate"]}')
    try:
        design_frame, confirmation = submission_simulation_pair(scenario, record['seed'],
            int(spec.get('n_matches', config.get('simulation_matches', 48))),
            int(spec.get('decisions_per_match', config.get('simulation_decisions_per_match', 250))))
        design_config = scientific_config(dict(config, seed=design_frame.attrs['simulation_seed'],
                                               att_only=True, diagnostic_m1=False, workers=1))
        design_rows = prepare_analysis(design_frame, design_config)
        design_started = time.monotonic()
        e_spec = frozen['model_specs']['e']
        full = fit_propensity(design_rows, design_config, design_config['feature_names'],
                              frozen_spec=e_spec, logger=progress, stage='simulation_design_e', require_oof=True)
        portable = fit_propensity(design_rows, design_config, design_config['support_features'],
                                  frozen_spec={'kind': 'logistic', 'C': .2}, logger=progress,
                                  stage='simulation_design_portable', require_oof=True)
        domain = select_domain(design_rows, portable['oof'], full['oof'], design_config)
        synthetic_frozen = dict(portable_model=portable, domain=domain, config=design_config,
            model_specs=copy.deepcopy(frozen['model_specs']),
            training_match_ids=sorted(design_rows.original_match_id.astype(str).unique()),
            role='independent_synthetic_design_locked_nuisance_specs')
        design_ids, confirmation_ids = set(design_rows.original_match_id), set(confirmation.original_match_id)
        if not design_ids.isdisjoint(confirmation_ids):
            raise AssertionError('Synthetic design/confirmation original-match leakage')
        record.update(design_seconds=time.monotonic()-design_started,
                      design_seed=design_frame.attrs['simulation_seed'],
                      confirmation_seed=confirmation.attrs['simulation_seed'],
                      design_match_hash=stable_hash(sorted(design_ids)),
                      confirmation_match_hash=stable_hash(sorted(confirmation_ids)),
                      independent_design_confirmation=True,
                      design_support_passed=bool(domain.get('selection_passed', False)))
        fitting_config = scientific_config(dict(config, seed=confirmation.attrs['simulation_seed'],
                                                att_only=True, diagnostic_m1=False, workers=1))
        confirmation_started = time.monotonic()
        fitted = fit_study(confirmation, fitting_config, frozen_design=synthetic_frozen, logger=progress)
        estimate = att_estimate(fitted['rows'])
        truth = submission_selected_truth(fitted['rows'])
        low, high = estimate['score_interval']
        target = fitted['rows'].loc[fitted['rows'].support & fitted['rows'].Treatment.eq(1)]
        record.update(status='ok', att=estimate['att'], score_interval=[low, high], truth=truth,
            error=estimate['att']-truth, covered=bool(low <= truth <= high),
            reject_zero=bool(low > 0 or high < 0),
            ci_kind=estimate['score_interval_kind'], score_df=estimate['cluster_score_df'],
            n_shots=estimate['n_shots'], eligible_matches=estimate['eligible_matches'],
            control_ess=estimate['control_ess'],
            realized_selected_truth=float((target.potential_y1-target.potential_y0).mean()),
            confirmation_seconds=time.monotonic()-confirmation_started,
            diagnostics=_compact_inference_diagnostics(fitted['diagnostics']),
            interpretation='Operating characteristics conditional on computationally valid pairs; DGP adequacy and visibility stress are reported without retuning')
    except Exception as exc:
        _submission_science_failure(record, exc)
    record['elapsed_seconds'] = time.monotonic()-started
    if logger is None:
        _submission_close_progress(progress, record)
    return record


def submission_null_task(spec, baseline_rows, config, frozen, logger=None):
    """One software null: fresh randomized-treatment e/m0 or dummy-Y m0 only."""
    started = time.monotonic()
    record = _submission_science_record(spec)
    kind = spec['null_kind']
    record.update(null_kind=kind, known_assignment_status='not_applicable', known_assignment_att=None)
    progress = logger or _worker_progress(config, f'null_{kind}_{record["replicate"]}')
    try:
        fitting_config = scientific_config(dict(config, seed=record['seed'], att_only=True,
                                                diagnostic_m1=False, workers=1))
        sample = submission_null_frame(baseline_rows, kind, record['seed'])
        if kind == 'within_match_treatment':
            fitted = fit_study(sample, fitting_config, frozen_design=frozen, logger=progress)
            rows = fitted['rows']
            record['reuse_policy'] = 'fresh_e_and_m0_frozen_portable_domain'
        else:
            rows, provenance = refit_on_fixed_population(sample, fitting_config,
                outcome_spec=frozen['model_specs']['m0'], logger=progress)
            fitted = dict(rows=rows, diagnostics=support_diagnostics(rows, fitting_config), provenance=provenance)
            if not rows.e.equals(baseline_rows.e) or not rows.support.equals(baseline_rows.support):
                raise AssertionError('Dummy null changed treatment-only e/support design')
            record['reuse_policy'] = 'reuse_e_support_folds_fresh_m0'
        estimate = att_estimate(rows)
        low, high = estimate['score_interval']
        record.update(status='ok', att=estimate['att'], score_interval=[low, high],
            reject_zero=bool(low > 0 or high < 0), ci_kind=estimate['score_interval_kind'],
            score_df=estimate['cluster_score_df'], n_shots=estimate['n_shots'],
            eligible_matches=estimate['eligible_matches'],
            diagnostics=_compact_inference_diagnostics(fitted['diagnostics']))
        if kind == 'within_match_treatment':
            probability = rows.groupby('original_match_id')['T'].transform('mean').to_numpy(float)
            if ((probability > 0) & (probability < 1)).all():
                oracle = rows.copy()
                oracle['e'] = probability
                record.update(known_assignment_status='available', known_assignment_att=att_estimate(oracle)['att'])
            else:
                record['known_assignment_status'] = 'unavailable_nonpositive_original_match_stratum'
        record['interpretation'] = 'Software randomization check; not a test of observational exchangeability'
    except Exception as exc:
        _submission_science_failure(record, exc)
    record['elapsed_seconds'] = time.monotonic()-started
    if logger is None:
        _submission_close_progress(progress, record)
    return record


def _submission_mc_binomial_interval(count, total):
    if not total:
        return None
    z = float(norm.ppf(.975))
    p = count / total
    denominator = 1 + z*z/total
    center = (p + z*z/(2*total)) / denominator
    radius = z*np.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
    return [float(max(0., center-radius)), float(min(1., center+radius))]


def submission_science_summary(records, requested, kind):
    """Retain finite poor-diagnostic draws; separate computational completeness."""
    valid = [record for record in records if record.get('status') == 'ok' and np.isfinite(record.get('att', np.nan))]
    successful = len(valid)
    values = np.asarray([record['att'] for record in valid], float)
    result = dict(kind=kind, requested=int(requested), attempted=len(records), successful=successful,
        failed=sum(record.get('status') == 'failed' for record in records),
        missing=max(0, int(requested)-len(records)),
        complete=len(records) == int(requested) and successful == int(requested),
        diagnostic_poor_successes=sum(not bool(record.get('diagnostics', {}).get('passed', False)) for record in valid),
        mean_att=float(values.mean()) if successful else None,
        mean_att_mcse=float(values.std(ddof=1)/np.sqrt(successful)) if successful > 1 else None,
        failure_categories={name: sum(record.get('failure_category') == name for record in records)
                            for name in ['structural', 'computational']},
        denominator='computationally_valid_records; failures and missing allocations separately reported')
    rejected = sum(bool(record.get('reject_zero')) for record in valid)
    result.update(reject_zero_rate=rejected/successful if successful else None,
                  reject_zero_mc_interval=_submission_mc_binomial_interval(rejected, successful))
    if kind == 'simulation':
        errors = np.asarray([record['att']-record['truth'] for record in valid], float)
        covered = sum(bool(record.get('covered')) for record in valid)
        result.update(bias=float(errors.mean()) if successful else None,
            bias_mcse=float(errors.std(ddof=1)/np.sqrt(successful)) if successful > 1 else None,
            rmse=float(np.sqrt(np.mean(errors**2))) if successful else None,
            coverage=covered/successful if successful else None,
            coverage_mc_interval=_submission_mc_binomial_interval(covered, successful),
            interpretation='Simulation operating characteristics under locked fitted classes; visibility/misspecification failures remain descriptive')
    else:
        result['interpretation'] = 'Software null behavior, not identification evidence'
    return result


In [ ]:
from PublicCausal.notebook_runtime import load_notebook_core
STUDY_CONFIG['root'] = str(PROJECT_ROOT)
STUDY_CONFIG.setdefault('study_root', str(PROJECT_ROOT / 'data' / 'public_statsbomb_causal'))
STUDY_CONFIG.setdefault('run_dir', str(Path(STUDY_CONFIG['study_root']) / 'runs' / STUDY_CONFIG['run_id'] / STUDY_CONFIG['phase']))
_core = load_notebook_core(STUDY_CONFIG.get('notebook_path', PROJECT_ROOT / 'results/models/public_statsbomb_causal_study.ipynb'), Path(STUDY_CONFIG['run_dir']) / 'runtime')
if 'core_source_hash' in STUDY_CONFIG:
    assert STUDY_CONFIG['core_source_hash'] == _core.CORE_SOURCE_HASH, 'Scientific source changed during execution'
STUDY_CONFIG['core_source_hash'] = _core.CORE_SOURCE_HASH
STUDY_RESULT = _core.execute_public_study(STUDY_CONFIG)
STUDY_RESULT


## Results report

Everything below reads the saved artifacts of the run of record and displays them; it refits nothing, never opens raw data and adds no new estimands. Report cells are tagged `public-causal-report`, not `public-causal-core`, so they never change the scientific core hash bound by the design lock.

The layout follows the earlier support-aware causal notebooks: Query → Model → Identify → Estimate → Refute → Interpretation. Missing artifacts (pilot, stopped or unsupported runs) are shown as not available. Tables and figures are also written to the run's `report/` folder.


In [ ]:
# %% Results report: reads saved run artifacts only (no refits, no data access, no new estimands)
import json as _rjson
from pathlib import Path as _RPath

import numpy as _rnp
import pandas as _rpd

REPORT_NOT_AVAILABLE = 'not available for this run'


def report_paths(phase_dir):
    """Locate development/results folders for a `full` phase or a single-phase run directory."""
    phase_dir = _RPath(phase_dir)
    development = phase_dir / 'development'
    confirmation = phase_dir / 'confirmation'
    return dict(phase_dir=phase_dir,
                development=development if development.is_dir() else phase_dir,
                results=confirmation if confirmation.is_dir() else phase_dir)


def _report_json(path):
    path = _RPath(path)
    return _rjson.loads(path.read_text()) if path.is_file() else None


def _report_parquet(path, columns=None):
    path = _RPath(path)
    if not path.is_file():
        return None
    frame = _rpd.read_parquet(path)
    return frame[[c for c in columns if c in frame.columns]] if columns else frame


def _report_csv(path):
    path = _RPath(path)
    return _rpd.read_csv(path) if path.is_file() else None


def _missing(what):
    return [('md', f'*{what}: {REPORT_NOT_AVAILABLE}.*')]


def _science(paths):
    for folder in (paths['results'], paths['development']):
        config = _report_json(folder / 'config.json')
        if config:
            return config.get('science', config)
    return {}


def _kv(mapping, keys=None, labels=None):
    keys = keys or list(mapping)
    labels = labels or {}
    return _rpd.DataFrame({'value': [mapping.get(k) for k in keys]}, index=[labels.get(k, k) for k in keys])


def _interval(values):
    return f'[{values[0]:.4f}, {values[1]:.4f}]' if values is not None and len(values) == 2 and values[0] is not None else '—'


def _figure(width=7, height=3.6, ncols=1):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, ncols, figsize=(width, height), constrained_layout=True)
    return fig, axes


def _smd(treated, control, weights=None):
    treated, control = _rnp.asarray(treated, float), _rnp.asarray(control, float)
    weights = _rnp.ones(len(control)) if weights is None else _rnp.asarray(weights, float)
    keep_t, keep_c = _rnp.isfinite(treated), _rnp.isfinite(control) & _rnp.isfinite(weights)
    a, b, w = treated[keep_t], control[keep_c], weights[keep_c]
    if len(a) < 2 or len(b) < 2 or w.sum() <= 0:
        return _rnp.nan
    bm = _rnp.average(b, weights=w)
    scale = _rnp.sqrt((a.var() + _rnp.average((b - bm) ** 2, weights=w)) / 2)
    return float(abs(a.mean() - bm) / scale) if scale > 1e-12 else 0.


# ----------------------------------------------------------------------------- sections

def section_overview(paths):
    items = []
    phase = _report_json(paths['results'] / 'phase_result.json')
    results = _report_json(paths['results'] / 'submission_results.json')
    lock = _report_json(paths['development'] / 'design_lock.json')
    if phase is None and results is None:
        return _missing('Phase result')
    config = _report_json(paths['results'] / 'config.json') or {}
    run_id = config.get('runtime', {}).get('run_id') or config.get('science', {}).get('run_id') or paths['phase_dir'].parent.name
    claim = (results or {}).get('claim') or (phase or {}).get('claim') or {}
    overview = dict(run_id=run_id, phase=(phase or {}).get('phase'), phase_status=(phase or {}).get('status'),
                    claim_status=claim.get('status') or (phase or {}).get('claim_status'),
                    units=(results or {}).get('units'),
                    design_lock=(lock or {}).get('status'), confirmation_allowed=(lock or {}).get('confirmation_allowed'),
                    frozen_at=(lock or {}).get('frozen_at'),
                    core_source_hash=((lock or {}).get('core_source_hash') or '')[:16] or None,
                    provenance_hash=((results or {}).get('provenance_hash') or '')[:16] or None)
    items.append(('table', 'Run of record', _kv(overview)))
    checks = claim.get('checks')
    if checks:
        items.append(('table', 'Claim checks (all must pass for `completed`)',
                      _rpd.DataFrame({'passed': [bool(v) for v in checks.values()]}, index=list(checks))))
    if claim.get('qualification'):
        items.append(('md', f"**Qualification.** {claim['qualification']}"))
    if claim.get('failed_or_missing'):
        items.append(('md', f"**Failed or missing checks:** {', '.join(map(str, claim['failed_or_missing']))}"))
    return items


def section_cohort(paths):
    items = []
    for label, folder in (('Euro development', paths['development']), ('World Cup confirmation', paths['results'])):
        audit = _report_json(folder / 'timing_audit.json')
        if audit is None or (label.startswith('World') and folder == paths['development']):
            continue
        exclusions = {k: v for k, v in audit.get('exclusions', {}).items() if k}
        total = int(sum(exclusions.values()) + audit.get('eligible', 0))
        combos = _rpd.DataFrame({'decisions': exclusions}).sort_values('decisions', ascending=False)
        combos['share of candidates'] = combos.decisions / max(total, 1)
        reasons = {}
        for key, count in exclusions.items():
            for reason in key.split('|'):
                reasons[reason] = reasons.get(reason, 0) + count
        flow = _rpd.DataFrame({'value': [total, audit.get('eligible'), audit.get('matches')]},
                              index=['candidate decisions', 'eligible decisions', 'matches'])
        items.append(('table', f'{label}: cohort flow', flow))
        items.append(('table', f'{label}: exclusion combinations', combos))
        fig, ax = _figure(7, 3.2)
        series = _rpd.Series(reasons).sort_values()
        ax.barh(series.index, series.values, color='#4c72b0')
        ax.set_xlabel('decisions excluded (a decision can carry several reasons)')
        ax.set_title(f'{label}: exclusion reasons')
        items.append(('figure', f'{label} exclusion reasons', fig))
        if audit.get('limitation'):
            items.append(('md', f"**Timing limitation.** {audit['limitation']}"))
    att = _report_json(paths['results'] / 'exports/att.json')
    if att:
        items.append(('table', 'Supported analysis sample', _kv(att, ['n_shots', 'n_controls', 'n_matches', 'control_ess'],
                      dict(n_shots='supported shots (treated)', n_controls='supported continuation decisions',
                           n_matches='matches', control_ess='control effective sample size'))))
    return items or _missing('Timing audit')


def section_design(paths):
    lock = _report_json(paths['development'] / 'design_lock.json')
    gates = _report_json(paths['development'] / 'development_gates.json')
    items = []
    if gates:
        items.append(('table', 'Development (Euro) lock gates', _rpd.DataFrame({'passed': [bool(v) for v in gates.values()]}, index=list(gates))))
    if lock is None:
        return items + _missing('Design lock')
    manifest = lock.get('specification_manifest', {})
    model = manifest.get('model', {})
    specs = model.get('specifications', {})
    frozen = {f'model {k}': _rjson.dumps(v, sort_keys=True) for k, v in specs.items()}
    frozen.update({'propensity selection rule': model.get('propensity_selection'), 'outer folds': model.get('outer_folds'),
                   'primary horizon': _rjson.dumps(manifest.get('horizon', {}).get('primary')),
                   'backend': _rjson.dumps(manifest.get('backend')), 'primary inference': manifest.get('inference', {}).get('primary'),
                   'subgroup strata': _rjson.dumps(manifest.get('subgroup', {}).get('strata')),
                   'profile': lock.get('profile'), 'frozen at': lock.get('frozen_at')})
    items.append(('table', 'Frozen design (immutable before World Cup access)', _rpd.DataFrame({'value': frozen})))
    validation = _report_json(paths['development'] / 'weighted_validation.json')
    if validation and validation.get('calibration'):
        items.append(('table', 'Euro weighted calibration (held-out bins)', _rpd.DataFrame(validation['calibration'])))
    return items


def section_model_fit(paths):
    items = []
    nuisance = _report_parquet(paths['results'] / 'exports/nuisance_rows.parquet', ['T', 'Treatment', 'e', 'portable_e', 'm0', 'Y15', 'support'])
    if nuisance is not None and 'e' in nuisance:
        treat = (nuisance['Treatment'] if 'Treatment' in nuisance else nuisance['T']).astype(int).to_numpy()
        e = _rnp.clip(nuisance.e.to_numpy(float), 1e-6, 1 - 1e-6)
        base = treat.mean()
        metrics = dict(decisions=len(e), shot_rate=base,
                       log_loss=float(-_rnp.mean(treat * _rnp.log(e) + (1 - treat) * _rnp.log(1 - e))),
                       log_loss_constant=float(-_rnp.mean(treat * _rnp.log(base) + (1 - treat) * _rnp.log(1 - base))),
                       brier=float(_rnp.mean((e - treat) ** 2)))
        try:
            from sklearn.metrics import roc_auc_score
            metrics['auc'] = float(roc_auc_score(treat, e))
        except Exception:
            pass
        items.append(('table', 'Held-out propensity fit (out-of-fold predictions, all eligible decisions)', _kv(metrics)))
        bins = _rpd.qcut(_rpd.Series(e), 10, duplicates='drop')
        calib = _rpd.DataFrame(dict(predicted=e, observed=treat)).groupby(bins.values, observed=True).mean()
        fig, ax = _figure(4.6, 4.2)
        ax.plot([0, 1], [0, 1], color='grey', lw=1, ls='--')
        ax.plot(calib.predicted, calib.observed, marker='o')
        ax.set_xlabel('predicted shot propensity (decile mean)'); ax.set_ylabel('observed shot rate')
        ax.set_title('Propensity calibration (held-out)')
        items.append(('figure', 'propensity calibration', fig))
    outcome = _report_csv(paths['results'] / 'exports/outcome_validation.csv')
    if outcome is not None:
        items.append(('table', 'Outcome model validation (held-out, by arm/population)', outcome))
    calibration = _report_csv(paths['results'] / 'exports/weighted_calibration.csv')
    if calibration is not None and {'predicted', 'observed'} <= set(calibration.columns):
        fig, ax = _figure(4.6, 4.2)
        for arm, group in calibration.groupby('arm' if 'arm' in calibration else _rnp.zeros(len(calibration))):
            ax.plot(group.predicted, group.observed, marker='o', label=f'arm {arm}')
        low, high = calibration[['predicted', 'observed']].min().min(), calibration[['predicted', 'observed']].max().max()
        ax.plot([low, high], [low, high], color='grey', lw=1, ls='--')
        ax.set_xlabel('predicted continuation reward'); ax.set_ylabel('observed reward'); ax.legend()
        ax.set_title('Outcome calibration (held-out)')
        items.append(('figure', 'outcome calibration', fig))
    return items or _missing('Model validation artifacts')


def section_support(paths):
    diagnostics = _report_json(paths['results'] / 'exports/support_diagnostics.json')
    if diagnostics is None:
        return _missing('Support diagnostics')
    science = _science(paths)
    items = []
    if diagnostics.get('gates'):
        items.append(('table', 'Support gates', _rpd.DataFrame({'passed': [bool(v) for v in diagnostics['gates'].values()]},
                                                               index=list(diagnostics['gates']))))
    keys = ['n_shots', 'n_controls', 'control_ess', 'key_max_smd', 'other_max_smd', 'max_weight_share', 'max_match_weight_share', 'shot_retention']
    thresholds = dict(key_max_smd=science.get('key_smd_max'), other_max_smd=science.get('other_smd_max'),
                      max_weight_share=science.get('max_weight_share'), max_match_weight_share=science.get('max_match_weight_share'))
    summary = _kv(diagnostics, keys)
    summary['threshold'] = [thresholds.get(k) for k in keys]
    items.append(('table', 'Support and weighting diagnostics', summary))
    nuisance = _report_parquet(paths['results'] / 'exports/nuisance_rows.parquet')
    rows = _report_parquet(paths['results'] / 'exports/att_contributions/rows.parquet')
    balance = diagnostics.get('balance', {})
    numeric = [f for f in science.get('feature_names', []) if f in balance]
    if nuisance is not None and rows is not None and numeric:
        treat_col = 'Treatment' if 'Treatment' in nuisance else 'T'
        table = {}
        for feature in numeric:
            if feature not in nuisance or not _rpd.api.types.is_numeric_dtype(nuisance[feature]):
                continue
            t_all, c_all = nuisance.loc[nuisance[treat_col] == 1, feature], nuisance.loc[nuisance[treat_col] == 0, feature]
            t_sup, c_sup = rows.loc[rows[treat_col] == 1, feature], rows.loc[rows[treat_col] == 0, feature]
            table[feature] = dict(eligible_unweighted=_smd(t_all, c_all), supported_unweighted=_smd(t_sup, c_sup),
                                  supported_weighted_saved=balance.get(feature),
                                  key=feature in science.get('key_balance_features', []))
        if table:
            frame = _rpd.DataFrame(table).T
            numeric_columns = ['eligible_unweighted', 'supported_unweighted', 'supported_weighted_saved']
            frame[numeric_columns] = frame[numeric_columns].astype(float)
            frame = frame.sort_values('eligible_unweighted', ascending=False)
            items.append(('table', 'Standardized mean differences: before and after support/weighting', frame))
            fig, ax = _figure(7, max(3, .32 * len(frame) + 1))
            y = _rnp.arange(len(frame))
            ax.scatter(frame.eligible_unweighted, y, label='eligible, unweighted', marker='o', color='#c44e52')
            ax.scatter(frame.supported_unweighted, y, label='supported, unweighted', marker='s', color='#dd8452')
            ax.scatter(frame.supported_weighted_saved, y, label='supported, ATT-weighted', marker='D', color='#4c72b0')
            for value, style in ((science.get('key_smd_max'), '--'), (science.get('other_smd_max'), ':')):
                if value:
                    ax.axvline(value, color='grey', ls=style, lw=1)
            ax.set_yticks(y); ax.set_yticklabels(frame.index); ax.set_xlabel('absolute SMD'); ax.legend(fontsize=8)
            ax.set_title('Covariate balance (love plot)')
            items.append(('figure', 'balance love plot', fig))
    if nuisance is not None and 'e' in nuisance:
        treat_col = 'Treatment' if 'Treatment' in nuisance else 'T'
        fig, ax = _figure(7, 3.4)
        logit = lambda p: _rnp.log(_rnp.clip(p, 1e-4, 1 - 1e-4) / (1 - _rnp.clip(p, 1e-4, 1 - 1e-4)))
        for value, label, color in ((0, 'continuation', '#4c72b0'), (1, 'shot', '#c44e52')):
            ax.hist(logit(nuisance.loc[nuisance[treat_col] == value, 'e']), bins=50, density=True, alpha=.55, label=label, color=color)
        ax.set_xlabel('held-out shot propensity (logit)'); ax.set_ylabel('density'); ax.legend()
        ax.set_title('Propensity overlap, all eligible decisions')
        items.append(('figure', 'propensity overlap', fig))
    image = paths['results'] / 'exports/support.png'
    if image.is_file():
        items.append(('image', 'Saved support figure (pipeline export)', image))
    return items


def section_estimate(paths):
    att = _report_json(paths['results'] / 'exports/att.json')
    if att is None:
        return _missing('ATT export')
    results = _report_json(paths['results'] / 'submission_results.json') or {}
    boot = _report_json(paths['results'] / 'exports/bootstrap_summary.json') or {}
    headline = _rpd.DataFrame({'value': {
        'ATT': att.get('att'), 'ATT per 100 supported decisions': att.get('att_per_100_supported_decisions'),
        'match-cluster score SE': att.get('cluster_score_se'), 'degrees of freedom': att.get('cluster_score_df'),
        'score 95% CI': _interval(att.get('score_interval')),
        'bootstrap percentile 95% CI': _interval(boot.get('percentile_interval')),
        'Hajek AIPW ATT': att.get('hajek_aipw_att'), 'supported shots': att.get('n_shots'), 'matches': att.get('n_matches')}})
    items = [('table', 'Primary supported ATT (World Cup 2022)', headline)]
    if results.get('units'):
        items.append(('md', f"Units: {results['units']}. Not EPV, net expected goals, or an unconditional scoring-probability change."))
    comparators = _report_parquet(paths['results'] / 'exports/comparator_summary.parquet')
    if comparators is not None:
        items.append(('table', 'Estimator comparators on the same supported sample', comparators))
    fig, ax = _figure(7, 2.8)
    rows = [('score CI (primary)', att.get('att'), att.get('score_interval'))]
    if boot.get('percentile_interval'):
        rows.append(('bootstrap percentile', boot.get('mean'), boot.get('percentile_interval')))
    for index, (label, centre, interval) in enumerate(rows):
        ax.errorbar(centre, index, xerr=[[centre - interval[0]], [interval[1] - centre]], fmt='o', capsize=4)
    ax.axvline(0, color='grey', lw=1)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels([r[0] for r in rows]); ax.set_xlabel('ATT (reward per supported decision)')
    ax.set_title('Primary estimate and 95% intervals')
    items.append(('figure', 'primary intervals', fig))
    scores = _report_parquet(paths['results'] / 'exports/att_contributions/cluster_scores.parquet')
    if scores is not None and 'score' in scores:
        fig, ax = _figure(7, 3)
        ordered = scores.sort_values('score')
        ax.bar(range(len(ordered)), ordered.score, color='#4c72b0')
        ax.axhline(0, color='grey', lw=1); ax.set_xlabel('match (sorted)'); ax.set_ylabel('cluster score')
        ax.set_title('Per-match score contributions')
        items.append(('figure', 'match cluster scores', fig))
    return items


def section_bootstrap(paths):
    boot = _report_json(paths['results'] / 'exports/bootstrap_summary.json')
    if boot is None:
        return _missing('Bootstrap summary')
    results = _report_json(paths['results'] / 'submission_results.json') or {}
    agreement = results.get('inference_agreement') or {}
    summary = _rpd.DataFrame({'value': {
        'requested draws': boot.get('requested'), 'successful draws': boot.get('successful'), 'attempted': boot.get('attempted'),
        'failed': boot.get('failed'), 'precision extension executed': boot.get('extension_executed'),
        'maximum successes': boot.get('maximum_successes'), 'mean': boot.get('mean'), 'SD': boot.get('sd'),
        'percentile 95% CI': _interval(boot.get('percentile_interval')),
        'endpoint MCSE (lower, upper)': _interval(boot.get('endpoint_mcse')), 'precision passed': boot.get('precision_passed'),
        'diagnostic-poor draws': boot.get('diagnostic_poor'), 'bootstrap SD / score SE': agreement.get('ratio'),
        'agreement rule': agreement.get('rule')}})
    items = [('table', 'Bootstrap summary', summary)]
    if boot.get('precision_rule'):
        items.append(('md', f"Precision rule: {boot['precision_rule']}."))
    draws = [d for d in boot.get('draws', []) if d.get('status') == 'ok' and d.get('att') is not None]
    if draws:
        science = _science(paths)
        diag = [d.get('diagnostics', {}) for d in draws]
        checks = dict(
            effective_donors=lambda d: d.get('control_ess', _rnp.inf) < max(science.get('min_ess', 0), science.get('ess_shot_fraction', 0) * d.get('n_shots', 0)),
            key_balance=lambda d: d.get('key_max_smd', 0) > science.get('key_smd_max', _rnp.inf),
            other_balance=lambda d: d.get('other_max_smd', 0) > science.get('other_smd_max', _rnp.inf),
            donor_concentration=lambda d: d.get('max_weight_share', 0) > science.get('max_weight_share', _rnp.inf),
            match_concentration=lambda d: d.get('max_match_weight_share', 0) > science.get('max_match_weight_share', _rnp.inf),
            fold_support_selection=lambda d: not d.get('fold_support_selection_passed', True))
        gates = _rpd.DataFrame({'failing draws': {name: int(sum(bool(test(d)) for d in diag)) for name, test in checks.items()}})
        gates['share'] = gates['failing draws'] / len(diag)
        items.append(('table', 'Resample support-gate failures (draws are retained per plan)', gates))
        items.append(('md', 'Cluster resampling repeats matches, which concentrates weights and worsens balance inside a resample. '
                            'Diagnostic-poor draws are kept so the interval is not selected on diagnostics; report this as a caveat.'))
        values = _rnp.array([d['att'] for d in draws])
        passed = _rnp.array([bool(d.get('diagnostics', {}).get('passed', False)) for d in draws])
        fig, ax = _figure(7, 3.4)
        bins = _rnp.histogram_bin_edges(values, bins=40)
        ax.hist([values[~passed], values[passed]], bins=bins, stacked=True, color=['#dd8452', '#4c72b0'],
                label=[f'diagnostic-poor ({(~passed).sum()})', f'all gates pass ({passed.sum()})'])
        att = (_report_json(paths['results'] / 'exports/att.json') or {}).get('att')
        if att is not None:
            ax.axvline(att, color='black', lw=1.5, label='point estimate')
        for bound in boot.get('percentile_interval') or []:
            ax.axvline(bound, color='black', ls='--', lw=1)
        ax.axvline(0, color='grey', lw=1)
        ax.set_xlabel('refit ATT'); ax.set_ylabel('draws'); ax.legend(fontsize=8)
        ax.set_title('Fixed-spec match-cluster bootstrap refits')
        items.append(('figure', 'bootstrap distribution', fig))
    return items


def section_nulls(paths):
    nulls = _report_json(paths['results'] / 'exports/nulls.json')
    if not nulls:
        return _missing('Null experiments')
    summary = _rpd.DataFrame({kind: dict(requested=v.get('requested'), successful=v.get('successful'), failed=v.get('failed'),
                                         mean_att=v.get('mean_att'), mean_att_mcse=v.get('mean_att_mcse'),
                                         reject_zero_rate=v.get('reject_zero_rate'),
                                         reject_zero_mc_interval=_interval(v.get('reject_zero_mc_interval')))
                              for kind, v in nulls.items() if isinstance(v, dict)}).T
    items = [('table', 'Null experiments (software checks, not identification evidence)', summary)]
    kinds = [k for k, v in nulls.items() if isinstance(v, dict) and v.get('records')]
    if kinds:
        fig, axes = _figure(7, 3.2, ncols=len(kinds))
        axes = _rnp.atleast_1d(axes)
        for ax, kind in zip(axes, kinds):
            values = [r['att'] for r in nulls[kind]['records'] if r.get('att') is not None]
            ax.hist(values, bins=20, color='#8172b2')
            ax.axvline(0, color='black', lw=1)
            ax.set_title(kind.replace('_', ' '), fontsize=9); ax.set_xlabel('null ATT')
        items.append(('figure', 'null distributions', fig))
    return items


def section_simulations(paths):
    simulations = _report_json(paths['results'] / 'exports/simulations.json')
    if not simulations or not simulations.get('by_scenario'):
        return _missing('Simulation study')
    table = _rpd.DataFrame({name: dict(true_att=(v.get('mean_att') or 0) - (v.get('bias') or 0), mean_att=v.get('mean_att'),
                                       bias=v.get('bias'), bias_mcse=v.get('bias_mcse'), rmse=v.get('rmse'),
                                       coverage=v.get('coverage'), coverage_mc_interval=_interval(v.get('coverage_mc_interval')),
                                       reject_zero_rate=v.get('reject_zero_rate'), successful=v.get('successful'),
                                       diagnostic_poor=v.get('diagnostic_poor_successes'))
                            for name, v in simulations['by_scenario'].items()}).T
    items = [('table', 'Simulation operating characteristics by scenario', table)]
    fig, axes = _figure(9, 3.4, ncols=2)
    y = _rnp.arange(len(table))
    cover = table.coverage.astype(float).to_numpy()
    lows = [v.get('coverage_mc_interval', [c, c])[0] for v, c in zip(simulations['by_scenario'].values(), cover)]
    highs = [v.get('coverage_mc_interval', [c, c])[1] for v, c in zip(simulations['by_scenario'].values(), cover)]
    axes[0].errorbar(cover, y, xerr=[cover - _rnp.array(lows), _rnp.array(highs) - cover], fmt='o', capsize=3)
    axes[0].axvline(.95, color='grey', ls='--', lw=1); axes[0].set_yticks(y); axes[0].set_yticklabels(table.index)
    axes[0].set_xlabel('95% CI coverage (MC interval)'); axes[0].set_title('Coverage')
    bias, mcse = table.bias.astype(float).to_numpy(), table.bias_mcse.astype(float).to_numpy()
    axes[1].errorbar(bias, y, xerr=1.96 * mcse, fmt='o', capsize=3, color='#c44e52')
    axes[1].axvline(0, color='grey', lw=1); axes[1].set_yticks(y); axes[1].set_yticklabels([])
    axes[1].xaxis.set_major_locator(__import__('matplotlib.ticker', fromlist=['MaxNLocator']).MaxNLocator(5))
    axes[1].set_xlabel('bias ± 1.96 MCSE'); axes[1].set_title('Bias')
    items.append(('figure', 'simulation coverage and bias', fig))
    return items


def section_sensitivity(paths):
    bounds = _report_parquet(paths['results'] / 'exports/sensitivity/bounds.parquet')
    summary = _report_json(paths['results'] / 'exports/sensitivity/summary.json') or {}
    if bounds is None:
        return _missing('Sensitivity bounds')
    items = [('table', 'DVDS sensitivity bounds by Gamma', bounds[[c for c in ['gamma', 'att_lower', 'att_upper', 'finite_sample_crossing'] if c in bounds]])]
    tipping = summary.get('tipping', {})
    if tipping.get('gamma_low') is not None:
        items.append(('md', f"Lower bound reaches zero between Gamma = {tipping['gamma_low']:.2f} and {tipping['gamma_high']:.2f} "
                            f"({tipping.get('status')}); this is an identification endpoint, not a significance level."))
    fig, ax = _figure(6, 3.4)
    ax.fill_between(bounds.gamma, bounds.att_lower, bounds.att_upper, alpha=.25, color='#4c72b0', label='ATT bounds')
    ax.plot(bounds.gamma, bounds.att_lower, marker='o', color='#4c72b0'); ax.plot(bounds.gamma, bounds.att_upper, marker='o', color='#4c72b0')
    ax.axhline(0, color='black', lw=1)
    if tipping.get('gamma_low') is not None:
        ax.axvline((tipping['gamma_low'] + tipping['gamma_high']) / 2, color='#c44e52', ls='--', lw=1, label='tipping point')
    ax.set_xscale('log'); ax.set_xlabel('Gamma (treatment-odds sensitivity)'); ax.set_ylabel('ATT')
    ax.legend(); ax.set_title('Unmeasured-confounding sensitivity')
    items.append(('figure', 'sensitivity curve', fig))
    if summary.get('limitations'):
        items.append(('md', f"**Limitation.** {summary['limitations']}"))
    return items


def section_robustness(paths):
    analyses = _report_json(paths['results'] / 'exports/analyses.json')
    if not analyses or not analyses.get('records'):
        return _missing('Robustness analyses')
    records = analyses['records']
    table = _rpd.DataFrame([dict(analysis=r.get('task_id'), kind=r.get('kind'), name=r.get('name'), status=r.get('status'),
                                 att=r.get('att'), ci_low=(r.get('score_interval') or [None, None])[0],
                                 ci_high=(r.get('score_interval') or [None, None])[1], n_shots=r.get('n_shots')) for r in records])
    items = [('table', 'Pre-declared analyses and targeted refits', table.set_index('analysis'))]
    ok = table.dropna(subset=['att', 'ci_low', 'ci_high']).astype(dict(att=float, ci_low=float, ci_high=float))
    if len(ok):
        att = (_report_json(paths['results'] / 'exports/att.json') or {}).get('att')
        fig, ax = _figure(7, .32 * len(ok) + 1.4)
        y = _rnp.arange(len(ok))[::-1]
        ax.errorbar(ok.att, y, xerr=[ok.att - ok.ci_low, ok.ci_high - ok.att], fmt='o', capsize=3)
        if att is not None:
            ax.axvline(att, color='#c44e52', ls='--', lw=1, label='primary')
        ax.axvline(0, color='black', lw=1)
        ax.set_yticks(y); ax.set_yticklabels([str(name).replace('analysis_', '') for name in ok.analysis]); ax.legend()
        ax.set_xlabel('ATT with score 95% CI'); ax.set_title('Robustness forest plot')
        items.append(('figure', 'robustness forest', fig))
    return items


def section_influence(paths):
    influence = _report_json(paths['results'] / 'exports/score_influence.json')
    if influence is None:
        return _missing('Influence screen')
    items = [('md', influence.get('interpretation', ''))]
    for key, label in (('matches', 'Most influential matches (removal, fixed nuisances)'), ('teams', 'Most influential teams')):
        if influence.get(key):
            frame = _rpd.DataFrame(influence[key])
            frame = frame.reindex(frame.delta.abs().sort_values(ascending=False).index).head(10)
            items.append(('table', label, frame.reset_index(drop=True)))
    if influence.get('matches'):
        frame = _rpd.DataFrame(influence['matches'])
        frame = frame.reindex(frame.delta.abs().sort_values(ascending=False).index).head(15)
        fig, ax = _figure(7, 3.2)
        ax.bar(frame.original_match_id.astype(str), frame.delta, color=_rnp.where(frame.delta > 0, '#4c72b0', '#c44e52'))
        ax.axhline(0, color='black', lw=1); ax.set_ylabel('ATT without match − ATT'); ax.tick_params(axis='x', rotation=70, labelsize=7)
        ax.set_title('Match influence')
        items.append(('figure', 'match influence', fig))
    return items


def section_subgroups(paths):
    joint = _report_json(paths['results'] / 'exports/subgroups_joint.json')
    if joint is None:
        return _missing('Subgroup analysis')
    strata = _rpd.DataFrame([dict(subgroup=s.get('subgroup'), boundary=s.get('boundary'), n_shots=s.get('n_shots'),
                                  sufficient=s.get('sufficient'), att=s.get('att'), marginal_95=_interval(s.get('marginal_interval')),
                                  simultaneous_95=_interval(s.get('simultaneous_interval'))) for s in joint.get('strata', [])])
    items = [('table', 'Subgroup strata (simultaneous intervals only for qualifying strata)', strata.set_index('subgroup') if len(strata) else strata)]
    if joint.get('contrasts'):
        items.append(('table', 'Subgroup contrasts', _rpd.DataFrame(joint['contrasts'])))
    items.append(('md', f"Joint family: {', '.join(joint.get('family', [])) or 'none'}; critical value {joint.get('critical_value')}. "
                        f"{joint.get('interpretation', '')}"))
    return items


# Pitch background: campo_futebol.png (360x193 px) is not drawn to scale, so its pixels are
# warped piecewise-linearly so that touchlines, penalty areas, goal areas, penalty spots and
# the D apex land at their true positions on a 105 x 68 m pitch (centred metres, as in the
# study). The centre circle is not pinned and stays slightly oversized. The image's baked-in
# checkerboard is removed by keeping only the dark line pixels.
_PITCH_IMAGE_NAME = 'campo_futebol.png'
_PITCH_IMAGE_CANDIDATES = (
    _RPath('results/models/pass_selection') / _PITCH_IMAGE_NAME,
    _RPath('pass_selection') / _PITCH_IMAGE_NAME,
)
_PITCH_X_PX = (0, 31, 54, 75, 85, 178.5, 273, 284, 304, 327, 359)
_PITCH_X_M = (0, 5.5, 11, 16.5, 20.15, 52.5, 84.85, 88.5, 94, 99.5, 105)       # from the left goal line
_PITCH_Y_PX = (0, 39, 65, 96, 125, 153, 192)
_PITCH_Y_M = (0, 13.84, 24.84, 34, 43.16, 54.16, 68)                            # from the top touchline
_PITCH_LAYER = {}
_CASE_COLORS = dict(teammate='#2a78d6', opponent='#eb6834', donor='#1baf7a', ink='#1f1f1e', muted='#5f5e58')


def _pitch_layer(px_per_m=10):
    """RGBA line layer of the warped pitch image, or None when the image is not available."""
    if px_per_m in _PITCH_LAYER:
        return _PITCH_LAYER[px_per_m]
    import os
    import matplotlib.image as mpimg
    candidates = ([_RPath(os.environ['PUBLIC_CAUSAL_PITCH_IMAGE'])] if os.environ.get('PUBLIC_CAUSAL_PITCH_IMAGE') else []) + list(_PITCH_IMAGE_CANDIDATES)
    path = next((p for p in candidates if p.is_file()), None)
    if path is None:
        _PITCH_LAYER[px_per_m] = None
        return None
    image = mpimg.imread(str(path)).astype(float)
    image = image if image.max() <= 1 else image / 255
    lum = image[..., :3].mean(axis=-1) * 255
    line_alpha = _rnp.clip((235 - lum) / 175, 0, 1)          # checkerboard (238/255) -> 0, lines (~43) -> 1
    width, height = int(105 * px_per_m), int(68 * px_per_m)
    src_x = _rnp.interp((_rnp.arange(width) + .5) / px_per_m, _PITCH_X_M, _PITCH_X_PX)
    src_y = _rnp.interp((_rnp.arange(height) + .5) / px_per_m, _PITCH_Y_M, _PITCH_Y_PX)
    x0 = _rnp.clip(_rnp.floor(src_x).astype(int), 0, line_alpha.shape[1] - 2); fx = src_x - x0
    y0 = _rnp.clip(_rnp.floor(src_y).astype(int), 0, line_alpha.shape[0] - 2); fy = (src_y - y0)[:, None]
    top = line_alpha[y0][:, x0] * (1 - fx) + line_alpha[y0][:, x0 + 1] * fx
    bottom = line_alpha[y0 + 1][:, x0] * (1 - fx) + line_alpha[y0 + 1][:, x0 + 1] * fx
    alpha = top * (1 - fy) + bottom * fy
    layer = _rnp.zeros((height, width, 4))
    layer[..., :3] = .16
    layer[..., 3] = _rnp.clip(alpha * 1.15, 0, 1)
    _PITCH_LAYER[px_per_m] = layer
    return layer


def draw_pitch(ax, xlim=(-52.5, 52.5), ylim=(-34, 34)):
    """Draw the pitch background in centred metres; falls back to a plain outline."""
    layer = _pitch_layer()
    line = (.16, .16, .16)
    if layer is not None:
        ax.imshow(layer, extent=(-52.5, 52.5, -34, 34), origin='upper', interpolation='bilinear', zorder=0)
    # The image's outer border is faint after resampling, so touchlines, goal lines and goal mouths are vector.
    ax.plot([-52.5, 52.5, 52.5, -52.5, -52.5], [-34, -34, 34, 34, -34], color=line, lw=1.2, zorder=0)
    for goal_x, offset in ((-52.5, -.6), (52.5, .6)):
        ax.plot([goal_x + offset] * 2, [-3.66, 3.66], color=line, lw=3, solid_capstyle='butt', zorder=0)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    return ax


def _case_frames(paths):
    """Raw 360 frames by match, display only (no estimands); None when raw data are not reachable."""
    root = next((p for p in [paths['results'], *paths['results'].parents] if (p / 'raw/data/three-sixty').is_dir()), None)
    if root is None:
        return None, None, None
    try:
        from PublicCausal.data import metric_xy, visible_polygon
    except ImportError:
        return None, None, None
    return root / 'raw/data/three-sixty', metric_xy, visible_polygon


def _case_figure(index, row, donors, snapshot, metric_xy, visible_polygon):
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    colors = _CASE_COLORS
    fig, ax = plt.subplots(figsize=(5.8, 7.4))
    draw_pitch(ax, xlim=(-1, 54), ylim=(-35, 35))
    handles = []
    if snapshot:
        polygon = _rnp.asarray(visible_polygon(snapshot['visible_area']))
        ax.fill(polygon[:, 0], polygon[:, 1], color=colors['teammate'], alpha=.08, lw=0, zorder=1)
        ax.plot(*_rnp.vstack([polygon, polygon[:1]]).T, color=colors['teammate'], alpha=.35, lw=1, zorder=1)
        handles.append(Line2D([], [], color=colors['teammate'], alpha=.35, lw=6, label='camera-visible area'))
        for teammate, marker, label in [(True, 'o', 'teammates'), (False, 'X', 'opponents')]:
            points = [metric_xy(p['location']) for p in snapshot['freeze_frame']
                      if p['teammate'] == teammate and not p.get('actor')]
            if points:
                points = _rnp.asarray(points)
                color = colors['teammate' if teammate else 'opponent']
                ax.scatter(points[:, 0], points[:, 1], s=70, marker=marker, color=color, edgecolor='white', linewidth=1.2, zorder=4)
                handles.append(Line2D([], [], ls='', marker=marker, ms=8, color=color, mec='white', label=label))
    if len(donors):
        ax.scatter(donors.x, donors.y, s=90, marker='s', facecolors='none', edgecolors=colors['donor'], linewidths=2, zorder=3)
        handles.append(Line2D([], [], ls='', marker='s', ms=9, mfc='none', mec=colors['donor'], mew=2,
                              label='matched continuation ball spots (other matches)'))
    ax.plot([row.x, 52.5], [row.y, 0], color=colors['ink'], lw=1, ls=':', zorder=2)
    ax.add_patch(plt.Circle((row.x, row.y), 3, fill=False, color=colors['ink'], lw=1.2, ls='--', zorder=5))
    ax.scatter([row.x], [row.y], s=110, color=colors['ink'], edgecolor='white', linewidth=1.5, zorder=6)
    handles = [Line2D([], [], ls='', marker='o', ms=9, color=colors['ink'], mec='white', label='shot location'),
               Line2D([], [], color=colors['ink'], ls='--', lw=1.2, label='3 m pressure disk')] + handles
    status = 'supported' if bool(row.support) else 'unsupported'
    ax.set_title(f'Case {index}: {status} shot, {row.distance:.1f} m from goal\n'
                 f'{int(row.defenders_3m)} opponent(s) within 3 m', loc='left', fontsize=10.5, color=colors['ink'])
    ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(.5, -.01), ncol=2, frameon=False, fontsize=8)
    reason = str(row.get('support_reason', '')).replace('_', ' ')
    ax.text(0, -.16, f'Support rule: {reason}. Attacking half shown. Source: StatsBomb/Hudl open data.',
            transform=ax.transAxes, fontsize=7, color=colors['muted'])
    fig.tight_layout()
    return fig


def _case_figures(paths):
    exports = paths['results'] / 'exports'
    cases = _report_parquet(exports / 'cases/cases.parquet')
    if cases is None or not len(cases):
        return None
    pairs = _report_parquet(exports / 'cases/case_donor_pairs.parquet')
    states = _report_parquet(exports / 'nuisance_rows.parquet', ['decision_row_id', 'x', 'y'])
    folder, metric_xy, visible_polygon = _case_frames(paths)
    frames, figures = {}, []
    for index, (_, row) in enumerate(cases.iterrows(), start=1):
        donors = _rpd.DataFrame(columns=['x', 'y'])
        if pairs is not None and states is not None:
            donor_ids = pairs.loc[pairs.shot_id == row.decision_row_id, 'donor_id']
            donors = states.loc[states.decision_row_id.isin(donor_ids), ['x', 'y']]
        snapshot = None
        if folder is not None:
            match = str(row.match_id)
            if match not in frames:
                source = folder / f'{match}.json'
                frames[match] = {f['event_uuid']: f for f in _rjson.loads(source.read_text())} if source.is_file() else {}
            snapshot = frames[match].get(row.event_id)
        figures.append((f'case {index}', _case_figure(index, row, donors, snapshot, metric_xy, visible_polygon)))
    return figures


def section_cases(paths):
    intro = ('md', 'Representative supported and unsupported shots with their comparators (for analyst review; '
                   'no ratings have been performed). Pitch markings: `campo_futebol.png`, warped so the penalty and '
                   'goal areas and penalty spots sit at true positions; the centre circle is approximate.')
    try:
        figures = _case_figures(paths)
    except Exception as exc:  # display-only: fall back to the pipeline's saved images
        figures = None
        intro = ('md', intro[1] + f' (Redraw failed: {type(exc).__name__}; showing saved pipeline figures.)')
    if figures:
        return [intro] + [('figure', title, fig) for title, fig in figures]
    folder = paths['results'] / 'exports/cases'
    images = sorted(folder.glob('case_*.png'), key=lambda p: int(''.join(ch for ch in p.stem if ch.isdigit()) or 0)) if folder.is_dir() else []
    if not images:
        return _missing('Case figures')
    items = [('md', 'Representative supported decisions with their comparators (for analyst review; no ratings have been performed).')]
    items.extend(('image', image.stem.replace('_', ' '), image) for image in images)
    return items


def section_limitations(paths):
    results = _report_json(paths['results'] / 'submission_results.json') or {}
    audit = _report_json(paths['results'] / 'timing_audit.json') or {}
    sensitivity = _report_json(paths['results'] / 'exports/sensitivity/summary.json') or {}
    boot = _report_json(paths['results'] / 'exports/bootstrap_summary.json') or {}
    lines = []
    if results.get('claim', {}).get('qualification'):
        lines.append(results['claim']['qualification'])
    if audit.get('limitation'):
        lines.append(audit['limitation'])
    if boot.get('successful'):
        lines.append(f"{boot.get('diagnostic_poor')} of {boot['successful']} bootstrap resamples fail at least one support diagnostic; they are retained by design.")
    tipping = sensitivity.get('tipping', {})
    if tipping.get('gamma_low') is not None:
        lines.append(f"Unmeasured confounding of treatment odds around Gamma ≈ {tipping['gamma_low']:.2f} would be enough to include zero.")
    if sensitivity.get('limitations'):
        lines.append(sensitivity['limitations'])
    if not lines:
        return _missing('Limitations summary')
    return [('md', '\n'.join(f'- {line}' for line in lines))]


REPORT_SECTIONS = [
    ('overview', 'Run of record and claim status', section_overview),
    ('cohort', 'Query: cohort flow and eligibility', section_cohort),
    ('design', 'Model: frozen Euro design', section_design),
    ('model_fit', 'Model: held-out nuisance fit', section_model_fit),
    ('support', 'Identify: support, overlap and balance', section_support),
    ('estimate', 'Estimate: supported ATT', section_estimate),
    ('bootstrap', 'Estimate: bootstrap uncertainty', section_bootstrap),
    ('nulls', 'Refute: null experiments', section_nulls),
    ('simulations', 'Refute: simulation operating characteristics', section_simulations),
    ('sensitivity', 'Refute: unmeasured-confounding sensitivity', section_sensitivity),
    ('robustness', 'Refute: pre-declared robustness analyses', section_robustness),
    ('influence', 'Refute: influential matches and teams', section_influence),
    ('subgroups', 'Interpretation: subgroups', section_subgroups),
    ('cases', 'Interpretation: case studies', section_cases),
    ('limitations', 'Interpretation: limitations', section_limitations),
]


def _report_slug(text):
    return ''.join(ch if ch.isalnum() else '_' for ch in str(text).lower()).strip('_')[:60]


def _save_items(name, items, out_dir):
    out_dir = _RPath(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for index, item in enumerate(items):
        if item[0] == 'table':
            item[2].to_csv(out_dir / f'{name}_{index:02d}_{_report_slug(item[1])}.csv')
        elif item[0] == 'figure':
            item[2].savefig(out_dir / f'{name}_{index:02d}_{_report_slug(item[1])}.png', dpi=130)


def report_section(name, phase_dir, out_dir=None):
    """Return (items, error) for one section; errors are captured so one section cannot hide the rest."""
    import traceback
    function = {key: fn for key, _, fn in REPORT_SECTIONS}[name]
    try:
        items = function(report_paths(phase_dir))
        if out_dir is not None:
            _save_items(name, items, out_dir)
        return items, None
    except Exception:
        return [('md', f'**Report section `{name}` failed:**\n\n```\n{traceback.format_exc()}\n```')], traceback.format_exc()


def build_report(phase_dir, out_dir=None):
    import matplotlib.pyplot as plt
    sections, errors = [], []
    for name, heading, _ in REPORT_SECTIONS:
        items, error = report_section(name, phase_dir, out_dir)
        sections.append(dict(name=name, heading=heading, items=items, error=error))
        if error:
            errors.append(dict(section=name, error=error))
        for item in items:
            if item[0] == 'figure':
                plt.close(item[2])
    return dict(phase_dir=str(phase_dir), sections=sections, errors=errors)


REPORT_ERRORS = []


def show_report_section(name, phase_dir, out_dir=None):
    """Notebook display of one section (tables as HTML, figures inline)."""
    import matplotlib.pyplot as plt
    from IPython.display import Image, Markdown, display
    items, error = report_section(name, phase_dir, out_dir)
    if error:
        REPORT_ERRORS.append(dict(section=name, error=error))
    for item in items:
        if item[0] == 'md':
            display(Markdown(item[1]))
        elif item[0] == 'table':
            display(Markdown(f'**{item[1]}**'))
            display(item[2])
        elif item[0] == 'figure':
            display(item[2])
            plt.close(item[2])
        elif item[0] == 'image':
            display(Markdown(f'**{item[1]}**'))
            display(Image(filename=str(item[2])))


In [ ]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib', 'inline')
REPORT_PHASE_DIR = Path(STUDY_CONFIG.get('report_phase_dir') or STUDY_CONFIG['run_dir'])
REPORT_OUT_DIR = Path(STUDY_CONFIG.get('report_out_dir') or REPORT_PHASE_DIR / 'report')
print(f'Reporting saved artifacts from {REPORT_PHASE_DIR}')
print(f'Tables and figures also saved to {REPORT_OUT_DIR}')


### Run of record and claim status


In [ ]:
show_report_section('overview', REPORT_PHASE_DIR, REPORT_OUT_DIR)


### Query


#### Cohort flow and eligibility


In [ ]:
show_report_section('cohort', REPORT_PHASE_DIR, REPORT_OUT_DIR)


### Model


#### Frozen Euro design


In [ ]:
show_report_section('design', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Held-out nuisance fit


In [ ]:
show_report_section('model_fit', REPORT_PHASE_DIR, REPORT_OUT_DIR)


### Identify


#### Support, overlap and balance


In [ ]:
show_report_section('support', REPORT_PHASE_DIR, REPORT_OUT_DIR)


### Estimate


#### Supported ATT


In [ ]:
show_report_section('estimate', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Bootstrap uncertainty


In [ ]:
show_report_section('bootstrap', REPORT_PHASE_DIR, REPORT_OUT_DIR)


### Refute


#### Null experiments


In [ ]:
show_report_section('nulls', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Simulation operating characteristics


In [ ]:
show_report_section('simulations', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Unmeasured-confounding sensitivity


In [ ]:
show_report_section('sensitivity', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Pre-declared robustness analyses


In [ ]:
show_report_section('robustness', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Influential matches and teams


In [ ]:
show_report_section('influence', REPORT_PHASE_DIR, REPORT_OUT_DIR)


### Interpretation


#### Subgroups


In [ ]:
show_report_section('subgroups', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Case studies


In [ ]:
show_report_section('cases', REPORT_PHASE_DIR, REPORT_OUT_DIR)


#### Limitations


In [ ]:
show_report_section('limitations', REPORT_PHASE_DIR, REPORT_OUT_DIR)


In [ ]:
print(f'Report sections with errors: {len(REPORT_ERRORS)}')
[error['section'] for error in REPORT_ERRORS]


## Interpretation and release boundaries

Read the saved support diagnostics before any ATT. A passing engineering smoke run does not validate a causal result. A development gate failure preserves the unseen confirmation cohort. Report the supported target, continuation mixture, horizon, signed reward components, uncertainty, sensitivity assumptions and failures alongside every finding.

Tables, calibration/overlap figures, row scores, simulation summaries and case packets are generated from saved artifacts under the dedicated public study folder. The public reproduction package excludes all private PFF data/artifacts; the audit remains a separate private comparison appendix. Data reuse requires the pinned StatsBomb/Hudl license and attribution. This workflow does not publish or contact analysts automatically.
